# Pandas — Working with Tabular Data

Most real data arrives as a table: rows of records, columns of fields, mixed types, gaps where
someone did not fill in a form. NumPy arrays are the wrong shape of tool for that — they are
homogeneous and unlabelled. Pandas is the right one.

This is the longest notebook of the four, because Pandas is the library you will spend the most
time in.

### What this notebook covers

- Series and DataFrame: what they are and how the index changes everything
- Loading, inspecting and understanding a dataset you have just met
- Selecting data: `[]`, `.loc`, `.iloc`, `.at`, `.iat`, and when each is right
- Filtering rows by condition, including `isin`, `between` and `query`
- Creating and modifying columns, conditionally and in bulk
- Missing data: finding it, and deciding what to do about it
- GroupBy in depth — the single most valuable skill in Pandas
- `agg` versus `transform`, and why `apply` is not automatically the professional choice
- Text and dates, which is most real-world cleaning work
- Combining datasets with `concat` and `merge`, and the mistakes merges invite
- Reshaping: pivot, pivot_table, melt
- Reading and writing CSV, Excel and JSON
- Cleaning a deliberately messy dataset from start to finish
- Performance habits that matter, and ones that do not
- A mini project: analysing an employee dataset end to end

### Prerequisites

Basic Python, and the NumPy notebook — or at least comfort with boolean masks, `axis`, and the
idea of vectorised operations. Pandas is built on NumPy and reuses its ideas constantly.

### A note on versions

Written against Pandas 3.x, and it runs on 2.x too. Print your version in the next cell and
compare it against what you see below, because three things changed in Pandas 3 that older
tutorials get wrong:

| | Pandas 2.x | Pandas 3.x |
| --- | --- | --- |
| Text column dtype | `object` | `str` |
| Timestamp resolution | `datetime64[ns]` | `datetime64[us]` |
| Chained assignment (`df["a"][0] = 1`) | may work, warns | never works, warns |
| `DataFrame.applymap` | deprecated, still there | removed — use `.map` |

Nothing here breaks on Pandas 2. But if this notebook prints `str` where your screen says
`object`, or `[us]` where yours says `[ns]`, that is the only reason — you are on the older
version and the difference is cosmetic. The two places where the *behaviour* genuinely differs
adapt to whichever version you are running, and say so at the time.

Google Colab ships Pandas 2.x at the time of writing, so expect `object` and `[ns]` there.

## Setup

In [1]:
# %pip install pandas numpy

In [2]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.5
numpy : 2.2.1


## Why Pandas exists

Suppose you have employee records. In pure Python you would probably reach for a list of
dictionaries — a reasonable choice, and one worth seeing before we replace it.

In [3]:
employees = [
    {"name": "Ananya Rao", "dept": "Engineering", "salary": 920000, "years": 4},
    {"name": "Rahul Menon", "dept": "Engineering", "salary": 1450000, "years": 9},
    {"name": "Sneha Kulkarni", "dept": "Marketing", "salary": 680000, "years": 3},
    {"name": "Imran Sheikh", "dept": "Sales", "salary": 540000, "years": 2},
    {"name": "Priya Nair", "dept": "Marketing", "salary": 875000, "years": 6},
]

total = 0
for row in employees:
    total += row["salary"]
print("average salary:", total / len(employees))

engineering_total = 0
engineering_count = 0
for row in employees:
    if row["dept"] == "Engineering":
        engineering_total += row["salary"]
        engineering_count += 1
print("engineering average:", engineering_total / engineering_count)

average salary: 893000.0
engineering average: 1185000.0


Nothing wrong with that code — it is clear, and for five records it is entirely adequate.

The trouble starts when the questions multiply. Average by department, for every department.
Top earner in each. Everyone with more than five years' experience, sorted by salary. Salary
compared to their department's average. Each one is another loop, another accumulator, another
chance to get an edge case wrong. And "average by department, for every department" needs a
dictionary of accumulators, which is where this approach stops being pleasant.

Here is the same data in Pandas.

In [4]:
df = pd.DataFrame(employees)
df

,name,dept,salary,years
0,Ananya Rao,Engineering,920000,4
1,Rahul Menon,Engineering,1450000,9
2,Sneha Kulkarni,Marketing,680000,3
3,Imran Sheikh,Sales,540000,2
4,Priya Nair,Marketing,875000,6


Already more useful — it printed as a table, with column names and row numbers. Now the questions:

In [5]:
print("average salary:", df["salary"].mean())
print()
print("average by department:")
print(df.groupby("dept")["salary"].mean())
print()
print("top earner per department:")
print(df.loc[df.groupby("dept")["salary"].idxmax(), ["dept", "name", "salary"]])

average salary: 893000.0

average by department:
dept
Engineering    1185000.0
Marketing       777500.0
Sales           540000.0
Name: salary, dtype: float64

top earner per department:
          dept          name   salary
1  Engineering   Rahul Menon  1450000
4    Marketing    Priya Nair   875000
3        Sales  Imran Sheikh   540000


Each answer is one line, and each line reads roughly like the question. That is the value
proposition: Pandas gives you a vocabulary for table operations, so you spend your time thinking
about the question rather than about loop bookkeeping.

We will build up to that last line properly — it is doing three things at once — but you can
already see its shape: group by department, find the index of the maximum salary in each group,
then look up those rows.

## The mental model

Two objects, and everything else follows from them.

**A Series is one labelled column.** It holds a NumPy array of values plus an **index** — a set of
labels, one per element.

```text
index        values
-----        ------
  0          920000
  1         1450000
  2          680000
```

**A DataFrame is a dictionary of Series that all share one index.**

```text
             name              dept          salary    years
index   ┌──────────────┬─────────────────┬──────────┬────────┐
  0     │ Ananya Rao   │ Engineering     │  920000  │   4    │
  1     │ Rahul Menon  │ Engineering     │ 1450000  │   9    │
  2     │ Sneha K.     │ Marketing       │  680000  │   3    │
        └──────────────┴─────────────────┴──────────┴────────┘
         each column is a Series with its own dtype
         all columns share the same index
```

Two consequences to keep in mind from the start:

1. **Columns are the natural unit of work.** `df["salary"] * 1.1` operates on a whole column at
   once, the way NumPy does. Working row by row is possible and usually the wrong instinct.
2. **The index is not decoration.** It is used to align data in every operation. Two Series added
   together are matched up *by label*, not by position — which is either extremely helpful or
   extremely confusing, depending on whether you were expecting it.

## Series

In [6]:
temperatures = pd.Series([28.4, 31.2, 29.8, 33.5, 30.1])
temperatures

0    28.4
1    31.2
2    29.8
3    33.5
4    30.1
dtype: float64

The left column is the index — automatically `0, 1, 2, 3, 4` since we did not supply one. The
right column is the values. At the bottom, `dtype: float64` — one type for the whole Series, as
with a NumPy array.

The interesting part is supplying your own labels:

In [7]:
temperatures = pd.Series(
    [28.4, 31.2, 29.8, 33.5, 30.1],
    index=["Mon", "Tue", "Wed", "Thu", "Fri"],
    name="max_temp_c",
)
temperatures

Mon    28.4
Tue    31.2
Wed    29.8
Thu    33.5
Fri    30.1
Name: max_temp_c, dtype: float64

Now the data carries its own labels. Look up by label, not by counting positions:

In [8]:
print("Wednesday:", temperatures["Wed"])
print("Tue to Thu:")
print(temperatures["Tue":"Thu"])

Wednesday: 29.8
Tue to Thu:
Tue    31.2
Wed    29.8
Thu    33.5
Name: max_temp_c, dtype: float64


Two things to notice.

`temperatures["Wed"]` is a lookup by name — no need to remember that Wednesday is at position 2.

`temperatures["Tue":"Thu"]` includes **both** endpoints. This is different from every other kind
of slicing in Python, where the end is excluded. The reasoning is that with labels there is no
"one past the end" to point at: if you ask for Tuesday through Thursday, you want Thursday.
Label slices include the end; position slices exclude it. It is worth saying that out loud
twice, because it causes real bugs.

In [9]:
print("by label, Tue:Thu →", list(temperatures["Tue":"Thu"].index))
print("by position, 1:3  →", list(temperatures[1:3].index))

by label, Tue:Thu → ['Tue', 'Wed', 'Thu']
by position, 1:3  → ['Tue', 'Wed']


### Creating Series from other things

In [10]:
from_dict = pd.Series({"Mon": 28.4, "Tue": 31.2, "Wed": 29.8})
print(from_dict)
print()

from_scalar = pd.Series(0.0, index=["a", "b", "c"])
print(from_scalar)
print()

from_numpy = pd.Series(np.arange(5) * 1.5)
print(from_numpy)

Mon    28.4
Tue    31.2
Wed    29.8
dtype: float64

a    0.0
b    0.0
c    0.0
dtype: float64

0    0.0
1    1.5
2    3.0
3    4.5
4    6.0
dtype: float64


A dictionary is the most natural source: keys become the index, values become the values. Passing
a single number with an index broadcasts it, which is how you create a placeholder column.

### Series attributes

In [11]:
print("values :", temperatures.values)
print("type of values:", type(temperatures.values))
print("index  :", list(temperatures.index))
print("dtype  :", temperatures.dtype)
print("name   :", temperatures.name)
print("size   :", temperatures.size)

values : [28.4 31.2 29.8 33.5 30.1]
type of values: <class 'numpy.ndarray'>
index  : ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
dtype  : float64
name   : max_temp_c
size   : 5


`.values` hands back the underlying NumPy array — proof that Pandas is a labelling layer over
NumPy arrays rather than a replacement for them. In new code prefer `.to_numpy()`, which is the
documented way and behaves more predictably across dtypes.

In [12]:
print(temperatures.to_numpy())

[28.4 31.2 29.8 33.5 30.1]


### Arithmetic and vectorisation

Everything from NumPy carries over: operations apply element-wise and the index comes along.

In [13]:
print("in Fahrenheit:")
print((temperatures * 9 / 5 + 32).round(1))
print()
print("above the weekly mean:")
print(temperatures - temperatures.mean())

in Fahrenheit:
Mon    83.1
Tue    88.2
Wed    85.6
Thu    92.3
Fri    86.2
Name: max_temp_c, dtype: float64

above the weekly mean:
Mon   -2.2
Tue    0.6
Wed   -0.8
Thu    2.9
Fri   -0.5
Name: max_temp_c, dtype: float64


### Index alignment — the behaviour that surprises everyone

When two Series are combined, Pandas matches them **by index label**, not by position.

In [14]:
week1 = pd.Series([100, 120, 90], index=["Mon", "Tue", "Wed"])
week2 = pd.Series([80, 95, 130], index=["Wed", "Mon", "Tue"])

print("week1:"); print(week1)
print("\nweek2 (deliberately in a different order):"); print(week2)
print("\nsum:")
print(week1 + week2)

week1:
Mon    100
Tue    120
Wed     90
dtype: int64

week2 (deliberately in a different order):
Wed     80
Mon     95
Tue    130
dtype: int64

sum:
Mon    195
Tue    250
Wed    170
dtype: int64


`week2` was written in a different order, and the sum is still correct: Monday's 100 was added to
Monday's 95, not to Wednesday's 80. With NumPy arrays that would have been wrong, silently.

This alignment is genuinely one of the best things about Pandas. It is also the source of the most
baffling bug in the library, which happens when labels do **not** all match:

In [15]:
week3 = pd.Series([100, 120, 90], index=["Mon", "Tue", "Wed"])
week4 = pd.Series([80, 95], index=["Tue", "Thu"])

print(week3 + week4)

Mon      NaN
Thu      NaN
Tue    200.0
Wed      NaN
dtype: float64


Every label that appears in only one of the two Series becomes `NaN`. Pandas took the union of the
labels — Mon, Tue, Wed, Thu — and could only compute a real answer for Tuesday.

That is the correct, conservative behaviour: it cannot add a number to something that does not
exist. But if you were expecting positional addition you now have mysterious missing values. When
that happens, compare the two indexes:

In [16]:
print("only in week3:", week3.index.difference(week4.index).tolist())
print("only in week4:", week4.index.difference(week3.index).tolist())

only in week3: ['Mon', 'Wed']
only in week4: ['Thu']


To fill instead of propagating `NaN`, use the method form with `fill_value`:

In [17]:
print(week3.add(week4, fill_value=0))

Mon    100.0
Thu     95.0
Tue    200.0
Wed     90.0
dtype: float64


Every operator has a method twin — `add`, `sub`, `mul`, `div`, `floordiv`, `pow` — and they all
take `fill_value`. Reach for them when a missing label should be treated as zero (or one, for
multiplication) rather than as unknown.

### Filtering a Series

Exactly the NumPy pattern: build a boolean mask, use it to select.

In [18]:
print("mask:")
print(temperatures > 30)
print("\nhot days:")
print(temperatures[temperatures > 30])
print("\nhow many:", (temperatures > 30).sum())

mask:
Mon    False
Tue     True
Wed    False
Thu     True
Fri     True
Name: max_temp_c, dtype: bool

hot days:
Tue    31.2
Thu    33.5
Fri    30.1
Name: max_temp_c, dtype: float64

how many: 3


The mask is itself a Series — with the same index — of `True`/`False`. Indexing with it keeps the
rows where it is `True`, and the labels come along, so you can still see *which* days were hot.

Combining conditions uses `&`, `|`, `~` with parentheses, for the same reason as in NumPy:

In [19]:
print(temperatures[(temperatures > 29) & (temperatures < 32)])

Tue    31.2
Wed    29.8
Fri    30.1
Name: max_temp_c, dtype: float64


### Missing values in a Series

In [20]:
rainfall = pd.Series([0.0, 12.4, np.nan, 3.1, np.nan, 28.7],
                     index=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"])
print(rainfall)
print("\nisna:")
print(rainfall.isna())
print("\ncount (non-missing):", rainfall.count())
print("size (all entries)  :", rainfall.size)
print("mean, skipping gaps :", rainfall.mean())
print("sum, skipping gaps  :", rainfall.sum())

Mon     0.0
Tue    12.4
Wed     NaN
Thu     3.1
Fri     NaN
Sat    28.7
dtype: float64

isna:
Mon    False
Tue    False
Wed     True
Thu    False
Fri     True
Sat    False
dtype: bool

count (non-missing): 4
size (all entries)  : 6
mean, skipping gaps : 11.05
sum, skipping gaps  : 44.2


Notice the difference from NumPy: `rainfall.mean()` **skips** the missing values rather than
returning `NaN`. Pandas assumes you want the mean of what you have, which is usually right but
means you should know how many values it was actually computed from — hence `.count()` versus
`.size`.

If you want NumPy's behaviour, pass `skipna=False`:

In [21]:
print("skipna=False:", rainfall.mean(skipna=False))

skipna=False: nan


And the standard ways to deal with the gaps:

In [22]:
print("dropped:")
print(rainfall.dropna())
print("\nfilled with 0:")
print(rainfall.fillna(0))
print("\nforward filled:")
print(rainfall.ffill())

dropped:
Mon     0.0
Tue    12.4
Thu     3.1
Sat    28.7
dtype: float64

filled with 0:
Mon     0.0
Tue    12.4
Wed     0.0
Thu     3.1
Fri     0.0
Sat    28.7
dtype: float64

forward filled:
Mon     0.0
Tue    12.4
Wed    12.4
Thu     3.1
Fri     3.1
Sat    28.7
dtype: float64


`ffill` ("forward fill") copies the last known value into the gap, which makes sense for something
like a stock price that persists until it changes, and is plainly wrong for rainfall — Wednesday
did not get Tuesday's 12.4mm. Filling with 0 asserts that no rain fell, which may be right if the
gap means "nothing recorded because nothing happened", and wrong if it means "the gauge broke".

There is no default answer. The section on missing data goes into this properly; for now, notice
that the choice of fill is a statement about your data, not a formatting decision.

If you have seen `rainfall.fillna(method="ffill")` in an older tutorial: that keyword was removed
in Pandas 3. Use `.ffill()` and `.bfill()` directly.

## Exercises — Series

**Level 1.** Create a Series of the marks `[68, 74, 55, 91, 83]` indexed by the subject names
`Maths, Physics, Chemistry, English, CompSci`. Print the highest mark, the subject it belongs to
(`idxmax` is worth a look), and the mean.

**Level 2.** Using the same Series, print only the subjects scored above the average, and count
how many there are. Then add 5 bonus marks to every subject, capped at 100.

**Level 3.** Two shopkeepers record sales:
`shop_a = pd.Series([120, 80, 45], index=["pens", "books", "bags"])` and
`shop_b = pd.Series([60, 30, 55], index=["books", "pens", "erasers"])`.
Compute total sales per item. Explain what happens to `bags` and `erasers`, then produce a version
where a missing item counts as zero. Which of the two results would you hand to the shopkeepers,
and why?

## DataFrames

A DataFrame can be built from most shapes of data you are likely to have.

### From a dictionary of columns

The most common form when you are typing data in by hand. Each key becomes a column name, each
value becomes that column's data.

In [23]:
courses = pd.DataFrame({
    "code": ["CS101", "CS102", "MA201", "PH101", "EN105"],
    "title": ["Intro to Programming", "Data Structures", "Linear Algebra",
              "Classical Mechanics", "Technical Writing"],
    "credits": [4, 4, 3, 4, 2],
    "enrolled": [180, 142, 96, 120, 210],
    "department": ["CS", "CS", "Maths", "Physics", "English"],
})
courses

,code,title,credits,enrolled,department
0,CS101,Intro to Programming,4,180,CS
1,CS102,Data Structures,4,142,CS
2,MA201,Linear Algebra,3,96,Maths
3,PH101,Classical Mechanics,4,120,Physics
4,EN105,Technical Writing,2,210,English


All the lists must be the same length; if they are not, Pandas raises
`All arrays must be of the same length`, which is a useful check on your typing.

### From a list of dictionaries

Natural when your data arrives one record at a time — from an API, a database cursor, or a scraper.

In [24]:
readings = pd.DataFrame([
    {"sensor": "S1", "temp": 21.4, "humidity": 55},
    {"sensor": "S2", "temp": 22.8, "humidity": 51},
    {"sensor": "S3", "temp": 20.9},
])
readings

,sensor,temp,humidity
0,S1,21.4,55.0
1,S2,22.8,51.0
2,S3,20.9,NaN


The third record had no `humidity`, and Pandas filled in `NaN` rather than failing. It takes the
union of all the keys it sees. Convenient, and worth knowing about — a typo in one key name
silently produces an extra, mostly-empty column.

### From a NumPy array

In [25]:
rng = np.random.default_rng(11)
matrix = rng.integers(40, 100, size=(4, 3))

marks = pd.DataFrame(matrix,
                     columns=["Maths", "Physics", "Chemistry"],
                     index=["Aarti", "Bilal", "Chirag", "Divya"])
marks

,Maths,Physics,Chemistry
Aarti,48,47,87
Bilal,69,75,76
Chirag,82,41,69
Divya,48,64,95


This is the bridge between the two libraries. A 2D array plus column names plus an index gives you
a DataFrame; `marks.to_numpy()` takes you back. Use it when a calculation is easier in NumPy — as
a rule, when all the columns are numeric and you want matrix operations.

In [26]:
print(marks.to_numpy())
print("\nsame data, back in NumPy:", marks.to_numpy().shape)

[[48 47 87]
 [69 75 76]
 [82 41 69]
 [48 64 95]]

same data, back in NumPy: (4, 3)


### From a list of lists

In [27]:
pd.DataFrame(
    [["Ananya", "Engineering", 920000],
     ["Rahul", "Engineering", 1450000],
     ["Sneha", "Marketing", 680000]],
    columns=["name", "dept", "salary"],
)

,name,dept,salary
0,Ananya,Engineering,920000
1,Rahul,Engineering,1450000
2,Sneha,Marketing,680000


Without `columns=` you get integer column names `0, 1, 2`, which works but makes every later line
of code cryptic. Always name your columns.

## The dataset for the rest of this notebook

Let us build something with enough size and mess to be interesting: a company's employee records.
It is generated with a fixed seed, so your output matches this notebook exactly.

In [28]:
rng = np.random.default_rng(seed=7)

first_names = ["Ananya", "Rahul", "Sneha", "Imran", "Priya", "Vikram", "Meera", "Arjun",
               "Kavita", "Rohan", "Fatima", "Nikhil", "Divya", "Sameer", "Anjali",
               "Karthik", "Neha", "Aditya", "Pooja", "Suresh", "Ritika", "Manish",
               "Shalini", "Deepak", "Aisha", "Varun", "Swati", "Rajesh", "Nandini", "Tarun"]
last_names = ["Rao", "Menon", "Kulkarni", "Sheikh", "Nair", "Singh", "Iyer", "Desai",
              "Joshi", "Verma", "Khan", "Gupta", "Bose", "Reddy", "Mehta"]

departments = np.array(["Engineering", "Marketing", "Sales", "Finance", "HR"])
dept_weights = [0.40, 0.15, 0.20, 0.15, 0.10]
base_salary = {"Engineering": 900_000, "Marketing": 700_000, "Sales": 650_000,
               "Finance": 800_000, "HR": 600_000}

n = 60
dept = rng.choice(departments, size=n, p=dept_weights)
years = rng.integers(1, 16, size=n)

salary = np.array([base_salary[d] for d in dept]) \
    + years * rng.integers(30_000, 60_000, size=n) \
    + rng.normal(0, 40_000, size=n)

staff = pd.DataFrame({
    "emp_id": [f"E{1000 + i}" for i in range(n)],
    "name": [f"{rng.choice(first_names)} {rng.choice(last_names)}" for _ in range(n)],
    "department": dept,
    "years_experience": years,
    "salary": salary.round(-3).astype(int),
    "city": rng.choice(["Bengaluru", "Pune", "Mumbai", "Hyderabad", "Delhi"],
                       size=n, p=[0.35, 0.2, 0.2, 0.15, 0.1]),
    "remote": rng.choice([True, False], size=n, p=[0.4, 0.6]),
    "joined": pd.to_datetime("2024-01-01") - pd.to_timedelta(years * 365 + rng.integers(0, 300, n), unit="D"),
})

staff.head()

,emp_id,name,department,years_experience,salary,city,remote,joined
0,E1000,Shalini Reddy,Sales,7,1018000,Pune,False,2016-05-14
1,E1001,Swati Desai,Finance,10,1115000,Mumbai,False,2013-10-11
2,E1002,Aisha Reddy,Finance,7,1190000,Bengaluru,False,2016-04-30
3,E1003,Kavita Rao,Engineering,10,1458000,Pune,False,2013-08-02
4,E1004,Vikram Rao,Engineering,9,1243000,Bengaluru,True,2014-06-06


A few notes on how that was generated, since the techniques are ones you will use:

- `rng.choice(departments, p=dept_weights)` samples with probabilities, so Engineering is the
  biggest department — realistic, and more useful than an even split.
- `[base_salary[d] for d in dept]` is a list comprehension looking each department up in a
  dictionary. Salary then depends on department *and* experience *and* random noise, which gives
  data where group comparisons actually show something.
- `salary.round(-3)` rounds to the nearest thousand. A negative `decimals` rounds to the left of
  the decimal point, which is a genuinely useful trick.
- The backslashes at the ends of lines continue one expression across several lines. Wrapping the
  expression in parentheses is generally preferred, but the backslash form is common enough that
  you should recognise it.

## Inspecting a dataset

When you meet a dataset for the first time, run through these before doing anything clever. Skipping
this step is how you end up computing the mean of a column that is secretly text.

In [29]:
staff.head()

,emp_id,name,department,years_experience,salary,city,remote,joined
0,E1000,Shalini Reddy,Sales,7,1018000,Pune,False,2016-05-14
1,E1001,Swati Desai,Finance,10,1115000,Mumbai,False,2013-10-11
2,E1002,Aisha Reddy,Finance,7,1190000,Bengaluru,False,2016-04-30
3,E1003,Kavita Rao,Engineering,10,1458000,Pune,False,2013-08-02
4,E1004,Vikram Rao,Engineering,9,1243000,Bengaluru,True,2014-06-06


In [30]:
staff.tail(3)

,emp_id,name,department,years_experience,salary,city,remote,joined
57,E1057,Pooja Mehta,Engineering,7,1251000,Pune,False,2016-06-04
58,E1058,Swati Singh,HR,4,801000,Mumbai,True,2019-10-06
59,E1059,Priya Sheikh,Sales,4,879000,Bengaluru,False,2019-04-23


`head()` shows the first five rows, `tail()` the last five; both take a count. In a notebook,
writing the name of a DataFrame as the last line of a cell displays it nicely, which is why
`staff.head()` appears without `print()`.

In [31]:
print("shape  :", staff.shape)
print("rows   :", len(staff))
print("columns:", list(staff.columns))
print("index  :", staff.index)

shape  : (60, 8)
rows   : 60
columns: ['emp_id', 'name', 'department', 'years_experience', 'salary', 'city', 'remote', 'joined']
index  : RangeIndex(start=0, stop=60, step=1)


`shape` is `(rows, columns)`, the same convention as NumPy. `len(df)` is the row count, which
occasionally surprises people who expect it to be the number of columns.

In [32]:
staff.dtypes

emp_id                         str
name                           str
department                     str
years_experience             int64
salary                       int64
city                           str
remote                        bool
joined              datetime64[us]
dtype: object

This is the most important early check. Reading the list:

- `str` — text. (Pandas 3 gives text its own dtype. In Pandas 2 and earlier you would see
  `object` here, which is a catch-all for "any Python object" and much less informative.)
- `int64`, `float64` — numbers.
- `bool` — true/false.
- `datetime64[us]` — real timestamps, not text that looks like dates.

If a column you expect to be numeric shows up as `str`, something in it is not a number — a
stray "N/A", a currency symbol, a thousands separator. Find that out now, not three steps later
when `mean()` fails.

In [33]:
staff.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   emp_id            60 non-null     str           
 1   name              60 non-null     str           
 2   department        60 non-null     str           
 3   years_experience  60 non-null     int64         
 4   salary            60 non-null     int64         
 5   city              60 non-null     str           
 6   remote            60 non-null     bool          
 7   joined            60 non-null     datetime64[us]
dtypes: bool(1), datetime64[us](1), int64(2), str(4)
memory usage: 3.5 KB


`info()` combines several checks: the row count, and for each column its name, how many
**non-null** values it has, and its dtype. The non-null count is the quickest way to spot missing
data — any column whose count is below the row count has gaps.

Add `memory_usage="deep"` to get an honest memory figure that includes the text data rather than
just the pointers to it.

In [34]:
staff.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   emp_id            60 non-null     str           
 1   name              60 non-null     str           
 2   department        60 non-null     str           
 3   years_experience  60 non-null     int64         
 4   salary            60 non-null     int64         
 5   city              60 non-null     str           
 6   remote            60 non-null     bool          
 7   joined            60 non-null     datetime64[us]
dtypes: bool(1), datetime64[us](1), int64(2), str(4)
memory usage: 15.0 KB


In [35]:
staff.describe()

,years_experience,salary,joined
count,60.000000,6.000000e+01,60
mean,8.066667,1.156683e+06,2015-06-16 23:12:00
min,1.000000,6.620000e+05,2008-04-29 00:00:00
25%,4.000000,9.382500e+05,2011-12-15 06:00:00
50%,8.000000,1.143500e+06,2015-06-27 12:00:00
75%,11.250000,1.321000e+06,2019-05-30 18:00:00
max,15.000000,1.727000e+06,2022-11-08 00:00:00
std,4.494315,2.716351e+05,NaN


`describe()` summarises the **numeric** columns: count, mean, standard deviation, min, the three
quartiles, and max. Read it for sanity rather than for insight:

- Is `min` negative where it should not be? A negative salary means a data problem.
- Is `max` absurd? An experience of 150 years is a typo.
- Is `count` lower than the number of rows? That column has gaps.
- Is the mean far from the 50% value? The distribution is skewed, so the mean may be a misleading
  summary — a handful of very high salaries pull it up.

To include text and boolean columns, ask for everything:

In [36]:
staff.describe(include="all")

,emp_id,name,department,years_experience,salary,city,remote,joined
count,60,60,60,60.000000,6.000000e+01,60,60,60
unique,60,56,5,NaN,NaN,5,2,NaN
top,E1000,Shalini Reddy,Engineering,NaN,NaN,Bengaluru,False,NaN
freq,1,2,25,NaN,NaN,22,33,NaN
mean,NaN,NaN,NaN,8.066667,1.156683e+06,NaN,NaN,2015-06-16 23:12:00
min,NaN,NaN,NaN,1.000000,6.620000e+05,NaN,NaN,2008-04-29 00:00:00
25%,NaN,NaN,NaN,4.000000,9.382500e+05,NaN,NaN,2011-12-15 06:00:00
50%,NaN,NaN,NaN,8.000000,1.143500e+06,NaN,NaN,2015-06-27 12:00:00
75%,NaN,NaN,NaN,11.250000,1.321000e+06,NaN,NaN,2019-05-30 18:00:00
max,NaN,NaN,NaN,15.000000,1.727000e+06,NaN,NaN,2022-11-08 00:00:00


For text columns the useful entries are `unique` (how many distinct values), `top` (the most
common) and `freq` (how often it occurred). The statistical rows are `NaN`, because the mean of a
name is not a thing.

### `value_counts` — the workhorse

In [37]:
staff["department"].value_counts()

department
Engineering    25
Finance        11
Sales          10
Marketing      10
HR              4
Name: count, dtype: int64

This answers "what is in this column and how much of each" in one call, sorted by frequency. It is
probably the function you will use most often for getting oriented.

In [38]:
print("as proportions:")
print(staff["department"].value_counts(normalize=True).round(3))
print("\nin alphabetical order instead:")
print(staff["department"].value_counts().sort_index())
print("\ncross-tabulated with remote working:")
print(staff.value_counts(["department", "remote"]).unstack())

as proportions:
department
Engineering    0.417
Finance        0.183
Sales          0.167
Marketing      0.167
HR             0.067
Name: proportion, dtype: float64

in alphabetical order instead:
department
Engineering    25
Finance        11
HR              4
Marketing      10
Sales          10
Name: count, dtype: int64

cross-tabulated with remote working:
remote       False  True 
department               
Sales          9.0    1.0
Finance        8.0    3.0
Engineering   11.0   14.0
Marketing      5.0    5.0
HR             NaN    4.0


`normalize=True` gives shares instead of counts. Passing two column names to
`df.value_counts([...])` counts combinations, and `.unstack()` turns the second level into
columns, which makes the result readable as a small table. (Reshaping with `unstack` gets a proper
section later.)

For categorical columns, always check `value_counts` before grouping on them. It is how you find
out that your dataset contains `"Engineering"`, `"engineering"` and `"Engg"` as three separate
departments.

In [39]:
print("unique cities  :", staff["city"].unique())
print("how many cities:", staff["city"].nunique())
print("\ndistinct values per column:")
print(staff.nunique())

unique cities  : <StringArray>
['Pune', 'Mumbai', 'Bengaluru', 'Hyderabad', 'Delhi']
Length: 5, dtype: str
how many cities: 5

distinct values per column:
emp_id              60
name                56
department           5
years_experience    15
salary              58
city                 5
remote               2
joined              60
dtype: int64


`nunique()` across the whole DataFrame is a quick structural check. A column whose `nunique` equals
the row count is probably an identifier (like `emp_id`); a column with `nunique() == 1` carries no
information at all and can usually be dropped.

### A random peek

In [40]:
staff.sample(5, random_state=42)

,emp_id,name,department,years_experience,salary,city,remote,joined
0,E1000,Shalini Reddy,Sales,7,1018000,Pune,False,2016-05-14
5,E1005,Sameer Rao,Finance,11,1239000,Mumbai,False,2012-06-29
36,E1036,Rohan Menon,Engineering,15,1727000,Hyderabad,False,2008-05-12
45,E1045,Rahul Verma,Sales,9,952000,Pune,False,2014-10-25
13,E1013,Imran Khan,Marketing,7,921000,Pune,False,2016-04-15


`head()` shows the first rows, which in a sorted file are not representative — if the data is
ordered by date or by department, the first five rows all look alike. `sample()` gives you a
random selection instead. `random_state` makes it reproducible.

## Selecting data

This is where beginners lose the most time, because there are five ways to select and they look
similar. The good news is that the rules are simple once stated plainly.

### `[]` — columns by name

In [41]:
salaries = staff["salary"]

print(type(salaries))
print(salaries.head())

<class 'pandas.Series'>
0    1018000
1    1115000
2    1190000
3    1458000
4    1243000
Name: salary, dtype: int64


One column name in brackets gives you a **Series**. A *list* of names gives you a **DataFrame**:

In [42]:
subset = staff[["name", "department", "salary"]]
print(type(subset))
subset.head()

<class 'pandas.DataFrame'>


,name,department,salary
0,Shalini Reddy,Sales,1018000
1,Swati Desai,Finance,1115000
2,Aisha Reddy,Finance,1190000
3,Kavita Rao,Engineering,1458000
4,Vikram Rao,Engineering,1243000


The double brackets confuse everyone at first. There is only one pair of brackets doing the
selecting; the inner pair is a Python list.

```text
staff[ "salary" ]                →  Series (one column)
staff[ ["name", "salary"] ]      →  DataFrame (a list of columns)
        ↑                  ↑
        this is just a list literal
```

So `staff[["salary"]]` — a one-item list — gives you a one-column DataFrame, not a Series. Both
are legitimate; pick based on what you need next. Plotting and arithmetic usually want a Series;
anything that should stay tabular wants a DataFrame.

In [43]:
print("Series:", type(staff["salary"]).__name__, staff["salary"].shape)
print("DataFrame:", type(staff[["salary"]]).__name__, staff[["salary"]].shape)

Series: Series (60,)
DataFrame: DataFrame (60, 1)


### Attribute access

In [44]:
print(staff.salary.mean())

1156683.3333333333


`staff.salary` works and is shorter. It also breaks in several ways: a column name with a space in
it is unreachable, and a column named after an existing method — `count`, `mean`, `shape` — is
hidden by that method, so `df.count` gives you the function rather than your data. You also cannot
create a new column this way.

Use it for quick interactive exploration; use `staff["salary"]` in code you intend to keep.

### `.loc` — select by label

`.loc` takes **labels**, and can take two of them: rows first, then columns.

In [45]:
print("one row by index label:")
print(staff.loc[0])

one row by index label:
emp_id                            E1000
name                      Shalini Reddy
department                        Sales
years_experience                      7
salary                          1018000
city                               Pune
remote                            False
joined              2016-05-14 00:00:00
Name: 0, dtype: object


The result is a Series where the *column names* have become the index. That is a row turned on its
side, which makes sense once you see it: a row has one value per column, so its labels are the
column names.

In [46]:
print("a single cell:", staff.loc[0, "name"])
print("\nseveral rows, two columns:")
print(staff.loc[0:3, ["name", "salary"]])

a single cell: Shalini Reddy

several rows, two columns:
            name   salary
0  Shalini Reddy  1018000
1    Swati Desai  1115000
2    Aisha Reddy  1190000
3     Kavita Rao  1458000


Look carefully at `staff.loc[0:3]` — it returned **four** rows, 0 through 3. `.loc` slices are
inclusive of both endpoints, as with the Series label slicing earlier. Our index happens to be
integers, which makes this look like positional slicing, but it is not: those are labels that
happen to be numbers.

### `.loc` with a condition — the form you will use most

In [47]:
high_earners = staff.loc[staff["salary"] > 1_200_000, ["name", "department", "salary"]]
print(high_earners.head())
print("\ncount:", len(high_earners))

               name   department   salary
3        Kavita Rao  Engineering  1458000
4        Vikram Rao  Engineering  1243000
5        Sameer Rao      Finance  1239000
6      Arjun Sheikh  Engineering  1707000
8  Nandini Kulkarni      Finance  1233000

count: 26


This line deserves a full breakdown, because it is the single most common Pandas idiom and every
part of it is doing something.

```python
staff.loc[ staff["salary"] > 1_200_000 , ["name", "department", "salary"] ]
           └──────── row selector ────┘  └──────── column selector ─────┘
```

- `staff["salary"]` is a Series of 60 salaries.
- `staff["salary"] > 1_200_000` compares every one of them to the threshold and produces a Series
  of 60 `True`/`False` values — a boolean mask, with the same index as `staff`.
- Putting that mask in the **first** position of `.loc[...]` keeps the rows where it is `True`.
  This works precisely because the mask's index matches the DataFrame's index: Pandas aligns them.
- The **second** position, after the comma, lists the columns to keep.
- The result is a new DataFrame with the surviving rows and the three requested columns.

Common mistakes with this line:

- **Forgetting the comma** and writing `staff.loc[mask, "name", "salary"]` — that is three
  arguments, and `.loc` takes at most two. The columns must be a list.
- **Using `.iloc` instead** — `.iloc` wants integers, so a boolean mask with a label index either
  errors or misbehaves. If you have a condition, you want `.loc`.
- **Writing `staff["salary" > 1_200_000]`** with the brackets in the wrong place, which compares
  the string `"salary"` to a number and raises a `TypeError`.

Note also that `1_200_000` is just `1200000` — Python allows underscores in numeric literals as
digit separators, and for large numbers they are worth using.

### `.iloc` — select by position

`.iloc` ignores labels entirely and counts from zero, exactly like a NumPy array or a Python list.

In [48]:
print("first row:")
print(staff.iloc[0])
print("\nfirst three rows, first two columns:")
print(staff.iloc[0:3, 0:2])
print("\nlast row, last column:", staff.iloc[-1, -1])
print("\nspecific positions:")
print(staff.iloc[[0, 5, 10], [1, 4]])

first row:
emp_id                            E1000
name                      Shalini Reddy
department                        Sales
years_experience                      7
salary                          1018000
city                               Pune
remote                            False
joined              2016-05-14 00:00:00
Name: 0, dtype: object

first three rows, first two columns:
  emp_id           name
0  E1000  Shalini Reddy
1  E1001    Swati Desai
2  E1002    Aisha Reddy

last row, last column: 2019-04-23 00:00:00

specific positions:
             name   salary
0   Shalini Reddy  1018000
5      Sameer Rao  1239000
10      Arjun Rao  1191000


`.iloc` slices **exclude** the endpoint, like ordinary Python slicing. So `.iloc[0:3]` gives three
rows and `.loc[0:3]` gives four. That is not an inconsistency for its own sake: positions have a
natural "one past the end", labels do not.

| | `.loc` | `.iloc` |
| --- | --- | --- |
| Selects by | label | position |
| Slice endpoint | **included** | excluded |
| Accepts boolean mask | yes | only as a plain array, rarely useful |
| Accepts a list | of labels | of integers |
| Works after sorting/filtering | labels stay with their rows | positions shift |

That last row is the practical reason to prefer `.loc`. After you sort or filter, row 0 is a
different employee than before, but the employee whose label is 0 is still the same person.

In [49]:
sorted_staff = staff.sort_values("salary", ascending=False)

print("iloc[0] after sorting — the top earner:")
print(sorted_staff.iloc[0][["name", "salary"]].to_dict())
print("\nloc[0] after sorting — still the original row 0:")
print(sorted_staff.loc[0][["name", "salary"]].to_dict())

iloc[0] after sorting — the top earner:
{'name': 'Rohan Menon', 'salary': 1727000}

loc[0] after sorting — still the original row 0:
{'name': 'Shalini Reddy', 'salary': 1018000}


Both are correct; they answer different questions. "Show me the first row of this sorted table" is
`.iloc[0]`. "Show me employee 0" is `.loc[0]`.

### `.at` and `.iat` — one cell, quickly

In [50]:
print("at  (label) :", staff.at[3, "name"])
print("iat (position):", staff.iat[3, 1])

at  (label) : Kavita Rao
iat (position): Kavita Rao


These do the same job as `.loc` and `.iloc` but only for a single cell, and they are faster because
they skip all the machinery for handling slices and lists. The speed only matters inside a loop
over many individual cells — and if you are doing that, the real fix is usually to stop looping.
Use them when you genuinely need one value.

### The summary table

| You want | Use | Example |
| --- | --- | --- |
| one column as a Series | `df["col"]` | `staff["salary"]` |
| several columns | `df[["a", "b"]]` | `staff[["name", "city"]]` |
| rows matching a condition | `df.loc[mask]` | `staff.loc[staff["remote"]]` |
| rows by condition + some columns | `df.loc[mask, cols]` | `staff.loc[mask, ["name", "salary"]]` |
| rows by index label | `df.loc[label]` | `staff.loc[7]` |
| rows by position | `df.iloc[n]` | `staff.iloc[7]` |
| a block by position | `df.iloc[r1:r2, c1:c2]` | `staff.iloc[:5, :3]` |
| one cell by label | `df.at[label, "col"]` | `staff.at[7, "name"]` |
| one cell by position | `df.iat[row, col]` | `staff.iat[7, 1]` |

If you remember only one thing: **`.loc` for labels and conditions, `.iloc` for positions.**
Nearly everything else is detail.

### Setting the index to something meaningful

The default `0, 1, 2, ...` index carries no information. When your rows have natural identifiers,
promoting one to the index makes lookups readable.

In [51]:
by_id = staff.set_index("emp_id")
print(by_id.head(3))

print("\nlook up an employee directly:")
print(by_id.loc["E1004", ["name", "department", "salary"]])

                 name department  years_experience   salary       city  \
emp_id                                                                   
E1000   Shalini Reddy      Sales                 7  1018000       Pune   
E1001     Swati Desai    Finance                10  1115000     Mumbai   
E1002     Aisha Reddy    Finance                 7  1190000  Bengaluru   

        remote     joined  
emp_id                     
E1000    False 2016-05-14  
E1001    False 2013-10-11  
E1002    False 2016-04-30  

look up an employee directly:
name           Vikram Rao
department    Engineering
salary            1243000
Name: E1004, dtype: object


`set_index` returns a new DataFrame; the original `staff` is unchanged. `reset_index()` reverses it,
moving the index back into a normal column.

In [52]:
print(by_id.reset_index().head(2))

  emp_id           name department  years_experience   salary    city  remote  \
0  E1000  Shalini Reddy      Sales                 7  1018000    Pune   False   
1  E1001    Swati Desai    Finance                10  1115000  Mumbai   False   

      joined  
0 2016-05-14  
1 2013-10-11  


Two warnings about setting an index:

- Index values do not have to be unique, and a duplicated index produces surprising results —
  `df.loc["X"]` will return several rows instead of one. Check with
  `df.index.is_unique` if you rely on uniqueness.
- If you set an index and then forget, `.loc[0]` stops working and the error message
  (`KeyError: 0`) is not obviously about the index. `df.index[:3]` tells you what the labels are.

In [53]:
print("index unique?", by_id.index.is_unique)
print("first few labels:", by_id.index[:3].tolist())

index unique? True
first few labels: ['E1000', 'E1001', 'E1002']


## Exercises — DataFrames and selection

**Level 1.** From `staff`, display the first eight rows; list the column names; print the shape;
and show just the `name` and `city` columns for the first five rows.

**Level 2.** Select all employees in Engineering earning more than ₹1,000,000, showing only
`name`, `years_experience` and `salary`. Then do the same for employees who are *not* in
Engineering, using `~`.

**Level 3.** Someone hands you this line and says it does not work:
`staff.loc[staff.salary > 1000000, "name", "salary"]`. Explain what is wrong with it, fix it, and
then write the equivalent using `.iloc` — and explain why the `.iloc` version is a worse way to
express this particular request.

## Filtering rows

Filtering is boolean masking, exactly as in NumPy, with the index doing the alignment.

In [54]:
mask = staff["remote"] == True
print(mask.head())
print("\nremote employees:", mask.sum(), "of", len(staff))

0    False
1    False
2    False
3    False
4     True
Name: remote, dtype: bool

remote employees: 27 of 60


Since `remote` is already boolean, `== True` is redundant. Write it directly:

In [55]:
remote_staff = staff[staff["remote"]]
print(remote_staff[["name", "department", "city"]].head())

              name   department       city
4       Vikram Rao  Engineering  Bengaluru
6     Arjun Sheikh  Engineering  Hyderabad
9      Tarun Joshi    Marketing       Pune
12  Imran Kulkarni  Engineering  Bengaluru
15     Sameer Nair        Sales  Hyderabad


For non-boolean columns you need a real comparison:

In [56]:
senior = staff[staff["years_experience"] >= 10]
print("employees with 10+ years:", len(senior))
print(senior[["name", "years_experience", "salary"]].head())

employees with 10+ years: 22
            name  years_experience   salary
1    Swati Desai                10  1115000
3     Kavita Rao                10  1458000
5     Sameer Rao                11  1239000
6   Arjun Sheikh                15  1707000
14     Sneha Rao                13  1309000


### `df[mask]` versus `df.loc[mask]`

Both work for selecting rows. `df[mask]` is shorter; `df.loc[mask]` also lets you pick columns in
the same expression, and it is the form you must use when *assigning*. Given that, many people
just use `.loc` everywhere for consistency. Either is fine for reading data.

In [57]:
print(staff[staff["salary"] > 1_400_000][["name", "salary"]].head(3))
print()
print(staff.loc[staff["salary"] > 1_400_000, ["name", "salary"]].head(3))

            name   salary
3     Kavita Rao  1458000
6   Arjun Sheikh  1707000
18   Suresh Bose  1497000

            name   salary
3     Kavita Rao  1458000
6   Arjun Sheikh  1707000
18   Suresh Bose  1497000


The first version selects rows, then selects columns from that result — two separate operations
chained together. The second does it in one. The results are identical here, but chaining
`[...][...]` is a habit worth dropping: when you assign through a chain, things go wrong, as the
section on chained assignment shows.

### Multiple conditions

In [58]:
target = staff[(staff["department"] == "Engineering") & (staff["years_experience"] > 8)]
print("senior engineers:", len(target))
print(target[["name", "years_experience", "salary"]].head())

senior engineers: 13
              name  years_experience   salary
3       Kavita Rao                10  1458000
4       Vikram Rao                 9  1243000
6     Arjun Sheikh                15  1707000
21  Arjun Kulkarni                11  1593000
31     Rohan Verma                13  1396000


As in NumPy: `&` for and, `|` for or, `~` for not, and **every condition in its own parentheses**.

In [59]:
try:
    staff[(staff["department"] == "Engineering") and (staff["years_experience"] > 8)]
except ValueError as e:
    print("ValueError:", e)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


`The truth value of a Series is ambiguous` — the same error as in NumPy, for the same reason.
Python's `and` wants one true-or-false answer and you handed it sixty. Use `&`.

In [60]:
either = staff[(staff["city"] == "Mumbai") | (staff["city"] == "Delhi")]
print("Mumbai or Delhi:", len(either))

not_engineering = staff[~(staff["department"] == "Engineering")]
print("not in Engineering:", len(not_engineering))

Mumbai or Delhi: 14
not in Engineering: 35


### `isin` — instead of a chain of `|`

In [61]:
metro = staff[staff["city"].isin(["Mumbai", "Delhi", "Bengaluru"])]
print("in a metro:", len(metro))

print("\nthe long way, for comparison:")
long_way = staff[
    (staff["city"] == "Mumbai")
    | (staff["city"] == "Delhi")
    | (staff["city"] == "Bengaluru")
]
print("same count:", len(long_way))

in a metro: 36

the long way, for comparison:
same count: 36


`isin` takes any collection and tests membership element-wise. Three advantages over the chain of
`|`: it is shorter, it does not grow when the list grows, and the list can come from a variable —
which is what you want when the set of cities is computed elsewhere rather than typed in.

`~` inverts it, giving "not in":

In [62]:
print("outside the metros:", len(staff[~staff["city"].isin(["Mumbai", "Delhi", "Bengaluru"])]))

outside the metros: 24


### `between` — instead of two comparisons

In [63]:
mid_career = staff[staff["years_experience"].between(5, 10)]
print("5 to 10 years:", len(mid_career))
print("equivalent:", len(staff[(staff["years_experience"] >= 5) & (staff["years_experience"] <= 10)]))

5 to 10 years: 26
equivalent: 26


`between(a, b)` is inclusive at both ends by default. Pass `inclusive="left"`, `"right"` or
`"neither"` to change that — and do check, because "between 5 and 10 years" is ambiguous in
English and precise in code.

### `query` — filtering with a string

In [64]:
result = staff.query("department == 'Engineering' and years_experience > 8")
print(len(result))
print(result[["name", "years_experience"]].head(3))

13
           name  years_experience
3    Kavita Rao                10
4    Vikram Rao                 9
6  Arjun Sheikh                15


`query` takes a string expression and evaluates it with the column names in scope. Inside the
string you write `and`, `or`, `not` as words, you do not repeat the DataFrame name, and you do not
need parentheses around each condition. For long filters that is a real readability gain:

```python
# with brackets
staff[(staff["department"] == "Engineering")
      & (staff["years_experience"] > 8)
      & (staff["salary"] < 1_500_000)
      & (staff["remote"])]

# with query
staff.query("department == 'Engineering' and years_experience > 8 "
            "and salary < 1_500_000 and remote")
```

The trade-offs, honestly:

- The expression is a string, so your editor cannot check it and a typo in a column name fails at
  runtime rather than being flagged as you type.
- You need `@` to refer to a Python variable: `staff.query("salary > @threshold")`.
- It is slightly slower to set up, which never matters for a filter you run once and matters if
  you run it in a loop.

Use `query` for long interactive filters where it reads better. Use brackets in library code, or
where the condition is built up programmatically. Neither is the "professional" choice.

In [65]:
threshold = 1_300_000
print("above threshold:", len(staff.query("salary > @threshold")))

above threshold: 16


### Filtering for a column name with a space

One case where `query` needs help — and a good reason to avoid spaces in column names entirely:

In [66]:
odd_names = pd.DataFrame({"employee name": ["A", "B"], "total pay": [100, 200]})
print(odd_names.query("`total pay` > 150"))

  employee name  total pay
1             B        200


Backticks around the name make it work. The better fix is to rename the columns when the data
arrives:

In [67]:
tidy = odd_names.rename(columns={"employee name": "employee_name", "total pay": "total_pay"})
print(tidy.columns.tolist())

['employee_name', 'total_pay']


A shortcut for renaming many columns at once — lowercase everything and replace spaces:

In [68]:
messy_columns = pd.DataFrame(columns=["Employee Name", "Total Pay", "Start Date"])
messy_columns.columns = messy_columns.columns.str.lower().str.replace(" ", "_")
print(messy_columns.columns.tolist())

['employee_name', 'total_pay', 'start_date']


`df.columns` is an Index, and Index objects have the same `.str` accessor as Series, so string
methods apply to all the column names at once. This one line is worth remembering — it is the
first thing many people do to any CSV they did not create.

## Creating and modifying columns

### Direct assignment

In [69]:
work = staff.copy()

work["monthly_salary"] = work["salary"] / 12
work["salary_lakhs"] = (work["salary"] / 100_000).round(2)

print(work[["name", "salary", "monthly_salary", "salary_lakhs"]].head(3))

            name   salary  monthly_salary  salary_lakhs
0  Shalini Reddy  1018000    84833.333333         10.18
1    Swati Desai  1115000    92916.666667         11.15
2    Aisha Reddy  1190000    99166.666667         11.90


Assigning to a column name that does not exist creates it; assigning to one that does replaces it.
The right-hand side is a whole-column calculation, so no loop is needed.

Notice `work = staff.copy()`. Without it we would be modifying `staff` itself, and later sections
expect the original. Copying before you modify is cheap insurance when you are experimenting.

### Comparing to the beginner approach

In [70]:
monthly_beginner = []
for i in range(len(staff)):
    monthly_beginner.append(staff["salary"].iloc[i] / 12)

work["monthly_loop"] = monthly_beginner
print(work[["monthly_salary", "monthly_loop"]].head(3))
print("identical:", np.allclose(work["monthly_salary"], work["monthly_loop"]))

   monthly_salary  monthly_loop
0    84833.333333  84833.333333
1    92916.666667  92916.666667
2    99166.666667  99166.666667
identical: True


The loop version gets the right answer and is three times as long. It is also slower by a wide
margin on real data, because `staff["salary"].iloc[i]` does a full lookup on every iteration.

Write the column expression. `work["salary"] / 12` is not a clever trick — it is the normal way to
do arithmetic in Pandas, and the loop is the unusual thing.

### Columns from several other columns

In [71]:
work["salary_per_year_exp"] = (work["salary"] / work["years_experience"]).round(0)
print(work[["name", "salary", "years_experience", "salary_per_year_exp"]].head(3))

            name   salary  years_experience  salary_per_year_exp
0  Shalini Reddy  1018000                 7             145429.0
1    Swati Desai  1115000                10             111500.0
2    Aisha Reddy  1190000                 7             170000.0


Two columns are combined element-wise, matched up by index. Row 0's salary is divided by row 0's
experience, and so on.

Watch for division by zero. Here `years_experience` starts at 1, but if it could be 0 you would get
`inf` rather than an error:

In [72]:
sample = pd.Series([10.0, 20.0])
print(sample / pd.Series([2.0, 0.0]))

0    5.0
1    inf
dtype: float64


`inf` then propagates through every later calculation. `.replace([np.inf, -np.inf], np.nan)` turns
those into proper missing values, which at least aggregate sensibly.

### Conditional columns

The most common real-world need, and there are three good ways depending on complexity.

In [73]:
work["is_senior"] = work["years_experience"] >= 10
print(work[["name", "years_experience", "is_senior"]].head(3))

            name  years_experience  is_senior
0  Shalini Reddy                 7      False
1    Swati Desai                10       True
2    Aisha Reddy                 7      False


A comparison is already a boolean column. Nothing more is needed for a two-way flag.

For two outcomes that are not `True`/`False`, use `np.where`:

In [74]:
work["level"] = np.where(work["years_experience"] >= 10, "Senior", "Junior")
print(work["level"].value_counts())

level
Junior    38
Senior    22
Name: count, dtype: int64


For more than two, `np.select`:

In [75]:
conditions = [
    work["years_experience"] >= 12,
    work["years_experience"] >= 7,
    work["years_experience"] >= 3,
]
labels = ["Principal", "Senior", "Mid"]

work["grade"] = np.select(conditions, labels, default="Junior")
print(work["grade"].value_counts())

grade
Senior       23
Principal    15
Mid          14
Junior        8
Name: count, dtype: int64


`np.select` takes the first matching condition, so the order matters: most restrictive first.
Written the other way round, everyone with 3+ years would be labelled "Mid" and the later
conditions would never be reached.

The beginner version of the same logic, with a function and `apply`:

In [76]:
def grade_of(years):
    if years >= 12:
        return "Principal"
    elif years >= 7:
        return "Senior"
    elif years >= 3:
        return "Mid"
    return "Junior"


work["grade_apply"] = work["years_experience"].apply(grade_of)
print("identical to np.select:", (work["grade"] == work["grade_apply"]).all())

identical to np.select: True


Both give the same answer, and — being honest — the `apply` version is easier to read. An
`if/elif` chain in a named function is about as clear as code gets, and it is trivial to extend.

The difference is speed. `apply` calls your Python function once per row; `np.select` evaluates
three vectorised comparisons over the whole column. Let us measure rather than assert, on two
million rows.

In [77]:
import time


def timed(label, fn, repeats=2):
    """Run fn a couple of times and report the best elapsed time."""
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        result = fn()
        best = min(best, time.perf_counter() - start)
    print(f"{label:34} {best:.4f} s")
    return result


big = pd.DataFrame({"years": rng.integers(1, 16, size=2_000_000)})

by_apply = timed("apply with if/elif", lambda: big["years"].apply(grade_of))
by_select = timed("np.select", lambda: np.select(
    [big["years"] >= 12, big["years"] >= 7, big["years"] >= 3],
    ["Principal", "Senior", "Mid"], default="Junior"))

print("\nsame result:", (by_apply == by_select).all())

apply with if/elif                 0.2007 s
np.select                          0.0570 s



same result: True


On this machine `np.select` came out around four times faster. That is a real gain, but it is
smaller than the folklore suggests — building a two-million-element array of strings is itself
most of the work, and both approaches have to do it.

Where vectorising genuinely transforms the timing is **arithmetic**, where the vectorised version
stays entirely in compiled code and never builds Python objects at all:

In [78]:
per_row = timed("apply, arithmetic", lambda: big["years"].apply(lambda y: y * 1.5 + 2))
vectorised = timed("column arithmetic", lambda: big["years"] * 1.5 + 2)

print("\nsame result:", np.allclose(per_row, vectorised))

apply, arithmetic                  0.2875 s
column arithmetic                  0.0084 s

same result: True


That is the twenty-fold difference people have in mind when they say "avoid `apply`" — and notice
it is a case where nobody would have written `apply` anyway, because `big["years"] * 1.5 + 2` is
also shorter and clearer.

The honest guidance:

- For **arithmetic**, always write the column expression. It is both faster and more readable, so
  there is no trade-off to weigh.
- For **branching logic**, `np.select` is meaningfully faster on large data, and `apply` with a
  named function is easier to read and extend. Pick based on size: below a few hundred thousand
  rows the difference is a few milliseconds and readability should win.
- "Use `apply` because it looks professional" and "never use `apply`" are both wrong.

### Binning a numeric column with `pd.cut`

When the categories are ranges of a number, `pd.cut` states the intent more clearly than a chain
of conditions.

In [79]:
work["experience_band"] = pd.cut(
    work["years_experience"],
    bins=[0, 3, 7, 12, 100],
    labels=["0-3", "4-7", "8-12", "13+"],
)

print(work["experience_band"].value_counts().sort_index())
print("\ndtype:", work["experience_band"].dtype)

experience_band
0-3     12
4-7     16
8-12    18
13+     14
Name: count, dtype: int64

dtype: category


The bins are the boundaries, so four labels need five boundaries. By default the intervals are
"left-open, right-closed" — `(0, 3]` means "more than 0, up to and including 3". Pass `right=False`
to flip that. Getting this backwards puts values in neighbouring bins, so check the counts against
something you know.

The result is a **categorical** column, which is an ordered set of fixed values. That is why
`.sort_index()` produced the bands in their natural order rather than alphabetically.

`pd.qcut` is the sibling that splits by quantile instead — equal numbers of rows per bin rather
than equal-width ranges:

In [80]:
work["salary_quartile"] = pd.qcut(work["salary"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(work["salary_quartile"].value_counts().sort_index())

salary_quartile
Q1    15
Q2    15
Q3    15
Q4    15
Name: count, dtype: int64


### `assign` — adding columns without mutating

In [81]:
summary = (
    staff
    .assign(
        monthly=lambda d: d["salary"] / 12,
        senior=lambda d: d["years_experience"] >= 10,
        salary_lakhs=lambda d: (d["salary"] / 100_000).round(1),
    )
    [["name", "salary_lakhs", "monthly", "senior"]]
)

print(summary.head(3))

            name  salary_lakhs       monthly  senior
0  Shalini Reddy          10.2  84833.333333   False
1    Swati Desai          11.2  92916.666667    True
2    Aisha Reddy          11.9  99166.666667   False


`assign` returns a **new** DataFrame with the extra columns; it never modifies the original. That
makes it the natural fit for a chain of operations, where each step hands its result to the next
and nothing is left half-modified if something fails partway.

Read the `lambda` here carefully, because this is where `assign` earns its place:

```python
.assign(monthly=lambda d: d["salary"] / 12)
                 ↑      ↑
                 |      the DataFrame as it exists at this point in the chain
                 a function called with that DataFrame
```

Why not just write `monthly=staff["salary"] / 12`? Because inside a chain, `staff` refers to the
*original* DataFrame, not to the filtered or modified version flowing through the chain. The
`lambda` receives whatever the previous step produced. This matters as soon as you filter first:

In [82]:
chained = (
    staff
    .query("department == 'Engineering'")
    .assign(share_of_dept=lambda d: d["salary"] / d["salary"].sum())
    .sort_values("share_of_dept", ascending=False)
    [["name", "salary", "share_of_dept"]]
)

print(chained.head(3))
print("shares sum to:", chained["share_of_dept"].sum())

            name   salary  share_of_dept
36   Rohan Menon  1727000       0.053712
6   Arjun Sheikh  1707000       0.053090
50   Rohan Reddy  1612000       0.050135
shares sum to: 1.0


`d["salary"].sum()` is the total for Engineering only, because `d` is the already-filtered frame.
Written as `staff["salary"].sum()` it would have been the company-wide total, and the shares would
not add to 1. This is a real bug that `assign` with a lambda prevents.

Also note how the chain reads: filter, then add a column, then sort, then pick columns. Each line
is one verb. The parentheses around the whole expression are what let you break it across lines —
without them Python would end the statement at the first newline.

### When not to use `assign`

For a single column on a DataFrame you are building up step by step, plain assignment is clearer:

```python
work["monthly"] = work["salary"] / 12          # clear
work = work.assign(monthly=work["salary"] / 12) # needlessly indirect
```

Use `assign` inside chains, and direct assignment when you are not chaining. Do not convert
working code to a chain because chains look sophisticated.

### Removing and renaming

In [83]:
trimmed = work.drop(columns=["monthly_loop", "grade_apply", "salary_lakhs"])
print("columns now:", trimmed.columns.tolist())

renamed = trimmed.rename(columns={"years_experience": "experience", "salary": "annual_salary"})
print("\nrenamed:", renamed.columns.tolist())

columns now: ['emp_id', 'name', 'department', 'years_experience', 'salary', 'city', 'remote', 'joined', 'monthly_salary', 'salary_per_year_exp', 'is_senior', 'level', 'grade', 'experience_band', 'salary_quartile']

renamed: ['emp_id', 'name', 'department', 'experience', 'annual_salary', 'city', 'remote', 'joined', 'monthly_salary', 'salary_per_year_exp', 'is_senior', 'level', 'grade', 'experience_band', 'salary_quartile']


`drop(columns=[...])` removes columns; `drop(index=[...])` removes rows by label. Both return a new
DataFrame. You will see `inplace=True` used to modify the original instead — it works, but it is
discouraged: it makes chaining impossible, it saves less memory than people assume, and the return
value being `None` leads to `df = df.drop(...,  inplace=True)` which quietly sets `df` to `None`.
Assign the result instead.

## Missing data

Real datasets have gaps. Our generated one does not, so let us introduce some — in a realistic
pattern, where a few employees have no recorded city and a few have no salary on file.

In [84]:
gappy = staff.copy()

rng_missing = np.random.default_rng(3)
gappy.loc[rng_missing.choice(gappy.index, 6, replace=False), "city"] = np.nan
gappy.loc[rng_missing.choice(gappy.index, 4, replace=False), "salary"] = np.nan
gappy.loc[rng_missing.choice(gappy.index, 3, replace=False), "years_experience"] = np.nan

print(gappy[["name", "department", "years_experience", "salary", "city"]].head(8))
print("\ndtypes after introducing gaps:")
print(gappy[["years_experience", "salary"]].dtypes)

            name   department  years_experience     salary       city
0  Shalini Reddy        Sales               7.0  1018000.0       Pune
1    Swati Desai      Finance               NaN  1115000.0     Mumbai
2    Aisha Reddy      Finance               7.0  1190000.0  Bengaluru
3     Kavita Rao  Engineering              10.0  1458000.0       Pune
4     Vikram Rao  Engineering               9.0  1243000.0        NaN
5     Sameer Rao      Finance              11.0  1239000.0     Mumbai
6   Arjun Sheikh  Engineering               NaN  1707000.0  Hyderabad
7  Manish Sheikh      Finance               3.0   854000.0       Pune

dtypes after introducing gaps:
years_experience    float64
salary              float64
dtype: object


Notice what happened to the dtypes. `years_experience` and `salary` were `int64`; introducing
`NaN` turned them into `float64`, which is why the values now print as `7.0` rather than `7`.

There is no way to store `NaN` in an integer column, because `NaN` is a floating-point concept, so
Pandas widens the column to float. This is expected and mostly harmless, but it explains a
recurring confusion — "why did my integer IDs turn into 1001.0?" The answer is almost always a
missing value somewhere in the column. If you need integers with gaps, the nullable `"Int64"`
dtype (capital I) exists for exactly that:

In [85]:
nullable = pd.Series([3, None, 7], dtype="Int64")
print(nullable)
print("dtype:", nullable.dtype, "| missing marker:", nullable[1])

0       3
1    <NA>
2       7
dtype: Int64
dtype: Int64 | missing marker: <NA>


### Finding it

In [86]:
print("missing per column:")
print(gappy.isna().sum())
print("\ntotal missing cells:", gappy.isna().sum().sum())
print("rows with any gap:", gappy.isna().any(axis=1).sum())
print("rows that are complete:", gappy.notna().all(axis=1).sum())

missing per column:
emp_id              0
name                0
department          0
years_experience    3
salary              4
city                6
remote              0
joined              0
dtype: int64

total missing cells: 13
rows with any gap: 13
rows that are complete: 47


`gappy.isna()` gives a DataFrame of `True`/`False` the same shape as the original. From there:

- `.sum()` counts down each column (the default `axis=0`) — missing values per column.
- `.sum().sum()` adds those up for a grand total.
- `.any(axis=1)` works across each row — "does this row have any gap at all".
- `.notna()` is the opposite of `isna()`; `isnull` and `notnull` are aliases with identical
  behaviour, so pick one spelling and stick to it.

As a proportion, which is usually the more useful framing:

In [87]:
print((gappy.isna().mean() * 100).round(1).astype(str) + "%")

emp_id               0.0%
name                 0.0%
department           0.0%
years_experience     5.0%
salary               6.7%
city                10.0%
remote               0.0%
joined               0.0%
dtype: str


`isna().mean()` gives the fraction missing per column, because the mean of a boolean column is the
share of `True`. A column that is 2% missing and one that is 60% missing call for completely
different decisions.

In [88]:
print("rows where salary is missing:")
print(gappy[gappy["salary"].isna()][["name", "department", "years_experience"]])

rows where salary is missing:
            name   department  years_experience
15   Sameer Nair        Sales               2.0
24    Arjun Bose  Engineering               8.0
28  Aditya Singh        Sales               8.0
36   Rohan Menon  Engineering              15.0


Always look at the rows with gaps before deciding what to do. Sometimes the pattern tells you the
cause — all the missing salaries are in one department, or all the missing dates are from one
month — and that changes the right response.

### Dropping

In [89]:
print("original rows        :", len(gappy))
print("dropna() — any gap   :", len(gappy.dropna()))
print("dropna(subset=salary):", len(gappy.dropna(subset=["salary"])))
print("dropna(how='all')    :", len(gappy.dropna(how="all")))
print("dropna(thresh=7)     :", len(gappy.dropna(thresh=7)))

original rows        : 60
dropna() — any gap   : 47
dropna(subset=salary): 56
dropna(how='all')    : 60
dropna(thresh=7)     : 60


- `dropna()` with no arguments removes every row with **any** missing value. That is aggressive: it
  threw away 13 rows here because of gaps in three different columns, most of which were irrelevant
  to whatever you were about to compute.
- `dropna(subset=["salary"])` only cares about the columns you name. This is usually what you want:
  drop rows that lack the thing you are analysing, keep the rest.
- `how="all"` removes only rows that are entirely empty — a common artefact of trailing lines in
  spreadsheets.
- `thresh=7` keeps rows with at least 7 non-null values.

The habit to build: name the columns you need. `dropna()` bare is a blunt instrument that silently
shrinks your dataset.

### Filling

In [90]:
filled_zero = gappy["salary"].fillna(0)
filled_mean = gappy["salary"].fillna(gappy["salary"].mean())
filled_median = gappy["salary"].fillna(gappy["salary"].median())

print("original mean (gaps skipped):", round(gappy["salary"].mean(), 0))
print("after filling with 0        :", round(filled_zero.mean(), 0))
print("after filling with the mean :", round(filled_mean.mean(), 0))
print("after filling with median   :", round(filled_median.mean(), 0))

original mean (gaps skipped): 1153946.0
after filling with 0        : 1077017.0
after filling with the mean : 1153946.0
after filling with median   : 1153250.0


This little table is the argument for thinking before filling.

Filling salary with **0** moved the average down by tens of thousands. It asserts that four
employees earn nothing, which is false, and it will distort every statistic you compute afterwards.

Filling with the **mean** leaves the mean unchanged — which sounds ideal and is actually the
problem. You have invented four data points that are perfectly average, which makes the data look
more certain than it is: the standard deviation shrinks, and any model trained on it will be
overconfident.

In [91]:
print("std with gaps left alone:", round(gappy["salary"].std(), 0))
print("std after mean-filling  :", round(filled_mean.std(), 0))

std with gaps left alone: 264330.0
std after mean-filling  : 255213.0


The spread dropped, and nothing about the real world changed. That is fabricated certainty.

Filling with the **median** is more robust to outliers than the mean, and is a reasonable default
when you must fill.

A better option when the data has structure: fill from the group the row belongs to.

In [92]:
by_dept_median = gappy.groupby("department")["salary"].transform("median")
smarter_fill = gappy["salary"].fillna(by_dept_median)

print(pd.DataFrame({
    "department": gappy.loc[gappy["salary"].isna(), "department"],
    "filled_with_overall_median": gappy["salary"].median(),
    "filled_with_dept_median": smarter_fill[gappy["salary"].isna()],
}))

     department  filled_with_overall_median  filled_with_dept_median
15        Sales                   1143500.0                 985000.0
24  Engineering                   1143500.0                1243000.0
28        Sales                   1143500.0                 985000.0
36  Engineering                   1143500.0                1243000.0


`transform("median")` computes each department's median and returns it **aligned to the original
rows** — so every row gets its own department's median, ready to slot into the gap. An HR employee
gets the HR median rather than the company-wide one. `transform` is covered properly in the
GroupBy section; this is a preview of why it matters.

### Forward and backward fill

In [93]:
sensor = pd.Series([21.5, np.nan, np.nan, 23.1, np.nan, 22.8],
                   index=pd.date_range("2024-03-01", periods=6, freq="h"))

print("raw:"); print(sensor)
print("\nffill (carry the last reading forward):"); print(sensor.ffill())
print("\nbfill (use the next reading):"); print(sensor.bfill())
print("\ninterpolate (straight line between known points):"); print(sensor.interpolate().round(2))

raw:
2024-03-01 00:00:00    21.5
2024-03-01 01:00:00     NaN
2024-03-01 02:00:00     NaN
2024-03-01 03:00:00    23.1
2024-03-01 04:00:00     NaN
2024-03-01 05:00:00    22.8
Freq: h, dtype: float64

ffill (carry the last reading forward):
2024-03-01 00:00:00    21.5
2024-03-01 01:00:00    21.5
2024-03-01 02:00:00    21.5
2024-03-01 03:00:00    23.1
2024-03-01 04:00:00    23.1
2024-03-01 05:00:00    22.8
Freq: h, dtype: float64

bfill (use the next reading):
2024-03-01 00:00:00    21.5
2024-03-01 01:00:00    23.1
2024-03-01 02:00:00    23.1
2024-03-01 03:00:00    23.1
2024-03-01 04:00:00    22.8
2024-03-01 05:00:00    22.8
Freq: h, dtype: float64

interpolate (straight line between known points):
2024-03-01 00:00:00    21.50
2024-03-01 01:00:00    22.03
2024-03-01 02:00:00    22.57
2024-03-01 03:00:00    23.10
2024-03-01 04:00:00    22.95
2024-03-01 05:00:00    22.80
Freq: h, dtype: float64


These only make sense for **ordered** data, typically a time series. For a temperature sensor
reporting every hour, carrying the last reading forward is defensible, and interpolating between
known readings is often better still.

For our employee table none of them make sense: the rows are in no meaningful order, so
"the previous row's salary" is not information about this employee. Applying `ffill` to unordered
data is a common and invisible mistake.

`ffill` also has a limit parameter, worth using so that one reading does not get carried across a
day-long outage:

In [94]:
print(sensor.ffill(limit=1))

2024-03-01 00:00:00    21.5
2024-03-01 01:00:00    21.5
2024-03-01 02:00:00     NaN
2024-03-01 03:00:00    23.1
2024-03-01 04:00:00    23.1
2024-03-01 05:00:00    22.8
Freq: h, dtype: float64


### The decision table

| Situation | Reasonable approach |
| --- | --- |
| A few rows missing the value you are analysing | Drop those rows (`dropna(subset=...)`) |
| Missing means "none happened" (e.g. no purchases) | Fill with 0 |
| Numeric, no structure to exploit, must keep the rows | Fill with the median |
| Numeric, with meaningful groups | Fill with the group's median via `transform` |
| Time series with occasional dropouts | `ffill` with a limit, or `interpolate` |
| Category missing | Fill with an explicit `"Unknown"` — do not guess |
| More than half the column missing | Consider dropping the column; ask why it is empty |

And the rule underneath all of it: whatever you do, say so. A filled value is your assumption, not
data, and anyone reading your results needs to know which is which.

In [95]:
gappy["city"] = gappy["city"].fillna("Unknown")
print(gappy["city"].value_counts())

city
Bengaluru    18
Hyderabad    12
Pune         11
Mumbai       10
Unknown       6
Delhi         3
Name: count, dtype: int64


Filling a category with `"Unknown"` is honest: it keeps the rows, and it makes the gap visible in
every later count and grouping rather than hiding it.

## Sorting

In [96]:
print(staff.sort_values("salary", ascending=False)[["name", "department", "salary"]].head(5))

            name   department   salary
36   Rohan Menon  Engineering  1727000
6   Arjun Sheikh  Engineering  1707000
43  Deepak Menon      Finance  1658000
50   Rohan Reddy  Engineering  1612000
33    Rohan Nair  Engineering  1593000


`sort_values` takes a column name and returns a new, reordered DataFrame. Note that the index comes
along — the labels are still the original row numbers, which is how you can tell this is the same
data reordered and not a new dataset.

In [97]:
print(staff.sort_values(["department", "salary"], ascending=[True, False])
      [["department", "name", "salary"]].head(8))

     department            name   salary
36  Engineering     Rohan Menon  1727000
6   Engineering    Arjun Sheikh  1707000
50  Engineering     Rohan Reddy  1612000
21  Engineering  Arjun Kulkarni  1593000
33  Engineering      Rohan Nair  1593000
35  Engineering    Sameer Gupta  1592000
3   Engineering      Kavita Rao  1458000
31  Engineering     Rohan Verma  1396000


Passing a list sorts by several columns in turn: department alphabetically, and within each
department, salary from high to low. The `ascending` list has one entry per sort column, which is
how you mix directions.

This "sort by group, then by value within group" pattern is how you produce a readable ranked
report without any grouping machinery.

In [98]:
print("sorted by index:")
print(staff.sort_values("salary").sort_index().head(3)[["name", "salary"]])

print("\nreset the index after sorting:")
print(staff.sort_values("salary", ascending=False)
      .reset_index(drop=True)[["name", "salary"]].head(3))

sorted by index:
            name   salary
0  Shalini Reddy  1018000
1    Swati Desai  1115000
2    Aisha Reddy  1190000

reset the index after sorting:
           name   salary
0   Rohan Menon  1727000
1  Arjun Sheikh  1707000
2  Deepak Menon  1658000


`sort_index()` puts the rows back in index order. `reset_index(drop=True)` renumbers from 0 and
throws the old labels away — use it when the original row numbers no longer mean anything, and
leave them when they do.

### `nlargest` — sorting when you only want the top few

In [99]:
print(staff.nlargest(5, "salary")[["name", "department", "salary"]])
print("\nsmallest three:")
print(staff.nsmallest(3, "salary")[["name", "salary"]])

              name   department   salary
36     Rohan Menon  Engineering  1727000
6     Arjun Sheikh  Engineering  1707000
43    Deepak Menon      Finance  1658000
50     Rohan Reddy  Engineering  1612000
21  Arjun Kulkarni  Engineering  1593000

smallest three:
             name  salary
22  Shalini Joshi  662000
26    Rajesh Bose  714000
19   Sameer Verma  728000


`nlargest(5, "salary")` is equivalent to `sort_values("salary", ascending=False).head(5)` and says
what it means in fewer moving parts. On a large table it is also faster, because finding the top 5
does not require ordering all million rows.

## Exercises — filtering, columns, missing data, sorting

**Level 1.** From `staff`, select employees in Bengaluru with more than 5 years of experience.
Then add a column `salary_monthly` and display the five highest monthly salaries with the
employee's name.

**Level 2.** Create a column `pay_band` with the values `"Low"` (below 800,000), `"Medium"`
(800,000 to 1,200,000) and `"High"` (above 1,200,000). Do it twice: once with `np.select` and once
with `pd.cut`. Check the two agree, then count how many employees fall in each band per department.

**Level 3.** Take `gappy`. Write a short function `report_missing(df)` that returns a DataFrame with
one row per column, showing the number and percentage of missing values, sorted worst first, and
omitting columns with no gaps. Then decide and justify a filling strategy for each affected column —
your justification matters more than the code.

## GroupBy

If you learn one thing from this notebook properly, make it this. "Split the data into groups, do
something to each group, put the results back together" covers an enormous share of real analysis
work, and Pandas has a direct expression for it.

The pattern has a standard name: **split–apply–combine**.

```text
                  department   salary
                  Engineering  1458000  ┐
  staff           Engineering  1243000  ├─ split by department
                  Sales        1018000  ┘
                  ...

  split      →    Engineering: [1458000, 1243000, ...]
                  Sales:       [1018000, ...]
                  Marketing:   [...]

  apply      →    mean of each group

  combine    →    Engineering  <one number>
                  Marketing    <one number>
                  Sales        <one number>
```

(The exact figures come out of the cells below — the diagram is only about the shape of the
operation.)

### Start with the beginner version

Before reaching for `groupby`, it is worth writing the loop once, so you know exactly what
`groupby` replaces.

In [100]:
totals = {}
counts = {}

for i in range(len(staff)):
    dept = staff["department"].iloc[i]
    pay = staff["salary"].iloc[i]
    if dept not in totals:
        totals[dept] = 0
        counts[dept] = 0
    totals[dept] += pay
    counts[dept] += 1

for dept in sorted(totals):
    print(f"{dept:12} {totals[dept] / counts[dept]:10,.0f}")

Engineering   1,286,120
Finance       1,211,273
HR              918,000
Marketing     1,046,200
Sales           979,000


Two dictionaries, an existence check, two accumulators, and a second loop to divide. It works, and
it is the mental model you should keep — that is genuinely what Pandas does internally.

Now the Pandas version:

In [101]:
print(staff.groupby("department")["salary"].mean().round(0))

department
Engineering    1286120.0
Finance        1211273.0
HR              918000.0
Marketing      1046200.0
Sales           979000.0
Name: salary, dtype: float64


One line. Read it left to right:

```python
staff.groupby("department")  ["salary"]  .mean()
      └── split by this ──┘  └ pick ─┘   └ apply ┘
                             the column
```

- `staff.groupby("department")` splits the rows into groups, one per distinct department.
- `["salary"]` narrows to the column we want summarised. Without it you would get every numeric
  column, which is sometimes exactly what you want.
- `.mean()` is applied to each group separately, and the results are combined into a Series
  indexed by department.

### The GroupBy object itself

`groupby` on its own does not compute anything. It returns a lazy object describing the split.

In [102]:
grouped = staff.groupby("department")

print(type(grouped))
print("number of groups:", grouped.ngroups)
print("group sizes:")
print(grouped.size())

<class 'pandas.api.typing.DataFrameGroupBy'>
number of groups: 5
group sizes:
department
Engineering    25
Finance        11
HR              4
Marketing      10
Sales          10
dtype: int64


That laziness is deliberate: splitting into groups is cheap, and Pandas waits to see what you want
computed before doing any real work.

You can inspect a group directly, which is useful when a result looks wrong:

In [103]:
print("the groups and which rows they contain (first two):")
for name, rows in list(grouped.groups.items())[:2]:
    print(f"  {name}: {list(rows)[:5]} ...")

print("\none group as a DataFrame:")
print(grouped.get_group("HR")[["name", "years_experience", "salary"]])

the groups and which rows they contain (first two):
  Engineering: [3, 4, 6, 10, 11] ...
  Finance: [1, 2, 5, 7, 8] ...

one group as a DataFrame:
            name  years_experience   salary
16  Aditya Mehta                 7  1036000
19  Sameer Verma                 4   728000
27    Ananya Rao                10  1107000
58   Swati Singh                 4   801000


And you can loop over the groups, which is occasionally the clearest way to do something that does
not fit the aggregation model:

In [104]:
for dept, group in staff.groupby("department"):
    top = group.loc[group["salary"].idxmax()]
    print(f"{dept:12} top earner: {top['name']:20} {top['salary']:,}")

Engineering  top earner: Rohan Menon          1,727,000
Finance      top earner: Deepak Menon         1,658,000
HR           top earner: Ananya Rao           1,107,000
Marketing    top earner: Arjun Verma          1,358,000
Sales        top earner: Suresh Bose          1,497,000


Each iteration gives you the group's key and the group's rows as a DataFrame. This is a legitimate
tool — not something to feel guilty about — but if what you are doing inside the loop is an
aggregation, there is almost certainly a one-liner for it.

### One aggregation at a time

In [105]:
print("mean  :"); print(staff.groupby("department")["salary"].mean().round(0))
print("\ncount :"); print(staff.groupby("department")["salary"].count())
print("\nmedian:"); print(staff.groupby("department")["salary"].median())
print("\nmax   :"); print(staff.groupby("department")["salary"].max())

mean  :
department
Engineering    1286120.0
Finance        1211273.0
HR              918000.0
Marketing      1046200.0
Sales           979000.0
Name: salary, dtype: float64

count :
department
Engineering    25
Finance        11
HR              4
Marketing      10
Sales          10
Name: salary, dtype: int64

median:
department
Engineering    1251000.0
Finance        1221000.0
HR              918500.0
Marketing       943000.0
Sales           970000.0
Name: salary, dtype: float64

max   :
department
Engineering    1727000
Finance        1658000
HR             1107000
Marketing      1358000
Sales          1497000
Name: salary, dtype: int64


All the usual aggregations work: `sum`, `mean`, `median`, `min`, `max`, `count`, `std`, `var`,
`first`, `last`, `nunique`. Each returns one value per group.

Watch the difference between `count` and `size`:

In [106]:
gappy_grouped = gappy.groupby("department")["salary"]
print(pd.DataFrame({"size": gappy.groupby("department").size(),
                    "count": gappy_grouped.count()}))

             size  count
department              
Engineering    25     23
Finance        11     11
HR              4      4
Marketing      10     10
Sales          10      8


`size()` counts rows; `count()` counts non-missing values in the column. When they differ, you have
missing data — and if you are reporting an average, `count` is the number it was actually computed
from. Reporting a departmental average based on two of eleven employees without saying so is how
analyses mislead.

### Several aggregations at once

In [107]:
print(staff.groupby("department")["salary"].agg(["count", "mean", "median", "min", "max"]).round(0))

             count       mean     median     min      max
department                                               
Engineering     25  1286120.0  1251000.0  903000  1727000
Finance         11  1211273.0  1221000.0  854000  1658000
HR               4   918000.0   918500.0  728000  1107000
Marketing       10  1046200.0   943000.0  714000  1358000
Sales           10   979000.0   970000.0  662000  1497000


`agg` with a list of function names gives one column per function. This is the compact form of
running four separate `groupby` lines, and it guarantees they are all computed on the same split.

The column names come straight from the function names, which is fine for a quick look and poor
for a report. Which brings us to the version you should prefer.

### Named aggregation — the readable form

In [108]:
summary = staff.groupby("department").agg(
    headcount=("emp_id", "count"),
    avg_salary=("salary", "mean"),
    highest_salary=("salary", "max"),
    avg_experience=("years_experience", "mean"),
    remote_share=("remote", "mean"),
)

print(summary.round(2))

             headcount  avg_salary  highest_salary  avg_experience  \
department                                                           
Engineering         25  1286120.00         1727000            8.16   
Finance             11  1211272.73         1658000            9.64   
HR                   4   918000.00         1107000            6.25   
Marketing           10  1046200.00         1358000            7.90   
Sales               10   979000.00         1497000            7.00   

             remote_share  
department                 
Engineering          0.56  
Finance              0.27  
HR                   1.00  
Marketing            0.50  
Sales                0.10  


This is the form worth memorising, so let us take it apart completely.

```python
staff.groupby("department").agg(
    avg_salary = ( "salary" , "mean" ),
    └─ new column ─┘ └ source ┘ └ how ┘
        name          column
)
```

- Each **keyword argument name** becomes a column in the result. You choose it, so choose something
  a reader will understand — `avg_salary`, not `salary_mean_1`.
- Each **value is a tuple** of `(column to aggregate, how to aggregate it)`.
- The "how" can be a string name like `"mean"`, or any function.

So the whole call reads: "group by department; in each group, count the emp_ids and call that
`headcount`, average the salaries and call that `avg_salary`, ...".

Why is this better than the list form? Because the output columns are named for what they mean,
different source columns can be aggregated differently in one call, and the code documents itself.

One subtlety worth noticing: `remote_share=("remote", "mean")` works because `remote` is boolean,
and the mean of a boolean column is the proportion that are `True`. That is the same trick as in
NumPy, and it comes up constantly.

### Getting there gradually

If the named form above looks like a lot at once, it decomposes exactly as you would hope:

In [109]:
grouped = staff.groupby("department")

headcount = grouped["emp_id"].count()
avg_salary = grouped["salary"].mean()
highest = grouped["salary"].max()

step_by_step = pd.DataFrame({
    "headcount": headcount,
    "avg_salary": avg_salary,
    "highest_salary": highest,
})

print(step_by_step.round(0))
print("\nsame as the agg version:",
      np.allclose(step_by_step["avg_salary"], summary["avg_salary"]))

             headcount  avg_salary  highest_salary
department                                        
Engineering         25   1286120.0         1727000
Finance             11   1211273.0         1658000
HR                   4    918000.0         1107000
Marketing           10   1046200.0         1358000
Sales               10    979000.0         1497000

same as the agg version: True


Three separate aggregations assembled into a DataFrame by hand. Identical result, and the reason
it works is that all three Series share the same index — the department names — so Pandas aligns
them automatically.

There is nothing wrong with writing it this way, especially while you are learning. The `agg`
version is preferable mainly because it splits the data once instead of three times, and because
it keeps one idea in one statement.

### A custom aggregation

If no built-in does what you need, pass a function. It receives each group's values as a Series.

In [110]:
def salary_spread(values):
    """Gap between the highest and lowest paid person in the group, in lakhs."""
    return (values.max() - values.min()) / 100_000


print(staff.groupby("department")["salary"].agg(salary_spread).round(2))

department
Engineering    8.24
Finance        8.04
HR             3.79
Marketing      6.44
Sales          8.35
Name: salary, dtype: float64


You can mix built-ins and custom functions in a named aggregation:

In [111]:
print(staff.groupby("department").agg(
    avg_salary=("salary", "mean"),
    spread_lakhs=("salary", salary_spread),
    seniors=("years_experience", lambda s: (s >= 10).sum()),
).round(2))

             avg_salary  spread_lakhs  seniors
department                                    
Engineering  1286120.00          8.24        8
Finance      1211272.73          8.04        6
HR            918000.00          3.79        1
Marketing    1046200.00          6.44        4
Sales         979000.00          8.35        3


`lambda s: (s >= 10).sum()` is a small anonymous function: given the group's experience values,
count how many are at least 10. A `lambda` is appropriate here because the logic is one short
expression used once. If it grows beyond that, give it a name with `def` — a named function can be
tested and reused, and the name explains the intent.

### Grouping by several columns

In [112]:
multi = staff.groupby(["department", "remote"])["salary"].mean().round(0)
print(multi)

department   remote
Engineering  False     1239091.0
             True      1323071.0
Finance      False     1183375.0
             True      1285667.0
HR           True       918000.0
Marketing    False     1142600.0
             True       949800.0
Sales        False     1000889.0
             True       782000.0
Name: salary, dtype: float64


The result has a **MultiIndex** — two levels of labels, department and remote. Reading it is fine;
working with it takes a little more care.

In [113]:
print("shape:", multi.shape)
print("index levels:", multi.index.names)
print("\none combination:", multi.loc[("Engineering", True)])

shape: (9,)
index levels: ['department', 'remote']

one combination: 1323071.0


Often the easiest thing is to flatten it back into a normal table. Two ways, for two different
purposes:

In [114]:
print("unstacked into a grid:")
print(multi.unstack().round(0))

print("\nflat table with reset_index:")
print(staff.groupby(["department", "remote"])["salary"].mean().round(0).reset_index())

unstacked into a grid:
remote           False      True 
department                       
Engineering  1239091.0  1323071.0
Finance      1183375.0  1285667.0
HR                 NaN   918000.0
Marketing    1142600.0   949800.0
Sales        1000889.0   782000.0

flat table with reset_index:
    department  remote     salary
0  Engineering   False  1239091.0
1  Engineering    True  1323071.0
2      Finance   False  1183375.0
3      Finance    True  1285667.0
4           HR    True   918000.0
5    Marketing   False  1142600.0
6    Marketing    True   949800.0
7        Sales   False  1000889.0
8        Sales    True   782000.0


`unstack()` moves the innermost index level into columns, giving a cross-tab — good for reading and
for plotting. `reset_index()` turns the index levels back into ordinary columns, giving a long,
tidy table — good for feeding into other operations.

You can also ask `groupby` not to build the index in the first place:

In [115]:
print(staff.groupby(["department", "remote"], as_index=False)["salary"].mean().round(0).head())

    department  remote     salary
0  Engineering   False  1239091.0
1  Engineering    True  1323071.0
2      Finance   False  1183375.0
3      Finance    True  1285667.0
4           HR    True   918000.0


`as_index=False` keeps the grouping keys as columns. Whether you prefer it is taste; it saves a
`reset_index()` call and makes the result feel more like a plain table.

### `agg` versus `transform` — the distinction that matters most

This trips up nearly everyone, and the difference is simple once you see it in terms of shape.

- **`agg`** returns **one row per group**. The result is smaller than the input.
- **`transform`** returns **one row per original row**. The result is the same size as the input,
  with each row carrying its group's value.

In [116]:
agg_result = staff.groupby("department")["salary"].mean()
transform_result = staff.groupby("department")["salary"].transform("mean")

print("agg      → shape", agg_result.shape)
print(agg_result.round(0))
print("\ntransform → shape", transform_result.shape)
print(transform_result.head(6).round(0))

agg      → shape (5,)
department
Engineering    1286120.0
Finance        1211273.0
HR              918000.0
Marketing      1046200.0
Sales           979000.0
Name: salary, dtype: float64

transform → shape (60,)
0     979000.0
1    1211273.0
2    1211273.0
3    1286120.0
4    1286120.0
5    1211273.0
Name: salary, dtype: float64


Same numbers, different shape. `transform` gave every one of the 60 employees the average for
their own department, which is precisely what you need to compare an individual to their group.

In [117]:
compare = staff.assign(
    dept_average=lambda d: d.groupby("department")["salary"].transform("mean").round(0),
    difference=lambda d: d["salary"] - d.groupby("department")["salary"].transform("mean"),
    pct_of_dept_avg=lambda d: (
        d["salary"] / d.groupby("department")["salary"].transform("mean") * 100
    ).round(1),
)

print(compare[["name", "department", "salary", "dept_average", "difference", "pct_of_dept_avg"]]
      .head(8).to_string())

            name   department   salary  dept_average     difference  pct_of_dept_avg
0  Shalini Reddy        Sales  1018000      979000.0   39000.000000            104.0
1    Swati Desai      Finance  1115000     1211273.0  -96272.727273             92.1
2    Aisha Reddy      Finance  1190000     1211273.0  -21272.727273             98.2
3     Kavita Rao  Engineering  1458000     1286120.0  171880.000000            113.4
4     Vikram Rao  Engineering  1243000     1286120.0  -43120.000000             96.6
5     Sameer Rao      Finance  1239000     1211273.0   27727.272727            102.3
6   Arjun Sheikh  Engineering  1707000     1286120.0  420880.000000            132.7
7  Manish Sheikh      Finance   854000     1211273.0 -357272.727273             70.5


That table — "how does this person compare to their own department" — is impossible to build with
`agg` alone, and it is one of the most frequently asked questions about any grouped dataset.

The beginner route to the same result is a merge, and it is worth seeing because it shows what
`transform` saves you:

In [118]:
dept_means = staff.groupby("department")["salary"].mean().reset_index()
dept_means.columns = ["department", "dept_average"]

merged = staff.merge(dept_means, on="department")
merged["difference"] = merged["salary"] - merged["dept_average"]

print(merged[["name", "department", "salary", "dept_average", "difference"]].head(4).to_string())

            name   department   salary  dept_average     difference
0  Shalini Reddy        Sales  1018000  9.790000e+05   39000.000000
1    Swati Desai      Finance  1115000  1.211273e+06  -96272.727273
2    Aisha Reddy      Finance  1190000  1.211273e+06  -21272.727273
3     Kavita Rao  Engineering  1458000  1.286120e+06  171880.000000


Compute the group means into a small table, then join it back on the grouping key. Three steps
instead of one, and a merge that you have to get right. `transform` does exactly this internally,
which is why the shapes always line up.

The mental shortcut:

> Asking about **groups**? `agg`.
> Asking about **rows, in the context of their group**? `transform`.

In [119]:
print("other useful transforms:")
print(pd.DataFrame({
    "salary": staff["salary"],
    "dept_rank": staff.groupby("department")["salary"].rank(ascending=False),
    "dept_max": staff.groupby("department")["salary"].transform("max"),
    "z_in_dept": (
        (staff["salary"] - staff.groupby("department")["salary"].transform("mean"))
        / staff.groupby("department")["salary"].transform("std")
    ).round(2),
}).head(6))

other useful transforms:
    salary  dept_rank  dept_max  z_in_dept
0  1018000        4.0   1497000       0.17
1  1115000        8.0   1658000      -0.40
2  1190000        7.0   1658000      -0.09
3  1458000        7.0   1727000       0.69
4  1243000       14.0   1727000      -0.17
5  1239000        4.0   1658000       0.11


`rank` inside a group gives each employee's position within their own department — "third
highest-paid engineer". The z-score version standardises within the group, which is how you
compare across departments fairly.

### `filter` — keeping whole groups

In [120]:
big_departments = staff.groupby("department").filter(lambda g: len(g) >= 10)

print("original rows:", len(staff))
print("rows kept    :", len(big_departments))
print("departments kept:", big_departments["department"].unique().tolist())

original rows: 60
rows kept    : 56
departments kept: ['Sales', 'Finance', 'Engineering', 'Marketing']


`groupby(...).filter(fn)` keeps or discards **entire groups** based on a test. The function receives
each group as a DataFrame and returns `True` to keep it. Here we kept only departments with at
least ten employees.

This is different from filtering rows. `staff[staff["department"] == "Engineering"]` selects rows by
their own values; `filter` selects rows by a property of the group they belong to — "employees in
departments whose average salary exceeds a million", for instance:

In [121]:
well_paid_departments = staff.groupby("department").filter(lambda g: g["salary"].mean() > 1_000_000)
print(well_paid_departments["department"].unique().tolist())

['Finance', 'Engineering', 'Marketing']


### `apply` on a GroupBy

When neither `agg`, `transform` nor `filter` fits, `apply` gives you the whole group as a DataFrame
and lets you return whatever you like.

In [122]:
def top_two_earners(group):
    return group.nlargest(2, "salary")[["name", "salary"]]


print(staff.groupby("department").apply(top_two_earners, include_groups=False).to_string())

                        name   salary
department                           
Engineering 36   Rohan Menon  1727000
            6   Arjun Sheikh  1707000
Finance     43  Deepak Menon  1658000
            56   Priya Joshi  1583000
HR          27    Ananya Rao  1107000
            16  Aditya Mehta  1036000
Marketing   42   Arjun Verma  1358000
            30   Priya Menon  1357000
Sales       18   Suresh Bose  1497000
            51  Sameer Reddy  1146000


Each group is passed in as a DataFrame, and the returned pieces are stitched together. This handles
things the other methods cannot — "top two per group" needs more than one row per group, so `agg`
is out, and it is not the same shape as the input, so `transform` is out.

`include_groups=False` tells Pandas not to pass the grouping column into your function, since
operating on it would be meaningless. In older Pandas this was the default and produced a
deprecation warning; being explicit is now the clean way to write it.

`groupby.apply` is flexible and slow — it runs your Python function once per group. With five
departments that is nothing. With fifty thousand customer IDs it is worth finding another way.

### A worked alternative to `groupby.apply`

The same "top two per department" result without `apply`:

In [123]:
ranked = staff.assign(
    rank_in_dept=lambda d: d.groupby("department")["salary"].rank(ascending=False, method="first")
)

print(ranked[ranked["rank_in_dept"] <= 2]
      .sort_values(["department", "rank_in_dept"])
      [["department", "name", "salary", "rank_in_dept"]]
      .to_string(index=False))

 department         name  salary  rank_in_dept
Engineering  Rohan Menon 1727000           1.0
Engineering Arjun Sheikh 1707000           2.0
    Finance Deepak Menon 1658000           1.0
    Finance  Priya Joshi 1583000           2.0
         HR   Ananya Rao 1107000           1.0
         HR Aditya Mehta 1036000           2.0
  Marketing  Arjun Verma 1358000           1.0
  Marketing  Priya Menon 1357000           2.0
      Sales  Suresh Bose 1497000           1.0
      Sales Sameer Reddy 1146000           2.0


Rank within each group, then filter on the rank. Two vectorised operations instead of one Python
call per group, and arguably clearer about what "top two" means.

`method="first"` decides what happens with ties: the row that appears first wins. The default,
`"average"`, would give two tied employees rank 1.5, and then `<= 2` would keep both — fine, until
someone asks why there are three rows in a "top two" report.

## Aggregation without grouping

The same functions work on a whole DataFrame or column, which is worth listing in one place.

In [124]:
print("column aggregations:")
print(staff[["salary", "years_experience"]].agg(["mean", "median", "std", "min", "max"]).round(1))

print("\ndifferent function per column:")
print(staff.agg({"salary": ["mean", "max"], "years_experience": ["mean"], "remote": ["sum"]}))

column aggregations:
           salary  years_experience
mean    1156683.3               8.1
median  1143500.0               8.0
std      271635.1               4.5
min      662000.0               1.0
max     1727000.0              15.0

different function per column:
            salary  years_experience  remote
mean  1.156683e+06          8.066667     NaN
max   1.727000e+06               NaN     NaN
sum            NaN               NaN    27.0


`staff.agg({...})` with a dictionary applies different functions to different columns. The gaps in
the output (`NaN`) are where a function was not requested for that column.

In [125]:
print("describe on one column:")
print(staff["salary"].describe().round(0))
print("\nquantiles:")
print(staff["salary"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(0))

describe on one column:
count         60.0
mean     1156683.0
std       271635.0
min       662000.0
25%       938250.0
50%      1143500.0
75%      1321000.0
max      1727000.0
Name: salary, dtype: float64

quantiles:
0.10     829200.0
0.25     938250.0
0.50    1143500.0
0.75    1321000.0
0.90    1592100.0
Name: salary, dtype: float64


`quantile` is how you get percentiles: `quantile(0.9)` is the value below which 90% of the data
falls. Reporting the median and the quartiles is usually more honest than the mean alone, because
it does not hide skew.

## `apply`, `map`, and their proper uses

Three functions with similar names, applied to different things. The confusion is understandable;
the rules are short.

| Method | Applies to | Called with | Returns |
| --- | --- | --- | --- |
| `Series.map` | a Series | each **value** | a Series |
| `Series.apply` | a Series | each **value** | a Series |
| `DataFrame.apply` | a DataFrame | each **column** (or row with `axis=1`) | Series or DataFrame |
| `DataFrame.map` | a DataFrame | each **cell** | a DataFrame |

### `Series.map` with a dictionary

This is `map`'s best use, and it has no `apply` equivalent that is as clear:

In [126]:
dept_codes = {
    "Engineering": "ENG",
    "Marketing": "MKT",
    "Sales": "SLS",
    "Finance": "FIN",
    "HR": "HR",
}

print(staff["department"].map(dept_codes).head())
print("\nvalue counts of the codes:")
print(staff["department"].map(dept_codes).value_counts())

0    SLS
1    FIN
2    FIN
3    ENG
4    ENG
Name: department, dtype: str

value counts of the codes:
department
ENG    25
FIN    11
SLS    10
MKT    10
HR      4
Name: count, dtype: int64


`map` with a dictionary is a lookup table applied to a column. Any value not in the dictionary
becomes `NaN` — which is a useful safety property, because it makes typos visible rather than
silently passing the original value through:

In [127]:
incomplete = {"Engineering": "ENG", "Marketing": "MKT"}
print(staff["department"].map(incomplete).isna().sum(), "values could not be mapped")

25 values could not be mapped


### `Series.apply` with a function

In [128]:
def salary_label(amount):
    if amount >= 1_500_000:
        return "very high"
    if amount >= 1_000_000:
        return "high"
    return "moderate"


print(staff["salary"].apply(salary_label).value_counts())

salary
high         32
moderate     20
very high     8
Name: count, dtype: int64


`apply` on a Series calls your function once per value. For simple transformations there is usually
a vectorised alternative that is faster and no harder to read — `np.select` for branching,
`.str` methods for text, arithmetic for numbers. Use `apply` when the logic genuinely does not
vectorise, or when the column is small and clarity wins.

### `DataFrame.apply` — one column at a time

In [129]:
print(staff[["salary", "years_experience"]].apply(lambda col: col.max() - col.min()))

salary              1065000
years_experience         14
dtype: int64


By default `DataFrame.apply` hands your function each **column** as a Series. So the lambda above
received the salary column, then the experience column, and the result is one value per column.

### `DataFrame.apply(axis=1)` — one row at a time

In [130]:
def describe_employee(row):
    return f"{row['name']} ({row['department']}, {row['years_experience']}y)"


print(staff.apply(describe_employee, axis=1).head(3).to_list())

['Shalini Reddy (Sales, 7y)', 'Swati Desai (Finance, 10y)', 'Aisha Reddy (Finance, 7y)']


With `axis=1` your function receives each **row** as a Series, indexed by column name. This is the
only way to write logic that needs several columns of the same row and cannot be expressed as
column arithmetic.

It is also the slowest thing in common Pandas use, because it builds a Series object for every
single row. Before using it, check whether the operation is really row-wise:

In [131]:
# Row-wise, via apply
slow = staff.apply(lambda r: r["salary"] / r["years_experience"], axis=1)

# The same thing, vectorised
fast = staff["salary"] / staff["years_experience"]

print("identical:", np.allclose(slow, fast))

_ = timed("apply with axis=1", lambda: staff.apply(lambda r: r["salary"] / r["years_experience"], axis=1))
_ = timed("column division", lambda: staff["salary"] / staff["years_experience"])

identical: True
apply with axis=1                  0.0005 s
column division                    0.0001 s


Even on 60 rows there is a visible gap, and it widens with size. The rule: if your `axis=1` lambda
only does arithmetic on columns, write the column arithmetic instead. Keep `axis=1` for genuinely
row-shaped logic like string formatting, or conditional logic that depends on several columns in a
way that would be awkward to vectorise.

### `DataFrame.map` — every cell

In [132]:
numbers = pd.DataFrame({"a": [1.234, 5.678], "b": [9.1011, 2.3456]})
print(numbers.map(lambda x: round(x, 1)))

     a    b
0  1.2  9.1
1  5.7  2.3


`DataFrame.map` applies a function to every cell independently. It is rarely the right tool —
`numbers.round(1)` does the same thing here, vectorised — but it exists for the case where you
genuinely need per-cell logic across a uniform table.

If you have seen `applymap` in older code, that was this method's previous name. It was deprecated
in Pandas 2.1 and removed in Pandas 3. Which of those you are running decides what the next cell
prints:

In [133]:
if hasattr(pd.DataFrame, "applymap"):
    print(f"pandas {pd.__version__}: applymap still exists, but it is deprecated — use .map")
else:
    print(f"pandas {pd.__version__}: applymap has been removed")
    try:
        numbers.applymap(lambda x: x)
    except AttributeError as e:
        print("  AttributeError:", e)

pandas 3.0.5: applymap has been removed
  AttributeError: 'DataFrame' object has no attribute 'applymap'


`hasattr(pd.DataFrame, "applymap")` asks whether the method exists before calling it, which is how
you write code that has to cope with more than one version of a library. On Pandas 2 the method is
still there and emits a `FutureWarning`; on Pandas 3 it is gone.

Either way, write `.map`. The rename is settled and the old name has no future.

## Exercises — GroupBy and aggregation

**Level 1.** Using `staff`, compute the average salary and the headcount per city. Sort the result
by average salary, highest first.

**Level 2.** Build one summary table per department containing: headcount, median salary, the
name of the highest-paid employee, and the share of employees who work remotely. (The name will
need a custom aggregation — think about what function of a group returns a name.)

**Level 3.** Add a column showing each employee's salary as a percentage of the *maximum* salary in
their own department, then list everyone earning less than 70% of their department's top salary.
Do it twice: once with `transform`, once with a `groupby` + `merge`. Compare the two for
readability and explain which you would keep.

## Text columns and the `.str` accessor

Text handling is a large part of real data work, because text is where inconsistency lives:
trailing spaces, mixed capitalisation, two spellings of the same city, a phone number with
brackets in it.

Python's string methods work on one string at a time. Pandas exposes the same methods for a whole
column through `.str`.

In [134]:
raw_names = pd.Series(["  ananya rao ", "RAHUL MENON", "sneha Kulkarni  ",
                       "imran SHEIKH", " priya nair"])

print("as given:")
print(raw_names.tolist())
print("\ncleaned:")
print(raw_names.str.strip().str.title().tolist())

as given:
['  ananya rao ', 'RAHUL MENON', 'sneha Kulkarni  ', 'imran SHEIKH', ' priya nair']

cleaned:
['Ananya Rao', 'Rahul Menon', 'Sneha Kulkarni', 'Imran Sheikh', 'Priya Nair']


`.str.strip()` removes leading and trailing whitespace; `.str.title()` capitalises each word. The
two chain together because each returns a Series, so the next `.str` call applies to that result.

The beginner version makes it obvious what is being replaced:

In [135]:
cleaned = []
for name in raw_names:
    cleaned.append(name.strip().title())

print(cleaned)

['Ananya Rao', 'Rahul Menon', 'Sneha Kulkarni', 'Imran Sheikh', 'Priya Nair']


Same methods, same order — the only difference is that `.str` applies them to every element for
you and gives back a Series instead of a list. That is the whole idea of the accessor: `.str.x()`
means "call `x()` on each string in this column".

### The methods you will actually use

In [136]:
cities = pd.Series(["Bengaluru", "new delhi", "MUMBAI", "Pune ", "hyderabad"])

print("lower     :", cities.str.lower().tolist())
print("upper     :", cities.str.upper().tolist())
print("title     :", cities.str.strip().str.title().tolist())
print("length    :", cities.str.len().tolist())
print("contains  :", cities.str.lower().str.contains("delhi").tolist())
print("startswith:", cities.str.lower().str.startswith("m").tolist())
print("replace   :", cities.str.replace(" ", "_").tolist())
print("slice     :", cities.str[:3].tolist())

lower     : ['bengaluru', 'new delhi', 'mumbai', 'pune ', 'hyderabad']
upper     : ['BENGALURU', 'NEW DELHI', 'MUMBAI', 'PUNE ', 'HYDERABAD']
title     : ['Bengaluru', 'New Delhi', 'Mumbai', 'Pune', 'Hyderabad']
length    : [9, 9, 6, 5, 9]
contains  : [False, True, False, False, False]
startswith: [False, False, True, False, False]
replace   : ['Bengaluru', 'new_delhi', 'MUMBAI', 'Pune_', 'hyderabad']
slice     : ['Ben', 'new', 'MUM', 'Pun', 'hyd']


`cities.str[:3]` is worth noticing: the accessor supports slicing, so you can take the first three
characters of every string without writing a loop or a lambda.

### Splitting and extracting

In [137]:
full_names = pd.Series(["Ananya Rao", "Rahul Menon", "Sneha Kulkarni"])

print("split into a list per row:")
print(full_names.str.split(" ").tolist())

print("\nsplit into columns:")
print(full_names.str.split(" ", expand=True))

split into a list per row:
[['Ananya', 'Rao'], ['Rahul', 'Menon'], ['Sneha', 'Kulkarni']]

split into columns:
        0         1
0  Ananya       Rao
1   Rahul     Menon
2   Sneha  Kulkarni


Without `expand=True` you get a column of lists, which is rarely what you want. With it, you get a
DataFrame — one column per piece — which you can then assign into your table:

In [138]:
parts = full_names.str.split(" ", expand=True)
parts.columns = ["first_name", "last_name"]
print(parts)

  first_name last_name
0     Ananya       Rao
1      Rahul     Menon
2      Sneha  Kulkarni


The shortcuts `.str.get(0)` and `.str[0]` pull one piece out of the split without expanding:

In [139]:
print("first names only:", full_names.str.split(" ").str[0].tolist())

first names only: ['Ananya', 'Rahul', 'Sneha']


For anything more structured than "split on a character", use `.str.extract` with a regular
expression.

In [140]:
codes = pd.Series(["EMP-2024-0153", "EMP-2023-0087", "EMP-2024-1120"])

extracted = codes.str.extract(r"EMP-(\d{4})-(\d{4})")
extracted.columns = ["year", "serial"]
print(extracted)
print("\ndtypes:", extracted.dtypes.tolist())

   year serial
0  2024   0153
1  2023   0087
2  2024   1120

dtypes: [<StringDtype(storage='python', na_value=nan)>, <StringDtype(storage='python', na_value=nan)>]


The pattern `EMP-(\d{4})-(\d{4})` means: the literal text `EMP-`, then four digits captured as a
group, then `-`, then four more digits captured as a second group. Each **capture group** — the
parts in parentheses — becomes a column.

Note that the extracted columns are text, not numbers. `\d` matched digit characters, and Pandas
does not guess that you want them converted. Convert explicitly:

In [141]:
extracted["year"] = extracted["year"].astype(int)
print(extracted.dtypes.tolist())

[dtype('int64'), <StringDtype(storage='python', na_value=nan)>]


Regular expressions deserve their own tutorial, but three patterns will carry you a long way:

| Pattern | Matches |
| --- | --- |
| `\d+` | one or more digits |
| `[A-Za-z]+` | one or more letters |
| `(...)` | a capture group — becomes a column in `extract` |

### Applying this to our dataset

In [142]:
staff_text = staff.assign(
    first_name=lambda d: d["name"].str.split(" ").str[0],
    last_name=lambda d: d["name"].str.split(" ").str[-1],
    initials=lambda d: (d["name"].str.split(" ").str[0].str[0]
                        + d["name"].str.split(" ").str[-1].str[0]),
    city_code=lambda d: d["city"].str.upper().str[:3],
)

print(staff_text[["name", "first_name", "last_name", "initials", "city", "city_code"]].head())

            name first_name last_name initials       city city_code
0  Shalini Reddy    Shalini     Reddy       SR       Pune       PUN
1    Swati Desai      Swati     Desai       SD     Mumbai       MUM
2    Aisha Reddy      Aisha     Reddy       AR  Bengaluru       BEN
3     Kavita Rao     Kavita       Rao       KR       Pune       PUN
4     Vikram Rao     Vikram       Rao       VR  Bengaluru       BEN


`.str[-1]` takes the last piece of the split, which handles a middle name gracefully. `.str[0]`
applied to the *result* of a split takes the first name, and `.str[0]` on that takes its first
character. Chaining accessors like this is compact; if you find yourself doing more than two or
three steps, assign an intermediate column and make it readable.

### Missing values in text columns

In [143]:
with_gaps = pd.Series(["Mumbai", None, "Pune", np.nan])

print("dtype:", with_gaps.dtype)
print("upper:", with_gaps.str.upper().tolist())
print("contains 'u':", with_gaps.str.contains("u").tolist())
print("contains with na=False:", with_gaps.str.contains("u", na=False).tolist())

dtype: str
upper: ['MUMBAI', nan, 'PUNE', nan]
contains 'u': [True, False, True, False]
contains with na=False: [True, False, True, False]


String methods propagate missing values rather than crashing — `None` stays missing after
`.str.upper()`. For methods that return booleans, be explicit about what a missing value should
count as with `na=False`, especially when the result goes straight into a filter. A mask containing
missing values behaves unpredictably as a row selector, so pin it down.

## Dates and times

Dates arrive as text and are useless until converted. This section is short but it is the section
people most often skip and then regret.

In [144]:
date_strings = pd.Series(["2024-01-15", "2024-03-22", "2024-07-08"])

print("as text:", date_strings.dtype)
converted = pd.to_datetime(date_strings)
print("converted:", converted.dtype)
print(converted)

as text: str
converted: datetime64[us]
0   2024-01-15
1   2024-03-22
2   2024-07-08
dtype: datetime64[us]


`pd.to_datetime` is the conversion function. Once a column is `datetime64`, you get date-aware
comparisons, sorting, arithmetic and the whole `.dt` accessor.

### Formats, and the one argument worth knowing

In [145]:
print(pd.to_datetime(pd.Series(["15/01/2024", "22/03/2024"]), format="%d/%m/%Y").tolist())
print(pd.to_datetime(pd.Series(["2024-01-15", "15/01/2024"]), format="mixed", dayfirst=True).tolist())

[Timestamp('2024-01-15 00:00:00'), Timestamp('2024-03-22 00:00:00')]
[Timestamp('2024-01-15 00:00:00'), Timestamp('2024-01-15 00:00:00')]


`format="%d/%m/%Y"` states the layout explicitly: day, month, four-digit year, separated by
slashes. Give the format whenever you know it — it is much faster on large columns, and it removes
the ambiguity that causes the single nastiest date bug there is.

That bug: `03/04/2024` is the 3rd of April in most of the world and the 4th of March in the United
States. Pandas has to pick one, and its default is month-first:

In [146]:
ambiguous = pd.Series(["03/04/2024"])

print("default        :", pd.to_datetime(ambiguous).dt.strftime("%d %B %Y").tolist())
print("dayfirst=True  :", pd.to_datetime(ambiguous, dayfirst=True).dt.strftime("%d %B %Y").tolist())
print("explicit format:", pd.to_datetime(ambiguous, format="%d/%m/%Y").dt.strftime("%d %B %Y").tolist())

default        : ['04 March 2024']
dayfirst=True  : ['03 April 2024']
explicit format: ['03 April 2024']


Two different months from the same text, and **no warning either way**. If your dates came from an
Indian, British or European source and you let Pandas guess, every date with a day of 12 or below
is silently wrong, and every date above 12 is right — which is the worst kind of bug, because
spot-checking a few rows will not find it.

You do get an error when the column contains a value that cannot be read month-first at all:

In [147]:
try:
    pd.to_datetime(pd.Series(["03/04/2024", "13/04/2024"]))
except ValueError as e:
    print("ValueError:", str(e)[:90], "...")

ValueError: time data "13/04/2024" doesn't match format "%m/%d/%Y". You might want to try:
    - passi ...


`13/04/2024` has no 13th month, so the guess fails and Pandas stops. Be grateful for that error —
it is the case where the mistake is detectable. Pass `format=` or `dayfirst=True` and the whole
question goes away:

In [148]:
print(pd.to_datetime(pd.Series(["03/04/2024", "13/04/2024"]), format="%d/%m/%Y")
      .dt.strftime("%d %B %Y").tolist())

['03 April 2024', '13 April 2024']


When some values are not dates at all, `errors="coerce"` turns the failures into `NaT`
("not a time", the datetime equivalent of `NaN`) instead of raising:

In [149]:
mixed_quality = pd.Series(["2024-01-15", "not a date", "2024-03-22", ""])

print(pd.to_datetime(mixed_quality, errors="coerce"))
print("\nfailed to parse:", pd.to_datetime(mixed_quality, errors="coerce").isna().sum())

0   2024-01-15
1          NaT
2   2024-03-22
3          NaT
dtype: datetime64[us]

failed to parse: 2


This is the right tool for messy input, with one condition: check how many failed. Silently
coercing a third of your dates to `NaT` and carrying on is worse than crashing.

### The `.dt` accessor

In [150]:
print(staff[["name", "joined"]].head(3))

parts = staff.assign(
    year=lambda d: d["joined"].dt.year,
    month=lambda d: d["joined"].dt.month,
    month_name=lambda d: d["joined"].dt.month_name(),
    day_name=lambda d: d["joined"].dt.day_name(),
    quarter=lambda d: d["joined"].dt.quarter,
    is_month_end=lambda d: d["joined"].dt.is_month_end,
)

print(parts[["joined", "year", "month", "month_name", "day_name", "quarter"]].head())

            name     joined
0  Shalini Reddy 2016-05-14
1    Swati Desai 2013-10-11
2    Aisha Reddy 2016-04-30
      joined  year  month month_name  day_name  quarter
0 2016-05-14  2016      5        May  Saturday        2
1 2013-10-11  2013     10    October    Friday        4
2 2016-04-30  2016      4      April  Saturday        2
3 2013-08-02  2013      8     August    Friday        3
4 2014-06-06  2014      6       June    Friday        2


`.dt` is to datetimes what `.str` is to text: an accessor exposing per-element properties across
the whole column. The useful ones are `year`, `month`, `day`, `hour`, `dayofweek`, `day_name()`,
`month_name()`, `quarter`, `days_in_month`, `is_month_end`.

Note that `year` is a property (no brackets) while `day_name()` is a method (brackets). That is
inconsistent and you will simply have to remember it; the error message when you get it wrong is
clear enough.

### Filtering and sorting by date

In [151]:
recent = staff[staff["joined"] >= "2018-01-01"]
print("joined 2018 or later:", len(recent))

print("\nlongest-serving employees:")
print(staff.nsmallest(3, "joined")[["name", "department", "joined"]])

joined 2018 or later: 19

longest-serving employees:
           name   department     joined
56  Priya Joshi      Finance 2008-04-29
36  Rohan Menon  Engineering 2008-05-12
33   Rohan Nair  Engineering 2008-08-08


Comparing a datetime column against a **string** works — Pandas parses the string for you. That is
convenient and it is also the one place where the ISO format `"YYYY-MM-DD"` is not just good
practice but genuinely unambiguous.

In [152]:
print("joined in 2015:", len(staff[staff["joined"].dt.year == 2015]))
print("joined in the first quarter of any year:", len(staff[staff["joined"].dt.quarter == 1]))
print("\nbetween two dates:")
window = staff[staff["joined"].between("2014-01-01", "2015-12-31")]
print(len(window), "employees")

joined in 2015: 3
joined in the first quarter of any year: 8

between two dates:
10 employees


### Date arithmetic

In [153]:
today = pd.Timestamp("2024-01-01")

tenure = staff.assign(
    days_at_company=lambda d: (today - d["joined"]).dt.days,
    years_at_company=lambda d: ((today - d["joined"]).dt.days / 365.25).round(1),
)

print(tenure[["name", "joined", "days_at_company", "years_at_company"]].head())

            name     joined  days_at_company  years_at_company
0  Shalini Reddy 2016-05-14             2788               7.6
1    Swati Desai 2013-10-11             3734              10.2
2    Aisha Reddy 2016-04-30             2802               7.7
3     Kavita Rao 2013-08-02             3804              10.4
4     Vikram Rao 2014-06-06             3496               9.6


Subtracting two datetimes gives a **Timedelta**, and `.dt.days` extracts the whole number of days
from it. Dividing by 365.25 approximates years — good enough for a tenure report, and worth a
comment in code so nobody wonders about the .25 (leap years).

In [154]:
print("shift a date forward:")
print((staff["joined"].head(3) + pd.Timedelta(days=90)).tolist())

print("\nadd months properly:")
print((staff["joined"].head(3) + pd.DateOffset(months=6)).tolist())

shift a date forward:
[Timestamp('2016-08-12 00:00:00'), Timestamp('2014-01-09 00:00:00'), Timestamp('2016-07-29 00:00:00')]

add months properly:
[Timestamp('2016-11-14 00:00:00'), Timestamp('2014-04-11 00:00:00'), Timestamp('2016-10-30 00:00:00')]


`Timedelta` handles fixed durations — days, hours, minutes. "One month" is not a fixed duration, so
adding 30 days to the 31st of January gives a different answer than a human would; `DateOffset`
understands calendar units and does what you expect.

### Resampling a time series

When the index is a datetime, you can group by time period with `resample`, which is `groupby` for
dates.

In [155]:
rng_sales = np.random.default_rng(15)
dates = pd.date_range("2024-01-01", periods=120, freq="D")

daily_sales = pd.Series(
    (20_000 + rng_sales.normal(0, 4_000, 120) + np.arange(120) * 60).round(0),
    index=dates,
    name="sales",
)

print(daily_sales.head())
print("\nmonthly totals:")
print(daily_sales.resample("ME").sum())
print("\nweekly averages (first four weeks):")
print(daily_sales.resample("W").mean().round(0).head(4))

2024-01-01    14277.0
2024-01-02    16314.0
2024-01-03    21696.0
2024-01-04    18084.0
2024-01-05    22342.0
Freq: D, Name: sales, dtype: float64

monthly totals:
2024-01-31    651215.0
2024-02-29    664569.0
2024-03-31    782043.0
2024-04-30    796853.0
Freq: ME, Name: sales, dtype: float64

weekly averages (first four weeks):
2024-01-07    18690.0
2024-01-14    22384.0
2024-01-21    22648.0
2024-01-28    21114.0
Freq: W-SUN, Name: sales, dtype: float64


`resample("ME")` groups by month end; `"W"` is weekly, `"QE"` quarterly, `"YE"` yearly, `"h"`
hourly. The frequency strings changed in recent Pandas versions — `"M"` became `"ME"` and `"Q"`
became `"QE"` — so older code may emit a deprecation warning here. The new names spell out
whether you mean the start or the end of the period, which is an improvement.

`resample` requires a datetime index, so `set_index` first if your dates are in a column:

In [156]:
sales_table = daily_sales.reset_index()
sales_table.columns = ["date", "sales"]

print(sales_table.set_index("date").resample("ME")["sales"].sum())

date
2024-01-31    651215.0
2024-02-29    664569.0
2024-03-31    782043.0
2024-04-30    796853.0
Freq: ME, Name: sales, dtype: float64


A rolling average is the other common time-series operation — useful for seeing a trend through
noise:

In [157]:
smoothed = pd.DataFrame({
    "daily": daily_sales,
    "rolling_7day": daily_sales.rolling(window=7).mean().round(0),
})

print(smoothed.head(9))

              daily  rolling_7day
2024-01-01  14277.0           NaN
2024-01-02  16314.0           NaN
2024-01-03  21696.0           NaN
2024-01-04  18084.0           NaN
2024-01-05  22342.0           NaN
2024-01-06  23529.0           NaN
2024-01-07  14586.0       18690.0
2024-01-08  24488.0       20148.0
2024-01-09  18097.0       20403.0


The first six rows of the rolling average are `NaN`, because a seven-day average needs seven days.
That is correct rather than a bug; `min_periods=1` relaxes it if you would rather have a partial
average at the start.

## Combining datasets

Two operations with two very different jobs, and mixing them up is a common source of confusion.

- **`concat`** stacks data — more rows (or more columns) of the same kind of thing.
- **`merge`** joins data — matching rows from two tables on a shared key.

### `concat` — stacking

In [158]:
q1 = pd.DataFrame({"month": ["Jan", "Feb", "Mar"], "revenue": [120, 135, 150]})
q2 = pd.DataFrame({"month": ["Apr", "May", "Jun"], "revenue": [160, 155, 170]})

stacked = pd.concat([q1, q2])
print(stacked)

  month  revenue
0   Jan      120
1   Feb      135
2   Mar      150
0   Apr      160
1   May      155
2   Jun      170


Notice the index: `0, 1, 2, 0, 1, 2`. `concat` preserves the original labels, so now two rows have
the label 0. That is usually not what you want:

In [159]:
print(pd.concat([q1, q2], ignore_index=True))

  month  revenue
0   Jan      120
1   Feb      135
2   Mar      150
3   Apr      160
4   May      155
5   Jun      170


`ignore_index=True` renumbers from scratch. Use it whenever the original row labels carry no
meaning, which for stacked data is nearly always.

In [160]:
print("keeping track of where each row came from:")
print(pd.concat([q1, q2], keys=["Q1", "Q2"]))

keeping track of where each row came from:
     month  revenue
Q1 0   Jan      120
   1   Feb      135
   2   Mar      150
Q2 0   Apr      160
   1   May      155
   2   Jun      170


`keys=` adds an outer index level recording the source. Useful when you stack several files and
later need to know which file a row came from.

### `concat` with mismatched columns

In [161]:
q3 = pd.DataFrame({"month": ["Jul"], "revenue": [180], "region": ["North"]})

print(pd.concat([q1, q3], ignore_index=True))

  month  revenue region
0   Jan      120    NaN
1   Feb      135    NaN
2   Mar      150    NaN
3   Jul      180  North


Columns present in only one frame are filled with `NaN` for the other rows. Pandas takes the union
of columns by default, which is forgiving — and means a typo in a column name gives you two
half-empty columns instead of an error. Check `df.columns` after concatenating unfamiliar data.

`join="inner"` keeps only the columns present in all the frames:

In [162]:
print(pd.concat([q1, q3], ignore_index=True, join="inner"))

  month  revenue
0   Jan      120
1   Feb      135
2   Mar      150
3   Jul      180


### `merge` — joining on a key

Time for a second table. Each department has a head and a budget.

In [163]:
dept_info = pd.DataFrame({
    "department": ["Engineering", "Marketing", "Sales", "Finance", "HR", "Legal"],
    "head": ["Vikram Iyer", "Neha Gupta", "Rajesh Verma", "Anjali Bose", "Tarun Khan", "Divya Mehta"],
    "annual_budget_cr": [48.0, 12.5, 18.0, 9.5, 6.0, 4.0],
})

dept_info

,department,head,annual_budget_cr
0,Engineering,Vikram Iyer,48.0
1,Marketing,Neha Gupta,12.5
2,Sales,Rajesh Verma,18.0
3,Finance,Anjali Bose,9.5
4,HR,Tarun Khan,6.0
5,Legal,Divya Mehta,4.0


Note that `dept_info` has a `Legal` department, which has no employees in `staff`. That asymmetry
is deliberate — it is how you see what the different join types actually do.

In [164]:
joined = staff.merge(dept_info, on="department")

print("staff rows      :", len(staff))
print("merged rows     :", len(joined))
print(joined[["name", "department", "head", "annual_budget_cr"]].head())

staff rows      : 60
merged rows     : 60
            name   department          head  annual_budget_cr
0  Shalini Reddy        Sales  Rajesh Verma              18.0
1    Swati Desai      Finance   Anjali Bose               9.5
2    Aisha Reddy      Finance   Anjali Bose               9.5
3     Kavita Rao  Engineering   Vikram Iyer              48.0
4     Vikram Rao  Engineering   Vikram Iyer              48.0


`staff.merge(dept_info, on="department")` matched each employee to their department's row and
copied `head` and `annual_budget_cr` onto it. The department name appears once per employee, so
those values are repeated — that is normal and expected for a many-to-one join.

### The four join types

```text
LEFT table (staff)              RIGHT table (dept_info)
departments present:            departments present:
  Engineering, Marketing,         Engineering, Marketing, Sales,
  Sales, Finance, HR              Finance, HR, Legal

how="inner"  →  rows whose key is in BOTH          (Legal dropped)
how="left"   →  ALL left rows, plus matches        (Legal dropped)
how="right"  →  ALL right rows, plus matches       (Legal kept, with NaN employee)
how="outer"  →  everything from both sides         (Legal kept, with NaN employee)
```

In [165]:
for how in ["inner", "left", "right", "outer"]:
    result = staff.merge(dept_info, on="department", how=how)
    print(f"how={how:6} → {len(result):3} rows, "
          f"{result['name'].isna().sum()} rows with no employee, "
          f"{result['head'].isna().sum()} rows with no department info")

how=inner  →  60 rows, 0 rows with no employee, 0 rows with no department info
how=left   →  60 rows, 0 rows with no employee, 0 rows with no department info
how=right  →  61 rows, 1 rows with no employee, 0 rows with no department info
how=outer  →  61 rows, 1 rows with no employee, 0 rows with no department info


Read that table carefully, because it is the whole story of joins:

- `inner` (the default) and `left` both give 60 rows here. Every employee's department exists in
  `dept_info`, so nothing is lost either way — and `Legal` is dropped because no employee belongs
  to it.
- `right` gives 61: all 60 employees plus one row for `Legal` with every employee field missing.
- `outer` gives the same 61 here, because the only unmatched key is on the right.

**Which should you use?** In practice, `how="left"` most of the time. It expresses "keep all my
rows, and add information from the other table where available", which is what you almost always
mean when enriching a dataset. `inner` silently drops rows whose key is missing — and silently
dropping data is how analyses go wrong without anyone noticing.

### Always check the row count after a merge

This is the single most valuable merge habit.

In [166]:
before = len(staff)
after = len(staff.merge(dept_info, on="department", how="left"))
print(f"{before} rows before, {after} after — unchanged, as expected for a many-to-one join")

60 rows before, 60 after — unchanged, as expected for a many-to-one join


If a left join changes your row count, the right table has duplicate keys and every match is being
multiplied. Here is that failure, deliberately:

In [167]:
duplicated_info = pd.concat([dept_info, dept_info.head(2)], ignore_index=True)
print("right table now has duplicate departments:")
print(duplicated_info["department"].value_counts().head(3))

bad_merge = staff.merge(duplicated_info, on="department", how="left")
print(f"\n{len(staff)} employees became {len(bad_merge)} rows")
print("Engineering employees before:", (staff["department"] == "Engineering").sum())
print("Engineering employees after :", (bad_merge["department"] == "Engineering").sum())

right table now has duplicate departments:
department
Engineering    2
Marketing      2
Sales          1
Name: count, dtype: int64

60 employees became 95 rows
Engineering employees before: 25
Engineering employees after : 50


Every Engineering employee got matched twice, so they appear twice. Any total computed from
`bad_merge` — headcount, payroll — is now inflated, and nothing warned you.

Two defences. First, check the row count, every time. Second, tell Pandas what you expect:

In [168]:
try:
    staff.merge(duplicated_info, on="department", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e)[:120])

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  department
Engineering
  Mar


`validate="many_to_one"` asserts that the right table's keys are unique. If they are not, you get
an error instead of quietly duplicated data. The options are `"one_to_one"`, `"one_to_many"`,
`"many_to_one"` and `"many_to_many"`.

Adding `validate=` to merges in code you care about is one of the highest-value habits in this
notebook.

### Different key names, and overlapping columns

In [169]:
targets = pd.DataFrame({
    "dept_name": ["Engineering", "Marketing", "Sales"],
    "target_cr": [50.0, 15.0, 22.0],
})

print(staff.merge(targets, left_on="department", right_on="dept_name", how="left")
      [["name", "department", "dept_name", "target_cr"]].head(3))

            name department dept_name  target_cr
0  Shalini Reddy      Sales     Sales       22.0
1    Swati Desai    Finance       NaN        NaN
2    Aisha Reddy    Finance       NaN        NaN


`left_on` / `right_on` handle columns with different names. You end up with both key columns, one
of which is redundant — drop it afterwards, or rename before merging.

When both tables have a column with the same name that is *not* the key, Pandas appends suffixes:

In [170]:
left = pd.DataFrame({"id": [1, 2], "score": [10, 20]})
right = pd.DataFrame({"id": [1, 2], "score": [99, 98]})

print(left.merge(right, on="id"))
print()
print(left.merge(right, on="id", suffixes=("_before", "_after")))

   id  score_x  score_y
0   1       10       99
1   2       20       98

   id  score_before  score_after
0   1            10           99
1   2            20           98


The default `_x` and `_y` are unhelpful in any output a human will read. Set `suffixes` explicitly
whenever the collision is expected.

### `indicator` — seeing where each row came from

In [171]:
check = staff.merge(dept_info, on="department", how="outer", indicator=True)
print(check["_merge"].value_counts())
print("\nrows that exist only in the department table:")
print(check[check["_merge"] == "right_only"][["department", "head"]])

_merge
both          60
right_only     1
left_only      0
Name: count, dtype: int64

rows that exist only in the department table:
   department         head
40      Legal  Divya Mehta


`indicator=True` adds a `_merge` column labelling each row `left_only`, `right_only` or `both`.
This is the fastest way to diagnose "why did my join lose rows" — and it found our `Legal`
department, which has a head and a budget but nobody working in it.

### Joining on the index

In [172]:
dept_indexed = dept_info.set_index("department")
print(staff.set_index("department").join(dept_indexed, how="left")
      [["name", "head"]].head(3))

                     name          head
department                             
Sales       Shalini Reddy  Rajesh Verma
Finance       Swati Desai   Anjali Bose
Finance       Aisha Reddy   Anjali Bose


`join` is a convenience method that merges on the index by default. `merge` can do the same thing
with `left_index=True` / `right_index=True`. Use whichever reads better; `merge` is the more
general tool and the one worth knowing properly.

## Reshaping

The same data can be laid out **long** (one row per observation) or **wide** (one row per subject,
one column per variable). Different tools want different shapes, so converting between them is a
routine task.

In [173]:
long_form = pd.DataFrame({
    "student": ["Aarti", "Aarti", "Aarti", "Bilal", "Bilal", "Bilal",
                "Chirag", "Chirag", "Chirag"],
    "subject": ["Maths", "Physics", "Chemistry"] * 3,
    "marks": [78, 82, 91, 65, 70, 58, 88, 94, 79],
})

long_form

,student,subject,marks
0,Aarti,Maths,78
1,Aarti,Physics,82
2,Aarti,Chemistry,91
3,Bilal,Maths,65
4,Bilal,Physics,70
5,Bilal,Chemistry,58
6,Chirag,Maths,88
7,Chirag,Physics,94
8,Chirag,Chemistry,79


### `pivot` — long to wide

In [174]:
wide_form = long_form.pivot(index="student", columns="subject", values="marks")
print(wide_form)

subject  Chemistry  Maths  Physics
student                           
Aarti           91     78       82
Bilal           58     65       70
Chirag          79     88       94


```python
long_form.pivot(index="student", columns="subject", values="marks")
                 └── rows ──┘   └─── columns ───┘  └── cells ──┘
```

- `index` — the column whose values become the row labels.
- `columns` — the column whose values become the column headers.
- `values` — the column that fills the cells.

`pivot` requires that each (index, columns) combination appears **exactly once**. If a student has
two Maths marks, it cannot decide what to put in the cell and raises an error:

In [175]:
with_duplicate = pd.concat([long_form, long_form.head(1)], ignore_index=True)
try:
    with_duplicate.pivot(index="student", columns="subject", values="marks")
except ValueError as e:
    print("ValueError:", e)

ValueError: Index contains duplicate entries, cannot reshape


### `pivot_table` — pivot with aggregation

In [176]:
print(with_duplicate.pivot_table(index="student", columns="subject",
                                 values="marks", aggfunc="mean"))

subject  Chemistry  Maths  Physics
student                           
Aarti         91.0   78.0     82.0
Bilal         58.0   65.0     70.0
Chirag        79.0   88.0     94.0


`pivot_table` handles duplicates by aggregating them, so the duplicate Maths mark for Aarti became
the mean of the two. It defaults to `aggfunc="mean"`, which is worth knowing — if you pivot a table
and the numbers look slightly off, you may be averaging where you meant to sum.

`pivot_table` is also the tool for cross-tabulated summaries of a bigger dataset:

In [177]:
print(staff.pivot_table(index="department", columns="remote",
                        values="salary", aggfunc="mean").round(0))

print("\nwith counts as well, and totals:")
print(staff.pivot_table(index="department", columns="remote", values="salary",
                        aggfunc=["mean", "count"], margins=True).round(0))

remote           False      True 
department                       
Engineering  1239091.0  1323071.0
Finance      1183375.0  1285667.0
HR                 NaN   918000.0
Marketing    1142600.0   949800.0
Sales        1000889.0   782000.0

with counts as well, and totals:


                  mean                       count          
remote           False       True        All False  True All
department                                                  
Engineering  1239091.0  1323071.0  1286120.0  11.0  14.0  25
Finance      1183375.0  1285667.0  1211273.0   8.0   3.0  11
HR                 NaN   918000.0   918000.0   NaN   4.0   4
Marketing    1142600.0   949800.0  1046200.0   5.0   5.0  10
Sales        1000889.0   782000.0   979000.0   9.0   1.0  10
All          1146000.0  1169741.0  1156683.0  33.0  27.0  60


`margins=True` adds the `All` row and column — the totals across each dimension. It saves computing
them separately and getting them inconsistent.

### `melt` — wide to long

In [178]:
back_to_long = wide_form.reset_index().melt(
    id_vars="student",
    var_name="subject",
    value_name="marks",
)

print(back_to_long.head(6))
print("\nrows:", len(back_to_long))

  student    subject  marks
0   Aarti  Chemistry     91
1   Bilal  Chemistry     58
2  Chirag  Chemistry     79
3   Aarti      Maths     78
4   Bilal      Maths     65
5  Chirag      Maths     88

rows: 9


`melt` is `pivot` in reverse. Its arguments:

- `id_vars` — the columns to keep as they are (the identifiers).
- `value_vars` — the columns to collapse. Omitted here, so everything else is collapsed.
- `var_name` — what to call the new column holding the old column names.
- `value_name` — what to call the new column holding the values.

This shape is what plotting libraries and statistical models usually want: one row per
observation, with the variable name as data rather than as a column header.

In [179]:
print("a realistic melt — quarterly figures spread across columns:")
quarterly = pd.DataFrame({
    "product": ["Keyboard", "Mouse", "Monitor"],
    "Q1": [120, 90, 200],
    "Q2": [135, 85, 210],
    "Q3": [150, 110, 190],
    "Q4": [160, 100, 230],
})
print(quarterly)

tidy_quarterly = quarterly.melt(id_vars="product", var_name="quarter", value_name="units")
print("\nmelted:")
print(tidy_quarterly.head(6))

print("\nnow grouping is easy:")
print(tidy_quarterly.groupby("quarter")["units"].sum())

a realistic melt — quarterly figures spread across columns:
    product   Q1   Q2   Q3   Q4
0  Keyboard  120  135  150  160
1     Mouse   90   85  110  100
2   Monitor  200  210  190  230

melted:
    product quarter  units
0  Keyboard      Q1    120
1     Mouse      Q1     90
2   Monitor      Q1    200
3  Keyboard      Q2    135
4     Mouse      Q2     85
5   Monitor      Q2    210

now grouping is easy:
quarter
Q1    410
Q2    430
Q3    450
Q4    490
Name: units, dtype: int64


That last step is the point. With quarters as columns you would have to name them all to total
them; with quarters as data, `groupby` handles it and the code does not change when Q5 arrives.

### `stack` and `unstack`

These move data between the index and the columns, which is the same operation in a different
guise.

In [180]:
print("wide form:")
print(wide_form)

print("\nstacked (columns pushed into the index):")
print(wide_form.stack())

print("\nunstacked again:")
print(wide_form.stack().unstack())

wide form:
subject  Chemistry  Maths  Physics
student                           
Aarti           91     78       82
Bilal           58     65       70
Chirag          79     88       94

stacked (columns pushed into the index):
student  subject  
Aarti    Chemistry    91
         Maths        78
         Physics      82
Bilal    Chemistry    58
         Maths        65
         Physics      70
Chirag   Chemistry    79
         Maths        88
         Physics      94
dtype: int64

unstacked again:
subject  Chemistry  Maths  Physics
student                           
Aarti           91     78       82
Bilal           58     65       70
Chirag          79     88       94


`stack` turns columns into an extra index level, producing a Series. `unstack` does the reverse.
You met `unstack` already in the GroupBy section — taking a grouped result with a MultiIndex and
spreading the inner level across columns is exactly this operation, and it is the most common
reason to reach for it.

In [181]:
print(staff.groupby(["department", "remote"])["salary"].mean().unstack().round(0))

remote           False      True 
department                       
Engineering  1239091.0  1323071.0
Finance      1183375.0  1285667.0
HR                 NaN   918000.0
Marketing    1142600.0   949800.0
Sales        1000889.0   782000.0


## Exercises — text, dates, merging, reshaping

**Level 1.** From `staff`, create a column holding each employee's surname in uppercase, and count
how many employees have a surname starting with a letter before "M" in the alphabet.

**Level 2.** Merge `staff` with `dept_info` using a left join. Verify the row count is unchanged,
then compute each department's total payroll as a percentage of its `annual_budget_cr`
(remember the budget is in crores — 1 crore is 10,000,000).

**Level 3.** Build a table with departments as rows, joining years (from `joined`) as columns, and
headcount in the cells, with zeros rather than `NaN` where nobody joined. Then melt it back to long
form and confirm the total headcount still adds up to 60. Which of the two layouts would you put
in a report, and which would you feed into further calculations?

## Reading and writing files

Everything so far has used data created inside the notebook. Real work starts with a file.

To keep this notebook self-contained and reproducible, we will write our own files first and then
read them back — which also happens to be the best way to understand what the reader options do.

In [182]:
from pathlib import Path

output_dir = Path("pandas_io_demo")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "staff.csv"
staff.to_csv(csv_path, index=False)

print("written to:", csv_path)
print("file size:", csv_path.stat().st_size, "bytes")
print("\nfirst three lines as text:")
print("\n".join(csv_path.read_text(encoding="utf-8").splitlines()[:3]))

written to:

 pandas_io_demo\staff.csv


file size: 3883 bytes

first three lines as text:
emp_id,name,department,years_experience,salary,city,remote,joined
E1000,Shalini Reddy,Sales,7,1018000,Pune,False,2016-05-14
E1001,Swati Desai,Finance,10,1115000,Mumbai,False,2013-10-11


`index=False` is almost always what you want when writing a CSV. Without it, the index is written
as an unnamed first column, and when you read the file back you get a stray column called
`Unnamed: 0`. That column then follows you around for the rest of the analysis.

`Path` from the standard library is worth using over string paths: `output_dir / "staff.csv"`
builds the path with the right separator for the operating system, so the same code works on
Windows and Linux.

In [183]:
loaded = pd.read_csv(csv_path)

print(loaded.dtypes)
print("\nrows:", len(loaded))

emp_id                str
name                  str
department            str
years_experience    int64
salary              int64
city                  str
remote               bool
joined                str
dtype: object

rows: 60


### What survives a round trip, and what does not

In [184]:
print("original dtypes vs loaded dtypes:")
print(pd.DataFrame({"original": staff.dtypes, "after_csv": loaded.dtypes}))

original dtypes vs loaded dtypes:
                        original after_csv
emp_id                       str       str
name                         str       str
department                   str       str
years_experience           int64     int64
salary                     int64     int64
city                         str       str
remote                      bool      bool
joined            datetime64[us]       str


The `joined` column went out as a datetime and came back as text. A CSV is just characters — it has
no way to record that a column was a date, so every reader has to be told.

In [185]:
loaded = pd.read_csv(csv_path, parse_dates=["joined"])
print(loaded.dtypes["joined"])

datetime64[us]


`parse_dates=["joined"]` fixes it. This is the single most common surprise when loading a CSV, and
the reason `df.dtypes` should be the first thing you look at after reading a file.

### `read_csv` options worth knowing

```python
pd.read_csv(
    path,
    usecols=["name", "salary"],   # read only these columns — saves time and memory
    dtype={"emp_id": "str"},      # force a column's type; stops IDs losing leading zeros
    parse_dates=["joined"],       # convert these to datetime
    na_values=["N/A", "-", "?"],  # treat these strings as missing
    thousands=",",                # handle "1,250,000"
    encoding="utf-8",             # see the note below
    nrows=1000,                   # read just the first rows, to look before committing
    skiprows=2,                   # skip junk header lines
)
```

`usecols` and `nrows` are the two to remember for large files: look at a thousand rows and two
columns before loading three gigabytes.

In [186]:
peek = pd.read_csv(csv_path, usecols=["name", "department", "salary"], nrows=5)
print(peek)

            name   department   salary
0  Shalini Reddy        Sales  1018000
1    Swati Desai      Finance  1115000
2    Aisha Reddy      Finance  1190000
3     Kavita Rao  Engineering  1458000
4     Vikram Rao  Engineering  1243000


### Missing-value markers

A CSV written by a person rarely uses an empty cell for missing. It uses `N/A`, or `-`, or `NULL`,
or `none`, or a space. Pandas recognises a standard list (including the empty string, `NA`, `NaN`,
`null`) but not the creative ones.

In [187]:
messy_csv = output_dir / "messy.csv"
messy_csv.write_text(
    "name,city,salary\n"
    "Ananya,Mumbai,920000\n"
    "Rahul,N/A,1450000\n"
    "Sneha,Pune,not available\n"
    "Imran,-,540000\n",
    encoding="utf-8",
)

naive = pd.read_csv(messy_csv)
print("read naively:")
print(naive)
print("\ndtypes:", naive.dtypes.tolist())
print("missing values found:", naive.isna().sum().sum())

read naively:
     name    city         salary
0  Ananya  Mumbai         920000
1   Rahul     NaN        1450000
2   Sneha    Pune  not available
3   Imran       -         540000

dtypes: [<StringDtype(storage='python', na_value=nan)>, <StringDtype(storage='python', na_value=nan)>, <StringDtype(storage='python', na_value=nan)>]
missing values found: 1


Salary came through as text, because `"not available"` is not a number, and one bad value forces
the whole column to text. And Pandas found no missing values at all, because `N/A` and `-` are
just strings as far as it knows.

In [188]:
better = pd.read_csv(messy_csv, na_values=["N/A", "-", "not available"])
print(better)
print("\ndtypes:", better.dtypes.tolist())
print("missing values found:", better.isna().sum().sum())

     name    city     salary
0  Ananya  Mumbai   920000.0
1   Rahul     NaN  1450000.0
2   Sneha    Pune        NaN
3   Imran     NaN   540000.0

dtypes: [<StringDtype(storage='python', na_value=nan)>, <StringDtype(storage='python', na_value=nan)>, dtype('float64')]
missing values found: 3


With the markers declared, salary is numeric and the gaps are real gaps. This is a five-second fix
that prevents an hour of confusion — but only if you look at the dtypes and notice.

### Encoding

In [189]:
accented = pd.DataFrame({"city": ["Bengaluru", "Kraków", "São Paulo"], "code": [1, 2, 3]})

utf8_path = output_dir / "utf8.csv"
accented.to_csv(utf8_path, index=False, encoding="utf-8")

print(pd.read_csv(utf8_path, encoding="utf-8"))

        city  code
0  Bengaluru     1
1     Kraków     2
2  São Paulo     3


Text files carry no record of their own encoding, so a file written as UTF-8 and read as something
else produces mangled characters or a `UnicodeDecodeError`.

In [190]:
try:
    pd.read_csv(utf8_path, encoding="ascii")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", str(e)[:100])

UnicodeDecodeError: 'ascii' codec can't decode byte 0xc3 in position 28: ordinal not in range(128)


The practical advice:

- Write UTF-8 always. `encoding="utf-8"` on every `to_csv`.
- When reading a file you did not create and it fails, try `encoding="utf-8"`, then
  `encoding="latin-1"` (which never fails, though it may produce odd characters), then
  `encoding="cp1252"` for files that came out of Excel on Windows.
- `encoding_errors="replace"` gets a stubborn file open with unreadable characters replaced,
  so you can at least see what you are dealing with.

### Excel

In [191]:
excel_path = output_dir / "staff.xlsx"

try:
    staff.head(10).to_excel(excel_path, index=False, sheet_name="Employees")
    from_excel = pd.read_excel(excel_path, sheet_name="Employees")
    print("round trip worked. dtypes:")
    print(from_excel.dtypes)
except ImportError as e:
    print("Excel support needs an extra package:", e)
    print("Install it with: %pip install openpyxl")

round trip worked. dtypes:
emp_id                         str
name                           str
department                     str
years_experience             int64
salary                       int64
city                           str
remote                        bool
joined              datetime64[us]
dtype: object


Excel support requires `openpyxl` (for `.xlsx`), which is not installed with Pandas. The cell above
is wrapped in `try`/`except` so the notebook still runs without it.

Excel *does* preserve types, which is why `joined` came back as a datetime without `parse_dates`.
It is also slower than CSV, limited in row count, and the file format invites manual edits that
break your pipeline. Read Excel when that is what you are given; prefer CSV or Parquet for data
you control.

Writing several sheets at once:

In [192]:
try:
    with pd.ExcelWriter(output_dir / "report.xlsx") as writer:
        staff.groupby("department")["salary"].mean().round(0).to_frame("avg_salary") \
            .to_excel(writer, sheet_name="By Department")
        staff.groupby("city")["salary"].mean().round(0).to_frame("avg_salary") \
            .to_excel(writer, sheet_name="By City")
    print("two-sheet report written")
except ImportError:
    print("openpyxl not available — skipping")

two-sheet report written


`with pd.ExcelWriter(...) as writer:` is a **context manager**. The `with` block guarantees the
file is properly closed and finalised when the block ends, even if an error happens inside it.
Without it you would have to remember to call `writer.close()`, and a forgotten close leaves a
corrupt file.

### JSON

In [193]:
json_path = output_dir / "staff.json"
staff.head(3).to_json(json_path, orient="records", indent=2, date_format="iso")

print(json_path.read_text(encoding="utf-8")[:400])

[
  {
    "emp_id":"E1000",
    "name":"Shalini Reddy",
    "department":"Sales",
    "years_experience":7,
    "salary":1018000,
    "city":"Pune",
    "remote":false,
    "joined":"2016-05-14T00:00:00.000"
  },
  {
    "emp_id":"E1001",
    "name":"Swati Desai",
    "department":"Finance",
    "years_experience":10,
    "salary":1115000,
    "city":"Mumbai",
    "remote":false,
    "joined":"201


`orient="records"` produces a list of objects — one per row — which is the shape most web APIs use
and the one another program is most likely to understand. The alternatives (`"columns"`,
`"index"`, `"split"`, `"table"`) restructure the same data; `records` is the sensible default for
interchange.

In [194]:
print(pd.read_json(json_path, orient="records"))

  emp_id           name department  years_experience   salary       city  \
0  E1000  Shalini Reddy      Sales                 7  1018000       Pune   
1  E1001    Swati Desai    Finance                10  1115000     Mumbai   
2  E1002    Aisha Reddy    Finance                 7  1190000  Bengaluru   

   remote                   joined  
0   False  2016-05-14T00:00:00.000  
1   False  2013-10-11T00:00:00.000  
2   False  2016-04-30T00:00:00.000  


Nested JSON is the common real-world case, and `pd.json_normalize` flattens it:

In [195]:
nested = [
    {"id": 1, "name": "Ananya", "role": {"title": "Engineer", "level": 3}},
    {"id": 2, "name": "Rahul", "role": {"title": "Manager", "level": 5}},
]

print(pd.json_normalize(nested))

   id    name role.title  role.level
0   1  Ananya   Engineer           3
1   2   Rahul    Manager           5


The nested keys became `role.title` and `role.level`. `sep="_"` changes the separator, which is
worth doing since dots in column names make `df.role.title` impossible.

### Reading a large file in chunks

When a file does not fit in memory, process it in pieces.

In [196]:
chunk_total = 0
chunk_rows = 0

for chunk in pd.read_csv(csv_path, chunksize=20):
    chunk_total += chunk["salary"].sum()
    chunk_rows += len(chunk)

print(f"processed {chunk_rows} rows in chunks")
print(f"total salary: {chunk_total:,}")
print(f"matches full read: {chunk_total == staff['salary'].sum()}")

processed 60 rows in chunks
total salary: 69,401,000
matches full read: True


`chunksize=20` makes `read_csv` return an iterator of DataFrames rather than one big frame. Each
chunk is a normal DataFrame, so you accumulate whatever you need and let each piece be freed.

This works for anything you can compute incrementally — sums, counts, filters. It does not work
directly for operations needing all the data at once, like a median or a global sort; those need a
two-pass approach or a tool built for out-of-core data.

In [197]:
# Filtering a large file down to something that fits in memory.
wanted = pd.concat(
    [chunk[chunk["department"] == "Engineering"] for chunk in pd.read_csv(csv_path, chunksize=20)],
    ignore_index=True,
)
print("engineering rows collected from chunks:", len(wanted))

engineering rows collected from chunks: 25


That is a list comprehension producing a filtered DataFrame per chunk, then one `concat` at the
end. It is the standard pattern for "the file is too big but the part I want is not".

## Cleaning a messy dataset

Everything so far has been on tidy data. Here is what an actual file looks like when it reaches you.

In [198]:
messy = pd.DataFrame({
    "Employee Name": ["  ananya rao", "RAHUL MENON", "Sneha Kulkarni", "imran sheikh  ",
                      "Priya Nair", "RAHUL MENON", "vikram iyer", "Meera Desai", None,
                      "arjun joshi"],
    "Dept ": ["Engineering", "engineering", "MARKETING", "Sales", "marketing",
              "engineering", "Engg", "Finance", "Sales", "HR "],
    "Salary": ["9,20,000", "14,50,000", "6,80,000", "5,40,000", "8,75,000",
               "14,50,000", "11,00,000", "N/A", "7,20,000", "6,50,000"],
    "Joining Date": ["15-01-2020", "2019/03/22", "08-07-2021", "2022-11-30", "14-02-2018",
                     "2019/03/22", "01-06-2017", "19-09-2020", "2021-04-05", "30-08-2022"],
    "Years Exp": ["4", "9", "3", "2", "6", "9", "7", "4", "3", "2"],
    "Active": ["Yes", "yes", "Y", "No", "YES", "yes", "1", "no", "Yes", "N"],
})

messy

,Employee Name,Dept,Salary,Joining Date,Years Exp,Active
0,ananya rao,Engineering,"9,20,000",15-01-2020,4,Yes
1,RAHUL MENON,engineering,"14,50,000",2019/03/22,9,yes
2,Sneha Kulkarni,MARKETING,"6,80,000",08-07-2021,3,Y
3,imran sheikh,Sales,"5,40,000",2022-11-30,2,No
4,Priya Nair,marketing,"8,75,000",14-02-2018,6,YES
5,RAHUL MENON,engineering,"14,50,000",2019/03/22,9,yes
6,vikram iyer,Engg,"11,00,000",01-06-2017,7,1
7,Meera Desai,Finance,N/A,19-09-2020,4,no
8,NaN,Sales,"7,20,000",2021-04-05,3,Yes
9,arjun joshi,HR,"6,50,000",30-08-2022,2,N


Count the problems:

1. Column names have spaces, capitals, and a trailing space in `"Dept "`.
2. Names have inconsistent capitalisation and stray whitespace, and one is missing.
3. `Salary` is text, uses Indian-style digit grouping, and has an `"N/A"`.
4. `Joining Date` mixes two formats.
5. `Years Exp` is text that should be a number.
6. `Active` has seven spellings of two values.
7. Row 5 is an exact duplicate of row 1.
8. `Dept` has `Engineering`, `engineering` and `Engg` for the same department.

We will fix these one at a time, checking after each step. Fixing them all in one heroic
expression is how you end up unable to tell which part went wrong.

### Step 1 — column names

In [199]:
clean = messy.copy()
clean.columns = (clean.columns
                 .str.strip()
                 .str.lower()
                 .str.replace(" ", "_"))

print(clean.columns.tolist())

['employee_name', 'dept', 'salary', 'joining_date', 'years_exp', 'active']


Strip whitespace, lowercase, replace spaces with underscores. Three steps, applied to all the
column names at once through the Index's `.str` accessor. From here on, every column is reachable
without backticks or surprises.

### Step 2 — trim and normalise the text

In [200]:
clean["employee_name"] = clean["employee_name"].str.strip().str.title()
clean["dept"] = clean["dept"].str.strip().str.title()

print(clean[["employee_name", "dept"]])

    employee_name         dept
0      Ananya Rao  Engineering
1     Rahul Menon  Engineering
2  Sneha Kulkarni    Marketing
3    Imran Sheikh        Sales
4      Priya Nair    Marketing
5     Rahul Menon  Engineering
6     Vikram Iyer         Engg
7     Meera Desai      Finance
8             NaN        Sales
9     Arjun Joshi           Hr


`.str.strip().str.title()` handles both whitespace and capitalisation. Names now compare correctly:
`"  ananya rao"` and `"Ananya Rao"` would have been two different people to any grouping operation.

Note the missing name survived as `NaN` rather than becoming the string `"Nan"` — string methods
skip missing values instead of stringifying them.

### Step 3 — the inconsistent categories

In [201]:
print("before:", sorted(clean["dept"].unique()))

dept_fixes = {"Engg": "Engineering", "Eng": "Engineering", "Hr": "HR"}
clean["dept"] = clean["dept"].replace(dept_fixes)

print("after :", sorted(clean["dept"].unique()))

before: ['Engg', 'Engineering', 'Finance', 'Hr', 'Marketing', 'Sales']
after : ['Engineering', 'Finance', 'HR', 'Marketing', 'Sales']


`.replace(dict)` maps specific values to new ones and **leaves everything else alone** — which is
the crucial difference from `.map(dict)`, where unmatched values become `NaN`.

Use `replace` for fixing a few known bad values in an otherwise good column. Use `map` when you
are translating a complete, closed set of values and want to be told about anything unexpected.

Finding the values that need fixing is the real work, and `value_counts` is how you do it. There is
no way to automate the judgement that `"Engg"` means `"Engineering"`; that is domain knowledge, and
you should write down the mapping you used.

### Step 4 — text to numbers

In [202]:
print("before:", clean["salary"].tolist()[:3], "dtype:", clean["salary"].dtype)

clean["salary"] = (clean["salary"]
                   .str.replace(",", "", regex=False)
                   .pipe(pd.to_numeric, errors="coerce"))

print("after :", clean["salary"].tolist()[:3], "dtype:", clean["salary"].dtype)
print("failed to convert:", clean["salary"].isna().sum())

before: ['9,20,000', '14,50,000', '6,80,000'] dtype: str
after : [920000.0, 1450000.0, 680000.0] dtype: float64
failed to convert: 1


Two steps: remove the digit separators, then convert. `pd.to_numeric(..., errors="coerce")` turns
anything unconvertible — our `"N/A"` — into `NaN` rather than raising.

`.pipe(pd.to_numeric, errors="coerce")` is a way of putting a plain function into a method chain:
`x.pipe(f, arg)` is exactly `f(x, arg)`. It keeps the chain reading left to right instead of
forcing you to wrap the whole expression in `pd.to_numeric(...)`. If that feels like a step too
far, the two-line version is perfectly good:

```python
clean["salary"] = clean["salary"].str.replace(",", "", regex=False)
clean["salary"] = pd.to_numeric(clean["salary"], errors="coerce")
```

`regex=False` in `str.replace` matters: by default Pandas treats the pattern as a regular
expression, and while a comma means the same thing either way, a `.` or `$` would not.

In [203]:
clean["years_exp"] = pd.to_numeric(clean["years_exp"])
print(clean[["salary", "years_exp"]].dtypes)

salary       float64
years_exp      int64
dtype: object


### Step 5 — the dates

In [204]:
print("mixed formats:", clean["joining_date"].tolist()[:4])

clean["joining_date"] = pd.to_datetime(clean["joining_date"], format="mixed", dayfirst=True)

print("\nparsed:")
print(clean["joining_date"])

mixed formats: ['15-01-2020', '2019/03/22', '08-07-2021', '2022-11-30']

parsed:
0   2020-01-15
1   2019-03-22
2   2021-07-08
3   2022-11-30
4   2018-02-14
5   2019-03-22
6   2017-06-01
7   2020-09-19
8   2021-05-04
9   2022-08-30
Name: joining_date, dtype: datetime64[us]


`format="mixed"` tells Pandas to work each value out individually, which is exactly the situation
here: some rows are `15-01-2020` and others are `2019/03/22`. `dayfirst=True` resolves the
ambiguity in favour of day-month-year, which is right for this data.

`format="mixed"` is slower than a single explicit format and should not be your default — but for
genuinely mixed input it beats writing the parsing loop yourself.

Always sanity-check parsed dates against something you know:

In [205]:
print("date range:", clean["joining_date"].min().date(), "to", clean["joining_date"].max().date())
print("any in the future?", (clean["joining_date"] > pd.Timestamp("2024-01-01")).sum())

date range: 2017-06-01 to 2022-11-30
any in the future? 0


### Step 6 — the boolean column

In [206]:
print("before:", clean["active"].unique().tolist())

true_values = {"yes", "y", "1", "true", "t"}
clean["active"] = clean["active"].str.strip().str.lower().isin(true_values)

print("after :", clean["active"].unique().tolist())
print(clean["active"].value_counts())

before: ['Yes', 'yes', 'Y', 'No', 'YES', '1', 'no', 'N']
after : [True, False]
active
True     7
False    3
Name: count, dtype: int64


Normalise to lowercase first, then test membership in the set of things that mean "true". Anything
else becomes `False`.

That last part deserves a moment's thought: this approach silently maps unrecognised values —
including missing ones — to `False`. That is fine when you are confident about the vocabulary and
dangerous otherwise. The careful version checks first:

In [207]:
known = {"yes", "y", "1", "true", "t", "no", "n", "0", "false", "f"}
unexpected = set(messy["Active"].str.strip().str.lower()) - known
print("values not in the known vocabulary:", unexpected or "none")

values not in the known vocabulary: none


Empty, so the conversion is safe. Had anything shown up there, we would want to look at it rather
than let it quietly become `False`.

### Step 7 — duplicates

In [208]:
print("total rows:", len(clean))
print("exact duplicate rows:", clean.duplicated().sum())
print("\nthe duplicates:")
print(clean[clean.duplicated(keep=False)][["employee_name", "dept", "salary"]])

total rows: 10
exact duplicate rows: 1

the duplicates:
  employee_name         dept     salary
1   Rahul Menon  Engineering  1450000.0
5   Rahul Menon  Engineering  1450000.0


`duplicated()` marks the **second and later** occurrences. `keep=False` marks all copies including
the first, which is what you want when you are looking at them rather than removing them.

In [209]:
clean = clean.drop_duplicates()
print("rows after dropping duplicates:", len(clean))

rows after dropping duplicates: 9


For real data, exact duplicates are the easy case. The harder one is near-duplicates — the same
person entered twice with a different spelling — and there you choose which columns define
identity:

In [210]:
print("rows with a duplicate name+dept combination:",
      clean.duplicated(subset=["employee_name", "dept"]).sum())

rows with a duplicate name+dept combination: 0


`subset=` restricts the check to the columns that identify a record. If two rows agree on those, the
rest may differ in ways that matter — one may be more recent, one may be more complete — and then
`keep="last"` or a sort-then-drop becomes the right tool rather than a blanket
`drop_duplicates()`.

### Step 8 — handle what is left missing

In [211]:
print(clean.isna().sum())
print("\nrows with gaps:")
print(clean[clean.isna().any(axis=1)])

employee_name    1
dept             0
salary           1
joining_date     0
years_exp        0
active           0
dtype: int64

rows with gaps:
  employee_name     dept    salary joining_date  years_exp  active
7   Meera Desai  Finance       NaN   2020-09-19          4   False
8           NaN    Sales  720000.0   2021-05-04          3    True


Two gaps: one missing name and one salary that was `"N/A"`.

Decisions, stated explicitly:

- The **missing name** is a record we cannot attribute to anyone. For a payroll report that row is
  unusable, so drop it. For a headcount it is still a real employee. Here we keep it and mark it,
  because dropping records should be a deliberate choice, not a side effect.
- The **missing salary** should not be invented. We leave it as `NaN`, where every Pandas
  aggregation will correctly exclude it, and record how many rows are affected.

In [212]:
clean["employee_name"] = clean["employee_name"].fillna("Unknown")

print(clean)
print("\nsalary still missing for", clean["salary"].isna().sum(), "row(s) — left as-is on purpose")

    employee_name         dept     salary joining_date  years_exp  active
0      Ananya Rao  Engineering   920000.0   2020-01-15          4    True
1     Rahul Menon  Engineering  1450000.0   2019-03-22          9    True
2  Sneha Kulkarni    Marketing   680000.0   2021-07-08          3    True
3    Imran Sheikh        Sales   540000.0   2022-11-30          2   False
4      Priya Nair    Marketing   875000.0   2018-02-14          6    True
6     Vikram Iyer  Engineering  1100000.0   2017-06-01          7    True
7     Meera Desai      Finance        NaN   2020-09-19          4   False
8         Unknown        Sales   720000.0   2021-05-04          3    True
9     Arjun Joshi           HR   650000.0   2022-08-30          2   False

salary still missing for 1 row(s) — left as-is on purpose


### The cleaned result, and a reusable function

In [213]:
print(clean.dtypes)
print()
print(clean.describe(include="all").T)

employee_name               str
dept                        str
salary                  float64
joining_date     datetime64[us]
years_exp                 int64
active                     bool
dtype: object

              count unique          top freq                 mean  \
employee_name     9      9   Ananya Rao    1                  NaN   
dept              9      5  Engineering    3                  NaN   
salary          8.0    NaN          NaN  NaN             866875.0   
joining_date      9    NaN          NaN  NaN  2020-06-09 00:00:00   
years_exp       9.0    NaN          NaN  NaN             4.444444   
active            9      2         True    6                  NaN   

                               min                  25%                  50%  \
employee_name                  NaN                  NaN                  NaN   
dept                           NaN                  NaN                  NaN   
salary                    540000.0             672500.0             7

Every column now has the right type, categories are consistent, dates are dates, and the one
genuine gap is still visible as a gap.

Cleaning code has a habit of being written once in a notebook and then needed again next month, so
it is worth collecting into a function:

In [214]:
def clean_employee_file(raw: pd.DataFrame) -> pd.DataFrame:
    """Standardise a raw employee export. Returns a new DataFrame."""
    dept_fixes = {"Engg": "Engineering", "Eng": "Engineering", "Hr": "HR"}
    true_values = {"yes", "y", "1", "true", "t"}

    out = raw.copy()
    out.columns = out.columns.str.strip().str.lower().str.replace(" ", "_")

    out["employee_name"] = out["employee_name"].str.strip().str.title().fillna("Unknown")
    out["dept"] = out["dept"].str.strip().str.title().replace(dept_fixes)
    out["salary"] = pd.to_numeric(out["salary"].str.replace(",", "", regex=False),
                                  errors="coerce")
    out["years_exp"] = pd.to_numeric(out["years_exp"], errors="coerce")
    out["joining_date"] = pd.to_datetime(out["joining_date"], format="mixed", dayfirst=True)
    out["active"] = out["active"].str.strip().str.lower().isin(true_values)

    return out.drop_duplicates().reset_index(drop=True)


again = clean_employee_file(messy)
print("same result:", again.equals(clean.reset_index(drop=True)))
print(again.dtypes.to_dict())

same result: True
{'employee_name': <StringDtype(storage='python', na_value=nan)>, 'dept': <StringDtype(storage='python', na_value=nan)>, 'salary': dtype('float64'), 'joining_date': dtype('<M8[us]'), 'years_exp': dtype('int64'), 'active': dtype('bool')}


`df.equals(other)` compares two DataFrames completely — values, dtypes and index — and is the right
way to check that a refactor did not change the output. `==` would give you a DataFrame of
element-wise comparisons and would say `False` for two `NaN`s that should count as equal.

The type hints in the signature (`raw: pd.DataFrame -> pd.DataFrame`) are optional in Python and
worth writing on a function like this: the next person to read it learns what goes in and what
comes out without reading the body.

### Cleaning up the demo files

In [215]:
import shutil

shutil.rmtree(output_dir)
print("removed", output_dir, "— exists:", output_dir.exists())

removed pandas_io_demo — exists: False


Tidying up after ourselves so the notebook does not leave files behind. In real work you would
obviously keep them.

## Beginner versus idiomatic Pandas — side by side

A collected comparison of the patterns covered in this notebook. The left column is not *wrong*;
it is what most people write first, and it is often the version to keep for a one-off script.

### Computing a new column

In [216]:
work = staff.head(6).copy()

# Beginner
values = []
for i in range(len(work)):
    values.append(work["salary"].iloc[i] / 12)
work["monthly_a"] = values

# Idiomatic
work["monthly_b"] = work["salary"] / 12

print(np.allclose(work["monthly_a"], work["monthly_b"]), "— same answer")

True — same answer


**Verdict: always use the column expression.** Shorter, faster, and there is no case where the loop
is clearer.

### Filtering

In [217]:
# Beginner
keep = []
for i in range(len(staff)):
    if staff["salary"].iloc[i] > 1_200_000 and staff["department"].iloc[i] == "Engineering":
        keep.append(i)
result_a = staff.iloc[keep]

# Idiomatic
result_b = staff[(staff["salary"] > 1_200_000) & (staff["department"] == "Engineering")]

print(len(result_a), len(result_b), "— same rows")

15 15 — same rows


**Verdict: use the mask.** The loop version also has a subtle trap — it collects *positions* and
then uses `.iloc`, which breaks if the DataFrame was filtered earlier and its index no longer
matches its positions.

### Aggregating by group

In [218]:
# Beginner
sums = {}
for i in range(len(staff)):
    d = staff["department"].iloc[i]
    sums[d] = sums.get(d, 0) + staff["salary"].iloc[i]

# Idiomatic
sums_b = staff.groupby("department")["salary"].sum()

print(sorted(sums.items()) == sorted(sums_b.to_dict().items()), "— same totals")

True — same totals


**Verdict: use `groupby`.** The dictionary version does not extend — add "and the count, and the
max, and the median" and it doubles in length each time, while `agg` takes one more line.

### Applying a text transformation

In [219]:
# Beginner
upper = []
for name in staff["name"]:
    upper.append(name.upper())

# Idiomatic
upper_b = staff["name"].str.upper()

print(upper[:2], "==", upper_b.head(2).tolist())

['SHALINI REDDY', 'SWATI DESAI'] == ['SHALINI REDDY', 'SWATI DESAI']


**Verdict: use `.str`.** It also handles missing values correctly, which the loop does not — the
loop raises `AttributeError: 'float' object has no attribute 'upper'` the first time it meets a
`NaN`.

In [220]:
with_nan = pd.Series(["Ananya", np.nan])
try:
    [n.upper() for n in with_nan]
except AttributeError as e:
    print("loop version:", e)
print("str version  :", with_nan.str.upper().tolist())

loop version: 'float' object has no attribute 'upper'
str version  : ['ANANYA', nan]


### Building a table row by row

In [221]:
# Beginner — appending in a loop
rows = []
for dept, group in staff.groupby("department"):
    rows.append({"department": dept, "headcount": len(group), "avg_salary": group["salary"].mean()})
built_a = pd.DataFrame(rows)

# Idiomatic
built_b = (staff.groupby("department")
           .agg(headcount=("emp_id", "count"), avg_salary=("salary", "mean"))
           .reset_index())

print(built_a.shape, built_b.shape, "— same shape")

(5, 3) (5, 3) — same shape


**Verdict: `agg` is better here, but the loop deserves defending.** Collecting dictionaries in a
list and building the DataFrame *once* at the end is a perfectly good pattern — it is how you
assemble a table from an API, a set of files, or anything that is not already a DataFrame.

What you must not do is grow a DataFrame one row at a time:

```python
# Don't do this
result = pd.DataFrame()
for ...:
    result = pd.concat([result, pd.DataFrame([new_row])])
```

Each `concat` copies the entire DataFrame, so building n rows costs n² work. Collect into a list,
then build once.

### The summary table

| Task | Beginner | Idiomatic | Is the change worth it? |
| --- | --- | --- | --- |
| New column from arithmetic | loop and append | `df["c"] = df["a"] / df["b"]` | Always |
| Filter rows | loop with `if` | boolean mask | Always |
| Group summary | dict of accumulators | `groupby().agg()` | Always |
| Text transformation | loop calling `.upper()` | `.str.upper()` | Always |
| Two-way flag | `apply` with `if/else` | `np.where` | Usually |
| Multi-way category | `apply` with `if/elif` | `np.select` or `pd.cut` | On large data |
| Compare row to its group | `groupby` then `merge` | `transform` | Usually |
| Assemble from non-Pandas data | list of dicts, then `DataFrame` | — | Keep the loop |
| Add rows repeatedly | `concat` in a loop | list then one `DataFrame` | Always |

## Performance

Some practical habits, in rough order of how much they matter.

### 1. Vectorise instead of iterating rows

In [222]:
big = pd.DataFrame({
    "a": rng.random(500_000),
    "b": rng.random(500_000),
})

_ = timed("iterrows", lambda: sum(r["a"] * r["b"] for _, r in big.head(20_000).iterrows()))
_ = timed("apply axis=1 (20k rows)", lambda: big.head(20_000).apply(lambda r: r["a"] * r["b"], axis=1).sum())
_ = timed("vectorised (all 500k rows)", lambda: (big["a"] * big["b"]).sum())

iterrows                           0.2158 s
apply axis=1 (20k rows)            0.0745 s
vectorised (all 500k rows)         0.0025 s


Note that the first two lines only processed 20,000 rows and the third did all 500,000 — and the
third still won by a wide margin. `iterrows()` in particular builds a Series object for every row,
which is why it is the slowest way to touch a DataFrame and why you should treat seeing it in code
as a smell.

### 2. Read only what you need

In [223]:
staff.to_csv("temp_perf.csv", index=False)

_ = timed("read all columns", lambda: pd.read_csv("temp_perf.csv"))
_ = timed("read two columns", lambda: pd.read_csv("temp_perf.csv", usecols=["name", "salary"]))

Path("temp_perf.csv").unlink()

read all columns                   0.0010 s
read two columns                   0.0006 s


On a 60-row file the difference is noise. The point is the habit: on a file with 200 columns where
you need 5, `usecols` cuts both the load time and the memory footprint by a large factor. Same
for `nrows` when you are exploring.

### 3. Use `category` for repetitive text

In [224]:
n_big = 200_000
repetitive = pd.DataFrame({
    "department": rng.choice(["Engineering", "Marketing", "Sales", "Finance", "HR"], n_big),
    "value": rng.random(n_big),
})

as_text = repetitive["department"].memory_usage(deep=True)
as_category = repetitive["department"].astype("category").memory_usage(deep=True)

print(f"as text    : {as_text:>10,} bytes")
print(f"as category: {as_category:>10,} bytes")
print(f"reduction  : {(1 - as_category / as_text):.0%}")

as text    : 11,160,549 bytes
as category:    200,411 bytes
reduction  : 98%


A categorical column stores each distinct value once and keeps small integer codes per row. With
five departments across 200,000 rows that is a large saving, and grouping on a categorical column
is faster too.

The rule of thumb: convert when the number of distinct values is small relative to the number of
rows. Converting a column of unique names or IDs to `category` makes things *worse*, because you
store the full lookup table plus the codes.

In [225]:
unique_ids = pd.Series([f"E{i}" for i in range(n_big)])
print("unique text     :", f"{unique_ids.memory_usage(deep=True):,} bytes")
print("unique category :", f"{unique_ids.astype('category').memory_usage(deep=True):,} bytes")

unique text     : 11,089,022 bytes


unique category : 11,889,022 bytes


### 4. Choose the right numeric dtype

In [226]:
years = pd.Series(rng.integers(1, 40, n_big))

print("int64:", f"{years.memory_usage(deep=True):>10,} bytes")
print("int16:", f"{years.astype('int16').memory_usage(deep=True):>10,} bytes")
print("int8 :", f"{years.astype('int8').memory_usage(deep=True):>10,} bytes")

int64:  1,600,132 bytes
int16:    400,132 bytes
int8 :    200,132 bytes


`int64` holds numbers up to about 9 quintillion. For an age, a year of experience or a rating out
of five, that is 8 bytes where 1 would do. `pd.to_numeric(s, downcast="integer")` picks the
smallest safe type for you.

Be careful, though: a too-small type overflows silently, and the failure is ugly.

In [227]:
small = pd.Series([100, 120], dtype="int8")
print("int8 max is 127. Adding 50:")
print((small + 50).tolist())

int8 max is 127. Adding 50:
[-106, -86]


120 + 50 gave a negative number. Only downcast columns whose realistic maximum you actually know,
and never downcast a column you are about to accumulate into.

### 5. Things that do not matter as much as people think

- **`inplace=True`** does not reliably save memory, and it prevents chaining. It is not a
  performance feature.
- **Chained indexing versus `.loc`** is about correctness, not speed.
- **Micro-optimising a one-off script.** If it runs in four seconds and you run it once a week,
  it is fast enough. Spend the effort on making it correct and readable.

The order that actually matters: get the algorithm right (vectorise, avoid row loops), then read
less data, then shrink dtypes. Measure before and after — `time.perf_counter()` as used above, or
`%%timeit` in a notebook cell — rather than guessing.

## Common mistakes

### Chained assignment

The most important one, and the behaviour changed in Pandas 3, so older advice about it is out
of date.

In [228]:
import warnings

demo = staff.head(5).copy()
before = demo.loc[0, "salary"]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    demo["salary"][0] = 1

print("pandas", pd.__version__)
for w in caught:
    print(f"  warned: {type(w.message).__name__}")

print(f"\nvalue before : {before}")
print(f"value after  : {demo.loc[0, 'salary']}")
print(f"write landed : {demo.loc[0, 'salary'] == 1}")

pandas 3.0.5
  warned: ChainedAssignmentError

value before : 1018000
value after  : 1018000
write landed : False


Look at what your own output says, because **this is the one place in the notebook where the
answer depends on your Pandas version** — and that is exactly the lesson.

- **On Pandas 3**, the warning is `ChainedAssignmentError` and `write landed` is `False`. Nothing
  happened at all.
- **On Pandas 2**, the warning is `SettingWithCopyWarning` and `write landed` may be either. It
  depends on how the data happens to be laid out in memory.

Why any of it: `demo["salary"]` produces a temporary Series, and `[0] = 1` writes into that
temporary, which is then discarded. Whether the temporary shares memory with the original — and so
whether your write leaks through to it — was an implementation detail in Pandas 2. Under
copy-on-write, always on since Pandas 3, the temporary is guaranteed to be a copy, so chained
assignment can never work.

That is the real argument for the change. The old behaviour was not *wrong* so much as
**unpredictable**, and unpredictable is worse: code that worked in testing could quietly stop
working on a differently shaped frame. Pandas 3 made it reliably do nothing, and tell you.

Either way the fix is identical — do the whole thing in one `.loc`:

In [229]:
demo.loc[0, "salary"] = 1
print("after .loc:", demo.loc[0, "salary"])

after .loc: 1


The general rule: **one set of brackets on the left of the `=`.** If you see two, rewrite it.

```python
df["col"][mask] = value          # broken
df[mask]["col"] = value          # broken
df.loc[mask, "col"] = value      # correct
```

### Forgetting that most methods return a new object

In [230]:
demo = staff.head(3).copy()

demo.sort_values("salary")
print("did sorting stick?", demo["salary"].tolist())

demo = demo.sort_values("salary")
print("after assigning  :", demo["salary"].tolist())

did sorting stick? [1018000, 1115000, 1190000]
after assigning  : [1018000, 1115000, 1190000]


`sort_values`, `drop`, `rename`, `fillna`, `reset_index` and nearly everything else return a new
DataFrame and leave the original alone. Forgetting to assign the result is the most common
"why did nothing happen" in Pandas.

### Comparing against `NaN`

In [231]:
values = pd.Series([1.0, np.nan, 3.0])

print("values == np.nan :", (values == np.nan).tolist())
print("values.isna()    :", values.isna().tolist())

values == np.nan : [False, False, False]
values.isna()    : [False, True, False]


`NaN` is not equal to anything, including itself. Always use `.isna()`.

### Assuming `groupby` keeps your row order

In [232]:
print(staff.groupby("department")["salary"].mean().index.tolist())
print(staff.groupby("department", sort=False)["salary"].mean().index.tolist())

['Engineering', 'Finance', 'HR', 'Marketing', 'Sales']
['Sales', 'Finance', 'Engineering', 'Marketing', 'HR']


`groupby` sorts the group keys by default. Pass `sort=False` to keep them in the order they first
appear — which is both faster and what you want when the keys have a natural order that is not
alphabetical.

### Integer columns turning into floats

Covered in the missing-data section, and worth repeating because it confuses people twice: a single
missing value converts an integer column to float. If you need integers with gaps, use the
nullable `"Int64"` dtype.

### Merging without checking

Also covered above: check the row count, and use `validate=`. A merge that silently multiplies
rows produces plausible, wrong numbers, and that is the worst failure mode there is.

## Debugging: three errors, diagnosed

### Error 1 — `KeyError` on a column that is definitely there

In [233]:
try:
    staff["Salary"]
except KeyError as e:
    print("KeyError:", e)

print("\nactual column names:", staff.columns.tolist())

KeyError: 'Salary'

actual column names: ['emp_id', 'name', 'department', 'years_experience', 'salary', 'city', 'remote', 'joined']


**What Python tells us.** `KeyError: 'Salary'`.

**Why it happened.** Column names are case-sensitive and whitespace-sensitive. Ours is `salary`.
This also happens with a trailing space in the name — invisible in the printed table and fatal to
every lookup.

**Fix.** Print `df.columns.tolist()` and compare character by character. If the name has stray
whitespace, `df.columns = df.columns.str.strip()` deals with it once and for all.

In [234]:
spaced = pd.DataFrame({"name ": ["a"], " city": ["b"]})
print("before:", spaced.columns.tolist())
spaced.columns = spaced.columns.str.strip()
print("after :", spaced.columns.tolist())

before: ['name ', ' city']
after : ['name', 'city']


### Error 2 — the ambiguous truth value

In [235]:
try:
    if staff["salary"] > 1_000_000:
        print("never reached")
except ValueError as e:
    print("ValueError:", e)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


**What Python tells us.** `The truth value of a Series is ambiguous. Use a.empty, a.bool(),
a.item(), a.any() or a.all().`

**Why it happened.** `staff["salary"] > 1_000_000` is 60 answers, and `if` needs one. Python cannot
decide whether "some are true" or "all are true" is meant, so it refuses.

**Fix.** Say which you mean:

In [236]:
if (staff["salary"] > 1_000_000).any():
    print("at least one employee earns over a million")

if (staff["salary"] > 100_000).all():
    print("everyone earns over a hundred thousand")

print("how many:", (staff["salary"] > 1_000_000).sum())

at least one employee earns over a million
everyone earns over a hundred thousand
how many: 40


The same error appears when you use `and` between two conditions in a filter. The fix there is `&`
with parentheses, not `.any()` — make sure you know which of the two situations you are in.

### Error 3 — a merge that quietly loses rows

In [237]:
partial_info = dept_info[dept_info["department"] != "HR"]

inner = staff.merge(partial_info, on="department")
left = staff.merge(partial_info, on="department", how="left")

print("staff rows      :", len(staff))
print("after inner join:", len(inner), "← rows disappeared")
print("after left join :", len(left))
print("\nwhere did they go?")
print(left[left["head"].isna()]["department"].value_counts())

staff rows      : 60
after inner join: 56 ← rows disappeared
after left join : 60

where did they go?
department
HR    4
Name: count, dtype: int64


**What Python tells us.** Nothing. Both merges succeed.

**Why it happened.** `partial_info` has no `HR` row, and an inner join keeps only keys present in
both tables, so every HR employee was dropped. Any headcount or payroll total computed from
`inner` is now wrong, by exactly the amount nobody will notice.

**Fix.** Use `how="left"` when you mean "keep all my rows", and check.

In [238]:
print("rows preserved:", len(left) == len(staff))
print("rows with no match, flagged:")
print(staff.merge(partial_info, on="department", how="left", indicator=True)["_merge"]
      .value_counts())

rows preserved: True
rows with no match, flagged:
_merge
both          56
left_only      4
right_only     0
Name: count, dtype: int64


**Better still.** Make the check part of the code, so it fails loudly next time:

In [239]:
def safe_left_merge(left_df, right_df, on):
    """Left-join and refuse to silently change the row count."""
    before = len(left_df)
    merged = left_df.merge(right_df, on=on, how="left", validate="many_to_one")
    if len(merged) != before:
        raise ValueError(f"merge changed row count: {before} → {len(merged)}")
    return merged


ok = safe_left_merge(staff, dept_info, on="department")
print("merged safely:", ok.shape)

try:
    safe_left_merge(staff, duplicated_info, on="department")
except (ValueError, pd.errors.MergeError) as e:
    print("caught:", type(e).__name__, "-", str(e)[:70])

merged safely: (60, 10)
caught: MergeError - Merge keys are not unique in right dataset; not a many-to-one merge

D


## Mini project — employee analytics

One dataset, start to finish: raw and slightly messy, through cleaning, into the summaries a
manager would actually ask for.

The brief, as it might land on your desk:

> We are reviewing compensation. Can you tell us how pay varies by department and location,
> whether experience is being rewarded consistently, who is paid unusually little for their
> experience, and whether remote employees are paid differently?

Every one of those is answerable with what this notebook has covered.

### The raw data

In [240]:
rng_proj = np.random.default_rng(2025)

n_emp = 240
depts = ["Engineering", "Product", "Sales", "Marketing", "Finance", "Support"]
dept_p = [0.32, 0.12, 0.18, 0.12, 0.10, 0.16]
dept_base = {"Engineering": 1_050_000, "Product": 1_150_000, "Sales": 720_000,
             "Marketing": 760_000, "Finance": 880_000, "Support": 520_000}
city_multiplier = {"Bengaluru": 1.08, "Mumbai": 1.05, "Pune": 0.96,
                   "Hyderabad": 0.98, "Chennai": 0.94, "Remote": 0.92}

dept_raw = rng_proj.choice(depts, n_emp, p=dept_p)
city_raw = rng_proj.choice(list(city_multiplier), n_emp,
                           p=[0.30, 0.18, 0.15, 0.15, 0.12, 0.10])
experience = np.clip(rng_proj.gamma(2.2, 3.0, n_emp).round(0), 0, 30).astype(int)

pay = (
    np.array([dept_base[d] for d in dept_raw])
    * np.array([city_multiplier[c] for c in city_raw])
    * (1 + experience * 0.055)
    * rng_proj.normal(1.0, 0.11, n_emp)
).round(-3)

hr_export = pd.DataFrame({
    "Emp ID": [f"EMP{2000 + i}" for i in range(n_emp)],
    "Full Name": [f"{rng_proj.choice(first_names)} {rng_proj.choice(last_names)}"
                  for _ in range(n_emp)],
    "Department": dept_raw,
    "Location": city_raw,
    "Experience (Years)": experience.astype(str),
    "Annual CTC": [f"{int(p):,}" for p in pay],
    "Employment Type": rng_proj.choice(["Full-time", "full time", "FULL-TIME", "Contract"],
                                       n_emp, p=[0.6, 0.15, 0.15, 0.10]),
    "Performance Rating": rng_proj.choice([1, 2, 3, 4, 5], n_emp,
                                          p=[0.05, 0.15, 0.40, 0.30, 0.10]),
    "Joining Date": pd.to_datetime("2024-06-30") - pd.to_timedelta(
        experience * 365 + rng_proj.integers(0, 360, n_emp), unit="D"),
})

# A realistic export is never perfectly complete.
hr_export.loc[rng_proj.choice(n_emp, 11, replace=False), "Annual CTC"] = None
hr_export.loc[rng_proj.choice(n_emp, 7, replace=False), "Location"] = None
hr_export = pd.concat([hr_export, hr_export.sample(4, random_state=1)], ignore_index=True)

hr_export.head()

,Emp ID,Full Name,Department,Location,Experience (Years),Annual CTC,Employment Type,Performance Rating,Joining Date
0,EMP2000,Shalini Singh,Support,Bengaluru,5,"713,000",Full-time,3,2018-11-05
1,EMP2001,Kavita Verma,Product,Bengaluru,2,"1,190,000",Full-time,3,2021-08-11
2,EMP2002,Neha Kulkarni,Finance,Bengaluru,6,"1,290,000",full time,5,2017-11-25
3,EMP2003,Varun Rao,Finance,Remote,3,"695,000",Full-time,4,2020-09-20
4,EMP2004,Neha Joshi,Support,Bengaluru,6,"871,000",Full-time,3,2018-03-15


Two things were done deliberately to the export: some values were blanked, and four rows were
duplicated. Both are ordinary for a spreadsheet that has been through a few hands.

### Step 1 — inspect before touching anything

In [241]:
print("shape:", hr_export.shape)
print()
hr_export.info()

shape: (244, 9)

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Emp ID              244 non-null    str           
 1   Full Name           244 non-null    str           
 2   Department          244 non-null    str           
 3   Location            237 non-null    str           
 4   Experience (Years)  244 non-null    str           
 5   Annual CTC          232 non-null    str           
 6   Employment Type     244 non-null    str           
 7   Performance Rating  244 non-null    int64         
 8   Joining Date        244 non-null    datetime64[us]
dtypes: datetime64[us](1), int64(1), str(7)
memory usage: 17.3 KB


Reading that output: 244 rows, nine columns, and two columns with fewer non-null values than rows.
`Experience (Years)` and `Annual CTC` are text when they should be numbers. `Performance Rating`
is already numeric. `Joining Date` is a real datetime, because we built it that way — a real
export would probably hand you a string.

In [242]:
for column in ["Department", "Location", "Employment Type"]:
    print(f"{column}:")
    print(hr_export[column].value_counts(dropna=False).to_string())
    print()

Department:
Department
Engineering    79
Sales          50
Support        38
Product        29
Finance        24
Marketing      24

Location:
Location
Bengaluru    76
Mumbai       39
Hyderabad    39
Chennai      31
Pune         28
Remote       24
NaN           7

Employment Type:
Employment Type
Full-time    139
FULL-TIME     43
full time     32
Contract      30



`dropna=False` includes the missing values in the count, which is the version you want when you are
auditing a column. `Employment Type` has the expected problem: four spellings for two real
categories.

### Step 2 — clean

In [243]:
def clean_hr_export(raw):
    """Turn the raw HR export into an analysable table."""
    out = raw.copy()

    out.columns = (out.columns
                   .str.strip()
                   .str.lower()
                   .str.replace(r"[^\w]+", "_", regex=True)
                   .str.strip("_"))

    out["annual_ctc"] = pd.to_numeric(
        out["annual_ctc"].str.replace(",", "", regex=False), errors="coerce")
    out["experience_years"] = pd.to_numeric(out["experience_years"], errors="coerce")

    out["employment_type"] = (out["employment_type"]
                              .str.strip()
                              .str.lower()
                              .str.replace(" ", "-")
                              .map({"full-time": "Full-time", "contract": "Contract"}))

    out["location"] = out["location"].fillna("Unknown")

    return out.drop_duplicates(subset="emp_id").reset_index(drop=True)


emp = clean_hr_export(hr_export)

print("columns:", emp.columns.tolist())
print("\nrows: %d (was %d)" % (len(emp), len(hr_export)))
print("\ndtypes:")
print(emp.dtypes)

columns: ['emp_id', 'full_name', 'department', 'location', 'experience_years', 'annual_ctc', 'employment_type', 'performance_rating', 'joining_date']

rows: 240 (was 244)

dtypes:
emp_id                           str
full_name                        str
department                       str
location                         str
experience_years               int64
annual_ctc                   float64
employment_type                  str
performance_rating             int64
joining_date          datetime64[us]
dtype: object


Four points about that function.

`str.replace(r"[^\w]+", "_", regex=True)` replaces any run of non-word characters with a single
underscore, which turns `"Experience (Years)"` into `"experience_years"` in one step rather than
needing a rule per punctuation mark. `.str.strip("_")` then removes a trailing underscore left by
a name ending in punctuation.

`drop_duplicates(subset="emp_id")` uses the employee ID as the definition of identity, rather than
requiring every column to match. That is the right call here: the ID is meant to be unique, so two
rows sharing one are the same person however the other fields look.

`employment_type` uses `.map()` rather than `.replace()` deliberately. After normalising the
spelling there should be exactly two values; `map` turns anything unexpected into `NaN`, so a new
category appearing in next month's export shows up as a gap rather than passing through silently.

`location` gaps become `"Unknown"` — an honest category that keeps the rows in every grouping.

In [244]:
print("missing values after cleaning:")
print(emp.isna().sum()[lambda s: s > 0])
print("\nduplicate IDs:", emp["emp_id"].duplicated().sum())
print("employment types:", emp["employment_type"].unique().tolist())

missing values after cleaning:
annual_ctc    11
dtype: int64

duplicate IDs: 0
employment types: ['Full-time', 'Contract']


`emp.isna().sum()[lambda s: s > 0]` filters the summary to columns that actually have gaps. Passing
a function to `[]` is a neat trick: it is called with the Series and its result used as the
selector, which saves naming an intermediate variable.

Eleven missing salaries remain, on purpose. We will exclude those rows from pay analysis and say so,
rather than inventing values.

### Step 3 — the overall picture

In [245]:
paid = emp.dropna(subset=["annual_ctc"])

print(f"employees on file        : {len(emp)}")
print(f"with salary recorded     : {len(paid)} ({len(paid) / len(emp):.1%})")
print()
print(paid["annual_ctc"].describe().round(0).to_string())
print()
print("median is %.0f but the mean is %.0f — the distribution is right-skewed"
      % (paid["annual_ctc"].median(), paid["annual_ctc"].mean()))

employees on file        : 240
with salary recorded     : 229 (95.4%)



count        229.0
mean     1171188.0
std       382248.0
min       468000.0
25%       910000.0
50%      1122000.0
75%      1424000.0
max      3086000.0

median is 1122000 but the mean is 1171188 — the distribution is right-skewed


The mean sitting above the median tells you a few high salaries are pulling the average up. For
compensation work that matters: quoting the mean makes typical pay look higher than it is. Report
the median, and the quartiles if people will accept them.

In [246]:
print("percentiles:")
print((paid["annual_ctc"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]) / 100_000)
      .round(1).rename("lakhs").to_string())

percentiles:
0.10     6.9
0.25     9.1
0.50    11.2
0.75    14.2
0.90    16.4
0.99    22.2


### Step 4 — pay by department

In [247]:
by_dept = (paid.groupby("department")
           .agg(headcount=("emp_id", "count"),
                median_ctc=("annual_ctc", "median"),
                mean_ctc=("annual_ctc", "mean"),
                lowest=("annual_ctc", "min"),
                highest=("annual_ctc", "max"),
                median_experience=("experience_years", "median"))
           .sort_values("median_ctc", ascending=False))

print((by_dept / 1).round(0).to_string())

             headcount  median_ctc   mean_ctc    lowest    highest  median_experience
department                                                                           
Product           26.0   1586000.0  1596154.0  957000.0  3086000.0                6.0
Engineering       73.0   1333000.0  1390699.0  641000.0  2304000.0                6.0
Finance           23.0   1136000.0  1211783.0  695000.0  1961000.0                5.0
Marketing         23.0    988000.0  1046304.0  671000.0  1419000.0                5.0
Sales             50.0    961500.0   988400.0  661000.0  1613000.0                6.0
Support           34.0    692000.0   700735.0  468000.0   999000.0                5.0

Sorted by median pay, Product and Engineering lead and Support trails — which is what the generator
was built to produce, so the analysis is recovering a real pattern rather than noise.

The `lowest` and `highest` columns are worth including in any pay summary: two departments with the
same median can have very different spreads, and the spread is usually what people are actually
asking about.

In [248]:
print("pay spread within each department (highest ÷ lowest):")
print((by_dept["highest"] / by_dept["lowest"]).round(2).sort_values(ascending=False).to_string())

pay spread within each department (highest ÷ lowest):
department
Engineering    3.59
Product        3.22
Finance        2.82
Sales          2.44
Support        2.13
Marketing      2.11


### Step 5 — is experience rewarded consistently?

In [249]:
bands = pd.cut(paid["experience_years"], bins=[-1, 2, 5, 10, 40],
               labels=["0-2", "3-5", "6-10", "11+"])

experience_table = (paid.assign(band=bands)
                    .groupby("band", observed=True)
                    .agg(headcount=("emp_id", "count"),
                         median_ctc=("annual_ctc", "median")))

experience_table["uplift_vs_previous"] = (
    experience_table["median_ctc"].pct_change().mul(100).round(1)
)

print(experience_table.round(0).to_string())

      headcount  median_ctc  uplift_vs_previous
band                                           
0-2          31    957000.0                 NaN
3-5          83    992000.0                 4.0
6-10         76   1324500.0                34.0
11+          39   1383000.0                 4.0


`bins=[-1, 2, 5, 10, 40]` starts at -1 because `cut` excludes the left edge by default, and we have
employees with zero years. Starting at 0 would have put them outside every bin and given them `NaN`.
That is a genuinely easy mistake to make, and the giveaway is a band count that does not add up.

`pct_change()` computes the percentage change from one row to the next, so each band shows the
median uplift over the band below it. `observed=True` on the groupby tells Pandas not to include
empty categorical bands in the result.

In [250]:
print("total in bands:", experience_table["headcount"].sum(), "of", len(paid))

total in bands: 229 of 229


### Step 6 — the cross-tabulation managers ask for

In [251]:
pivot = paid.pivot_table(index="department", columns="location",
                         values="annual_ctc", aggfunc="median")

print((pivot / 100_000).round(1).to_string())
print("\n(figures in lakhs; blank cells mean no employee in that combination)")

location     Bengaluru  Chennai  Hyderabad  Mumbai  Pune  Remote  Unknown
department                                                               
Engineering       14.4     12.7       11.6    14.0  13.8    13.4     16.8
Finance           14.6     10.7       10.6    12.4   9.8    10.9      NaN
Marketing         10.2      9.1       10.1    13.0   9.3     9.2     10.9
Product           14.4     15.4       16.5    16.4  15.8    15.6      NaN
Sales             10.7      9.2       11.0     9.5   7.8     9.8     11.1
Support            7.0      6.4        6.5     7.1   7.9     5.8      5.8

(figures in lakhs; blank cells mean no employee in that combination)


A pivot table is the natural shape for "two dimensions, one number", and this one immediately shows
both effects at once — department down the side, location across the top.

Empty cells are real information: with 240 employees across six departments and six locations,
some of the 36 combinations have nobody in them. Filling those with zero would be a lie, so leave
them blank.

In [252]:
print("how many employees per cell:")
print(paid.pivot_table(index="department", columns="location",
                       values="annual_ctc", aggfunc="count", fill_value=0).to_string())

how many employees per cell:
location     Bengaluru  Chennai  Hyderabad  Mumbai  Pune  Remote  Unknown
department                                                               
Engineering         19       14         12      14     8       4        2
Finance              5        3          3       6     3       3        0
Marketing            8        3          4       3     3       1        1
Product             12        1          5       1     3       4        0
Sales               15        5          6      10     6       6        2
Support             11        4          7       4     5       2        1


Always look at the counts next to a pivot of medians. A cell holding one employee reads exactly
like a cell holding thirty, and one person's salary is not a location premium.

### Step 7 — who is paid unusually little for their experience?

This is the question that needs `transform`, because it compares each individual to their own group.

In [253]:
flagged = paid.assign(
    dept_median=lambda d: d.groupby("department")["annual_ctc"].transform("median"),
    ratio_to_dept=lambda d: (d["annual_ctc"] / d.groupby("department")["annual_ctc"]
                             .transform("median")).round(2),
    exp_rank_in_dept=lambda d: d.groupby("department")["experience_years"]
    .rank(ascending=False, method="min"),
)

THRESHOLD = 0.85
MIN_EXPERIENCE = 5

underpaid = (flagged[(flagged["ratio_to_dept"] < THRESHOLD)
                     & (flagged["experience_years"] >= MIN_EXPERIENCE)]
             .sort_values("ratio_to_dept")
             [["full_name", "department", "location", "experience_years",
               "annual_ctc", "dept_median", "ratio_to_dept"]])

print(f"flagged: {len(underpaid)} employees with {MIN_EXPERIENCE}+ years "
      f"earning under {THRESHOLD:.0%} of their department median\n")
print(underpaid.head(10).to_string(index=False))

flagged: 4 employees with 5+ years earning under 85% of their department median

   full_name  department  location  experience_years  annual_ctc  dept_median  ratio_to_dept
Shalini Khan Engineering Hyderabad                 7   1019000.0    1333000.0           0.76
  Anjali Rao       Sales      Pune                 5    772000.0     961500.0           0.80
Nikhil Joshi     Support    Remote                 5    568000.0     692000.0           0.82
 Sameer Khan     Finance   Chennai                 6    943000.0    1136000.0           0.83


Two conditions combined: below 85% of the department median, and experienced enough that low pay
is not simply explained by being new. The result is a shortlist for a human to look at, which is
what this kind of analysis should produce — not a verdict.

The two thresholds are in named constants rather than buried in the expression, which matters more
than it looks. Someone reviewing this will want to ask "what if we said 80%?", and with the numbers
named that is a one-character edit in an obvious place.

Worth stating plainly: this flags a *pattern*, not unfairness. Someone might have joined from a
different role, or be in a genuinely different job under the same department label. The analysis
narrows the question; it does not answer it.

### Step 8 — do remote employees earn differently?

In [254]:
remote_comparison = (paid.assign(is_remote=lambda d: d["location"] == "Remote")
                     .groupby("is_remote")
                     .agg(headcount=("emp_id", "count"),
                          median_ctc=("annual_ctc", "median"),
                          median_experience=("experience_years", "median")))

print(remote_comparison.round(0).to_string())

gap = (remote_comparison.loc[True, "median_ctc"]
       / remote_comparison.loc[False, "median_ctc"] - 1)
print(f"\nremote median is {gap:+.1%} relative to on-site")

           headcount  median_ctc  median_experience
is_remote                                          
False            209   1126000.0                6.0
True              20   1040500.0                6.0

remote median is -7.6% relative to on-site


There is a gap. Before reporting it as a remote-pay penalty, check whether the two groups are
comparable — if remote employees are less experienced or concentrated in lower-paying departments,
the gap is explained by that instead.

In [255]:
print("department mix, on-site vs remote (% of each group):")
mix = (paid.assign(is_remote=lambda d: d["location"] == "Remote")
       .groupby("is_remote")["department"]
       .value_counts(normalize=True)
       .unstack()
       .mul(100)
       .round(1))
print(mix.to_string())

department mix, on-site vs remote (% of each group):
department  Engineering  Finance  Marketing  Product  Sales  Support
is_remote                                                           
False              33.0      9.6       10.5     10.5   21.1     15.3
True               20.0     15.0        5.0     20.0   30.0     10.0


That is the whole lesson of the project in one table. A raw group difference is a starting point,
not a finding. Comparing like with like — same department, similar experience — is the next step:

In [256]:
like_for_like = (paid.assign(is_remote=lambda d: d["location"] == "Remote")
                 .pivot_table(index="department", columns="is_remote",
                              values="annual_ctc", aggfunc="median"))
like_for_like.columns = ["on_site", "remote"]
like_for_like["difference_pct"] = ((like_for_like["remote"] / like_for_like["on_site"] - 1)
                                   * 100).round(1)

print(like_for_like.round(0).to_string())

               on_site     remote  difference_pct
department                                       
Engineering  1333000.0  1335000.0             0.0
Finance      1183000.0  1092000.0            -8.0
Marketing    1024000.0   924000.0           -10.0
Product      1586000.0  1562500.0            -2.0
Sales         961500.0   982500.0             2.0
Support       693500.0   576500.0           -17.0


Within each department the picture is more mixed than the headline number suggested, and a couple
of cells are missing entirely because no remote employee works in that department. That is a more
honest answer to the question than a single percentage, and it is the kind of result worth handing
back to whoever asked.

### Step 9 — the deliverable

In [257]:
def compensation_report(df):
    """Assemble the tables a compensation review needs."""
    with_pay = df.dropna(subset=["annual_ctc"])

    return {
        "coverage": pd.Series({
            "employees": len(df),
            "with_salary": len(with_pay),
            "coverage_pct": round(100 * len(with_pay) / len(df), 1),
        }),
        "by_department": (with_pay.groupby("department")
                          .agg(headcount=("emp_id", "count"),
                               median_ctc=("annual_ctc", "median"),
                               median_exp=("experience_years", "median"))
                          .sort_values("median_ctc", ascending=False)),
        "by_rating": (with_pay.groupby("performance_rating")
                      .agg(headcount=("emp_id", "count"),
                           median_ctc=("annual_ctc", "median"))),
        "top_ten": (with_pay.nlargest(10, "annual_ctc")
                    [["full_name", "department", "experience_years", "annual_ctc"]]),
    }


report = compensation_report(emp)

for title, table in report.items():
    print(f"--- {title} ---")
    print(table.round(0).to_string())
    print()

--- coverage ---


employees       240.0
with_salary     229.0
coverage_pct     95.0

--- by_department ---
             headcount  median_ctc  median_exp
department                                    
Product             26   1586000.0         6.0
Engineering         73   1333000.0         6.0
Finance             23   1136000.0         5.0
Marketing           23    988000.0         5.0
Sales               50    961500.0         6.0
Support             34    692000.0         5.0

--- by_rating ---
                    headcount  median_ctc
performance_rating                       
1                           9    944000.0
2                          22   1176000.0
3                         101   1110000.0
4                          73   1092000.0
5                          24   1304000.0

--- top_ten ---
         full_name   department  experience_years  annual_ctc
133   Shalini Bose      Product                19   3086000.0
197     Meera Khan  Engineering                19   2304000.0
13    Fatima Joshi

### What the project demonstrated

Working backwards through the steps:

- **Cleaning came first and took the most code.** That is normal. Analysis on badly typed columns
  produces confident nonsense.
- **`groupby` + `agg`** answered every "by department" question.
- **`transform`** answered the only question about individuals in context, which is exactly the
  distinction that section laboured.
- **`pivot_table`** handled two dimensions at once, and the matching count table kept it honest.
- **The statistical care mattered more than the Pandas.** Median over mean for skewed pay, counts
  alongside medians, and checking whether a group difference survives controlling for department.

Notice also what is missing: not a single loop over rows, and not one `apply(axis=1)`.

The one thing this analysis is crying out for is a chart — the department comparison and the
experience-versus-pay relationship are both much easier to read as a picture than as a table. That
is the next notebook.

## Solutions

### Series

In [258]:
# Level 1
marks = pd.Series([68, 74, 55, 91, 83],
                  index=["Maths", "Physics", "Chemistry", "English", "CompSci"])

print("highest mark :", marks.max())
print("best subject :", marks.idxmax())
print("mean         :", marks.mean())

highest mark : 91
best subject : English
mean         : 74.2


`max()` gives the value, `idxmax()` gives the **index label** where it occurs. That pairing —
`max`/`idxmax`, `min`/`idxmin` — is how you get from "what was the best score" to "which subject
was it".

In [259]:
# Level 2
above = marks[marks > marks.mean()]
print("above average:")
print(above)
print("count:", len(above))

with_bonus = (marks + 5).clip(upper=100)
print("\nwith bonus, capped at 100:")
print(with_bonus)

above average:
English    91
CompSci    83
dtype: int64
count: 2

with bonus, capped at 100:
Maths        73
Physics      79
Chemistry    60
English      96
CompSci      88
dtype: int64


`marks > marks.mean()` compares every mark to the single mean value — the scalar broadcasts.
`.clip(upper=100)` caps the result; `np.minimum(marks + 5, 100)` does the same thing, and
`clip` says it more directly.

In [260]:
# Level 3
shop_a = pd.Series([120, 80, 45], index=["pens", "books", "bags"])
shop_b = pd.Series([60, 30, 55], index=["books", "pens", "erasers"])

print("plain addition:")
print(shop_a + shop_b)
print("\nwith fill_value=0:")
print(shop_a.add(shop_b, fill_value=0))

plain addition:
bags         NaN
books      140.0
erasers      NaN
pens       150.0
dtype: float64

with fill_value=0:
bags        45.0
books      140.0
erasers     55.0
pens       150.0
dtype: float64


`bags` and `erasers` become `NaN` in the plain addition, because each appears in only one Series
and Pandas will not assume the other is zero.

Which version to hand over depends on what the question is. For **total sales across both shops**,
`fill_value=0` is right: shop B genuinely sold zero bags, and 45 is the correct total. The `NaN`
version is right when a missing label means "not stocked, so not comparable" — you would not want
to report that shop B sells bags badly if it does not sell bags at all.

The general principle: `NaN` means "unknown", `0` means "known to be none". Choose whichever your
data actually is, and never let the default make the decision for you.

### DataFrames and selection

In [261]:
# Level 1
print(staff.head(8)[["name", "department"]])
print("\ncolumns:", staff.columns.tolist())
print("shape:", staff.shape)
print()
print(staff.loc[:4, ["name", "city"]])

            name   department
0  Shalini Reddy        Sales
1    Swati Desai      Finance
2    Aisha Reddy      Finance
3     Kavita Rao  Engineering
4     Vikram Rao  Engineering
5     Sameer Rao      Finance
6   Arjun Sheikh  Engineering
7  Manish Sheikh      Finance

columns: ['emp_id', 'name', 'department', 'years_experience', 'salary', 'city', 'remote', 'joined']
shape: (60, 8)

            name       city
0  Shalini Reddy       Pune
1    Swati Desai     Mumbai
2    Aisha Reddy  Bengaluru
3     Kavita Rao       Pune
4     Vikram Rao  Bengaluru


In [262]:
# Level 2
eng_high = staff.loc[
    (staff["department"] == "Engineering") & (staff["salary"] > 1_000_000),
    ["name", "years_experience", "salary"],
]
print(f"{len(eng_high)} engineers earning over 1,000,000")
print(eng_high.head())

non_eng_high = staff.loc[
    ~(staff["department"] == "Engineering") & (staff["salary"] > 1_000_000),
    ["name", "department", "salary"],
]
print(f"\n{len(non_eng_high)} non-engineers earning over 1,000,000")
print(non_eng_high.head())

21 engineers earning over 1,000,000
            name  years_experience   salary
3     Kavita Rao                10  1458000
4     Vikram Rao                 9  1243000
6   Arjun Sheikh                15  1707000
10     Arjun Rao                 6  1191000
11   Rohan Joshi                 4  1076000

19 non-engineers earning over 1,000,000
               name department   salary
0     Shalini Reddy      Sales  1018000
1       Swati Desai    Finance  1115000
2       Aisha Reddy    Finance  1190000
5        Sameer Rao    Finance  1239000
8  Nandini Kulkarni    Finance  1233000


`~(staff["department"] == "Engineering")` reads as "not equal to Engineering", and
`staff["department"] != "Engineering"` says the same thing more directly. Use `!=` for a single
negated comparison; save `~` for negating a compound condition or a named mask.

In [263]:
# Level 3
try:
    staff.loc[staff.salary > 1_000_000, "name", "salary"]
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:100]}")

IndexingError: Too many indexers


`.loc` accepts at most two things: a row selector and a column selector. The broken line passes
three, because the two column names were written as separate arguments instead of as one list.

In [264]:
fixed = staff.loc[staff.salary > 1_000_000, ["name", "salary"]]
print(fixed.head(3))

positions = np.where(staff["salary"].to_numpy() > 1_000_000)[0]
column_positions = [staff.columns.get_loc("name"), staff.columns.get_loc("salary")]
iloc_version = staff.iloc[positions, column_positions]

print("\nsame rows via iloc:", iloc_version.shape == fixed.shape)

            name   salary
0  Shalini Reddy  1018000
1    Swati Desai  1115000
2    Aisha Reddy  1190000

same rows via iloc: True


The `.iloc` version works and is plainly worse. It has to convert the condition into integer
positions with `np.where`, and look up the column positions with `get_loc`, and it would break the
moment someone reorders the columns. `.loc` lets you say what you mean — this condition, those
names — which is why it is the right tool for a request phrased in labels and conditions.

### Filtering, columns, missing data, sorting

In [265]:
# Level 1
blr_senior = staff[(staff["city"] == "Bengaluru") & (staff["years_experience"] > 5)]
print(f"{len(blr_senior)} senior employees in Bengaluru")

top_monthly = (staff.assign(salary_monthly=lambda d: (d["salary"] / 12).round(0))
               .nlargest(5, "salary_monthly")
               [["name", "department", "salary_monthly"]])
print(top_monthly.to_string(index=False))

13 senior employees in Bengaluru
          name  department  salary_monthly
   Rohan Menon Engineering        143917.0
  Arjun Sheikh Engineering        142250.0
  Deepak Menon     Finance        138167.0
   Rohan Reddy Engineering        134333.0
Arjun Kulkarni Engineering        132750.0


In [266]:
# Level 2
banded = staff.assign(
    pay_band_select=np.select(
        [staff["salary"] > 1_200_000, staff["salary"] >= 800_000],
        ["High", "Medium"],
        default="Low",
    ),
    pay_band_cut=pd.cut(staff["salary"],
                        bins=[0, 800_000, 1_200_000, np.inf],
                        labels=["Low", "Medium", "High"],
                        right=False),
)

print("do the two agree?",
      (banded["pay_band_select"] == banded["pay_band_cut"].astype(str)).all())
print()
print(pd.crosstab(banded["department"], banded["pay_band_select"]))

do the two agree? True

pay_band_select  High  Low  Medium
department                        
Engineering        15    0      10
Finance             6    0       5
HR                  0    1       3
Marketing           4    1       5
Sales               1    2       7


The two approaches need care to line up, and the exercise is partly about noticing why.

`np.select` checks `> 1_200_000` first and then `>= 800_000`, so the boundaries are explicit.
`pd.cut` with `right=False` makes each bin `[low, high)` — closed on the left, open on the right —
which matches "800,000 to 1,200,000 is Medium, and 1,200,000 itself is High".

With `cut`'s default `right=True` the bins would be `(low, high]` and a salary of exactly
1,200,000 would land in Medium, disagreeing with the `np.select` version. The lesson: whenever you
bin, state which end is inclusive and test a value sitting exactly on a boundary.

`pd.crosstab(a, b)` counts combinations of two columns — a shortcut for
`groupby([a, b]).size().unstack(fill_value=0)`.

In [267]:
# Level 3
def report_missing(df):
    """One row per column with gaps: how many, and what share of the rows."""
    counts = df.isna().sum()
    counts = counts[counts > 0]
    return (pd.DataFrame({
        "missing": counts,
        "percent": (counts / len(df) * 100).round(1),
    }).sort_values("missing", ascending=False))


print(report_missing(gappy).to_string())

                  missing  percent
salary                  4      6.7
years_experience        3      5.0


Building the result as a DataFrame rather than printing inside the function means the caller can
sort it, filter it, or write it to a file. `counts[counts > 0]` drops the clean columns by filtering
the Series with a mask over itself.

And the strategies, which is the part that matters:

- **`years_experience` (3 missing, 5%)** — a small number of rows missing a field that drives the
  analysis. Drop those rows for any experience-based calculation, and say how many were dropped.
  Do not fill: an invented experience level changes the relationship you are trying to measure.
- **`salary` (4 missing, 6.7%)** — leave as `NaN`. Pandas aggregations skip them correctly, and
  filling with the mean would fabricate certainty (the demonstration above showed the standard
  deviation shrinking). If the rows must be kept for a model, fill with the *department* median via
  `transform` and add a `salary_was_missing` flag column so the imputation stays visible.
- **`city` (6 missing, 10%)** — a category, so fill with `"Unknown"`. Keeps the rows, keeps the
  headcount right, and makes the gap visible in every grouping instead of silently excluding those
  employees from city comparisons.

### GroupBy and aggregation

In [268]:
# Level 1
by_city = (staff.groupby("city")
           .agg(headcount=("emp_id", "count"), avg_salary=("salary", "mean"))
           .sort_values("avg_salary", ascending=False))
print(by_city.round(0).to_string())

           headcount  avg_salary
city                            
Hyderabad         12   1256250.0
Delhi              3   1198000.0
Mumbai            11   1169909.0
Bengaluru         22   1125000.0
Pune              12   1092750.0


In [269]:
# Level 2
dept_summary = staff.groupby("department").agg(
    headcount=("emp_id", "count"),
    median_salary=("salary", "median"),
    remote_share=("remote", "mean"),
)

dept_summary["top_earner"] = (staff.loc[staff.groupby("department")["salary"].idxmax()]
                              .set_index("department")["name"])

print(dept_summary.round(2).to_string())

             headcount  median_salary  remote_share    top_earner
department                                                       
Engineering         25      1251000.0          0.56   Rohan Menon
Finance             11      1221000.0          0.27  Deepak Menon
HR                   4       918500.0          1.00    Ananya Rao
Marketing           10       943000.0          0.50   Arjun Verma
Sales               10       970000.0          0.10   Suresh Bose


The `top_earner` column is the interesting part. `groupby("department")["salary"].idxmax()` gives
the **index label** of the highest-paid row in each department; `staff.loc[...]` looks those rows
up; `set_index("department")["name"]` turns the result into a Series indexed by department, which
then aligns with `dept_summary` automatically on assignment.

A named aggregation cannot do this directly, because `agg` functions receive only the one column
being aggregated — the salary values, with no access to the names beside them. When you need a
value *from another column* of the winning row, `idxmax` plus `.loc` is the pattern.

In [270]:
# an alternative with a custom function on the whole group
print(staff.groupby("department")
      .apply(lambda g: g.loc[g["salary"].idxmax(), "name"], include_groups=False)
      .rename("top_earner")
      .to_string())

department
Engineering     Rohan Menon
Finance        Deepak Menon
HR               Ananya Rao
Marketing       Arjun Verma
Sales           Suresh Bose


In [271]:
# Level 3
with_transform = staff.assign(
    dept_max=lambda d: d.groupby("department")["salary"].transform("max"),
    pct_of_dept_max=lambda d: (d["salary"] / d.groupby("department")["salary"]
                               .transform("max") * 100).round(1),
)

below_70 = with_transform[with_transform["pct_of_dept_max"] < 70]
print(f"{len(below_70)} employees below 70% of their department's top salary")
print(below_70[["name", "department", "salary", "dept_max", "pct_of_dept_max"]]
      .head(6).to_string(index=False))

28 employees below 70% of their department's top salary
         name  department  salary  dept_max  pct_of_dept_max
Shalini Reddy       Sales 1018000   1497000             68.0
  Swati Desai     Finance 1115000   1658000             67.2
Manish Sheikh     Finance  854000   1658000             51.5
  Tarun Joshi   Marketing  908000   1358000             66.9
    Arjun Rao Engineering 1191000   1727000             69.0
  Rohan Joshi Engineering 1076000   1727000             62.3


In [272]:
# the same thing with groupby + merge
maxes = (staff.groupby("department")["salary"].max()
         .rename("dept_max").reset_index())

with_merge = staff.merge(maxes, on="department", how="left")
with_merge["pct_of_dept_max"] = (with_merge["salary"] / with_merge["dept_max"] * 100).round(1)

print("\nsame answer:",
      len(with_merge[with_merge["pct_of_dept_max"] < 70]) == len(below_70))


same answer: True


Both work. The `transform` version is one expression and cannot get the join wrong. The merge
version is three steps, needs a `rename` to avoid a column collision, and would silently multiply
rows if `maxes` ever had duplicate departments.

Keep the `transform` version. The merge is worth having written once, because it shows what
`transform` is doing — and because there are cases (bringing in several columns from another table,
or a lookup that is not derived from the data itself) where a merge is the only option.

### Text, dates, merging, reshaping

In [273]:
# Level 1
surnames = staff.assign(surname=lambda d: d["name"].str.split(" ").str[-1].str.upper())
before_m = surnames[surnames["surname"].str[0] < "M"]

print(f"{len(before_m)} employees with a surname starting before 'M'")
print(surnames["surname"].head().tolist())

23 employees with a surname starting before 'M'
['REDDY', 'DESAI', 'REDDY', 'RAO', 'RAO']


Comparing strings with `<` compares them alphabetically, so `surname.str[0] < "M"` keeps A through
L. It works because the surnames are uppercase — with mixed case, every lowercase letter sorts
after every uppercase one and the answer would be wrong. Normalising case before comparing is not
optional.

In [274]:
# Level 2
merged = staff.merge(dept_info, on="department", how="left", validate="many_to_one")
print("row count unchanged:", len(merged) == len(staff))

payroll = (merged.groupby("department")
           .agg(payroll=("salary", "sum"),
                budget_cr=("annual_budget_cr", "first")))

payroll["budget"] = payroll["budget_cr"] * 10_000_000
payroll["payroll_pct_of_budget"] = (payroll["payroll"] / payroll["budget"] * 100).round(1)

print(payroll[["payroll", "budget", "payroll_pct_of_budget"]].round(0).to_string())

row count unchanged: True
              payroll       budget  payroll_pct_of_budget
department                                               
Engineering  32153000  480000000.0                    7.0
Finance      13324000   95000000.0                   14.0
HR            3672000   60000000.0                    6.0
Marketing    10462000  125000000.0                    8.0
Sales         9790000  180000000.0                    5.0


`budget_cr=("annual_budget_cr", "first")` uses `first` because the budget is the same for every row
in the department — it was copied onto each employee by the merge. Using `sum` there would multiply
the budget by the headcount, which is the classic error when aggregating a merged table.

That is worth stating as a rule: after a one-to-many merge, columns from the "one" side are
repeated, so aggregate them with `first`, `max` or `mean` — never `sum`.

In [275]:
# Level 3
joiners = (staff.assign(join_year=lambda d: d["joined"].dt.year)
           .pivot_table(index="department", columns="join_year",
                        values="emp_id", aggfunc="count", fill_value=0))

print(joiners.to_string())
print("\ntotal:", joiners.to_numpy().sum())

long_again = (joiners.reset_index()
              .melt(id_vars="department", var_name="join_year", value_name="headcount"))

print("\nmelted, non-zero rows only:")
print(long_again[long_again["headcount"] > 0].head(6).to_string(index=False))
print("\ntotal after melting:", long_again["headcount"].sum())

join_year    2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022
department                                                                                           
Engineering     4     1     1     0     1     1     5     2     1     2     1     1     1     0     4
Finance         2     2     0     0     1     1     1     0     1     0     1     0     2     0     0
HR              0     0     0     0     0     1     0     0     1     0     0     2     0     0     0
Marketing       0     1     2     1     0     0     0     0     2     1     0     0     1     2     0
Sales           1     0     0     0     0     2     1     1     1     0     0     2     0     1     1

total: 60

melted, non-zero rows only:
 department join_year  headcount
Engineering      2008          4
    Finance      2008          2
      Sales      2008          1
Engineering      2009          1
    Finance      2009          2
  Marketing      2009          1

total after me

`fill_value=0` is right here because a department with nobody joining in 2016 genuinely had zero
joiners — the count is known, not missing. Contrast with the pay pivot in the mini project, where
an empty cell meant "no data" and zero would have been a lie. Same argument, opposite conclusion,
decided by what the number means.

The wide version goes in the report: departments down the side, years across the top, easy to
scan. The long version goes into further calculations — one row per observation, so `groupby`,
filtering and plotting all work without naming the years.

## Quick reference

**Loading and saving**

```python
pd.read_csv(path, usecols=[...], parse_dates=[...], na_values=[...], nrows=1000)
pd.read_excel(path, sheet_name="Sheet1")    pd.read_json(path, orient="records")
df.to_csv(path, index=False)                df.to_excel(path, index=False)
for chunk in pd.read_csv(path, chunksize=100_000): ...
```

**Inspecting**

```python
df.head()   df.tail()   df.sample(5)   df.shape   df.columns   df.index
df.dtypes   df.info()   df.describe()  df.nunique()
df["col"].value_counts()   df["col"].unique()   df.isna().sum()
```

**Selecting**

```python
df["col"]                    df[["a", "b"]]
df.loc[mask]                 df.loc[mask, ["a", "b"]]      df.loc[label, "col"]
df.iloc[0]                   df.iloc[0:5, 0:3]             df.iat[0, 1]
df.set_index("id")           df.reset_index(drop=True)
```

**Filtering**

```python
df[df["a"] > 5]                       df[(df["a"] > 5) & (df["b"] == "x")]
df[df["c"].isin([1, 2, 3])]           df[df["a"].between(5, 10)]
df[~df["c"].isin([1, 2])]             df.query("a > 5 and b == 'x'")
```

**Columns**

```python
df["new"] = df["a"] * 2               df["flag"] = df["a"] > 5
df["label"] = np.where(cond, "yes", "no")
df["cat"] = np.select([c1, c2], ["A", "B"], default="C")
df["band"] = pd.cut(df["a"], bins=[0, 10, 20], labels=["low", "high"])
df.assign(x=lambda d: d["a"] / d["b"])
df.drop(columns=["a"])                df.rename(columns={"a": "b"})
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
```

**Missing data**

```python
df.isna()   df.notna()   df.isna().sum()   df.isna().mean()
df.dropna(subset=["salary"])          df.fillna(0)   df.fillna(df["a"].median())
df["a"].ffill()   df["a"].bfill()     df["a"].interpolate()
df.groupby("g")["a"].transform("median")   # group-aware filling
```

**Grouping**

```python
df.groupby("g")["a"].mean()
df.groupby("g").agg(n=("a", "count"), avg=("a", "mean"), best=("a", "max"))
df.groupby("g")["a"].transform("mean")     # one value per original row
df.groupby("g").filter(lambda sub: len(sub) > 5)
df.groupby(["g", "h"])["a"].mean().unstack()
df.groupby("g", sort=False, as_index=False)["a"].sum()
```

**Text and dates**

```python
s.str.strip()   s.str.lower()   s.str.title()   s.str.len()
s.str.contains("x", na=False)   s.str.replace("a", "b", regex=False)
s.str.split(" ", expand=True)   s.str.extract(r"(\d+)")   s.str[:3]

pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
s.dt.year   s.dt.month_name()   s.dt.dayofweek   s.dt.quarter
s.resample("ME").sum()          s.rolling(7).mean()
```

**Combining and reshaping**

```python
pd.concat([a, b], ignore_index=True)
a.merge(b, on="key", how="left", validate="many_to_one", indicator=True)
a.merge(b, left_on="x", right_on="y", suffixes=("_a", "_b"))
df.pivot(index="i", columns="c", values="v")
df.pivot_table(index="i", columns="c", values="v", aggfunc="mean", margins=True)
df.melt(id_vars="id", var_name="variable", value_name="value")
pd.crosstab(df["a"], df["b"])
```

**Sorting and ranking**

```python
df.sort_values("a")                   df.sort_values(["a", "b"], ascending=[True, False])
df.sort_index()                       df.nlargest(5, "a")   df.nsmallest(5, "a")
df["a"].rank(ascending=False)         df["a"].idxmax()
```

### Where to go next

Open `03_Matplotlib.ipynb`. The tables built here — pay by department, pay against experience,
joiners per year — are all easier to understand as pictures, and every one of them plots directly
from a DataFrame.